In [21]:
!python -m pip install -q pandas numpy scikit-learn



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import sys
import importlib.util
from pathlib import Path
import pandas as pd

def import_module_from_path(module_name: str, file_path: str):
    path = Path(file_path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"Cannot find file: {path}")

    spec = importlib.util.spec_from_file_location(module_name, str(path))
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot create import spec for: {path}")

    mod = importlib.util.module_from_spec(spec)

    # CRITICAL: register module before executing (dataclasses relies on this)
    sys.modules[module_name] = mod

    spec.loader.exec_module(mod)
    return mod

# --- paths (relative to hospital_readmission.ipynb) ---
DATA_PATH = Path("database/diabetic_data.csv")
MAP_PATH  = Path("database/IDS_mapping.csv")

# --- import your project code ---
project_code = import_module_from_path("project_code", "code.py")
print("Imported:", project_code.__file__)

# --- load raw dataset (keeps tokens like '?' as literal strings) ---
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at: {DATA_PATH.resolve()}")

df_raw = pd.read_csv(
    DATA_PATH,
    dtype="string",         # consistent for mapping + later cleaning
    keep_default_na=False,  # do NOT auto-convert 'NA', 'N/A', etc. into NaN
    na_values=[""],         # only empty strings become NaN (rare here, but safe)
    low_memory=False,
)

print("df_raw shape:", df_raw.shape)
print("Columns:", len(df_raw.columns))
df_raw.head()


Imported: C:\Users\Cris-SX\Desktop\UA\CHAD-MASTER-IA\TAU\tau-final-project\code\lvl1\lvl2\tau-final-code\code.py
df_raw shape: (101766, 50)
Columns: 50


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [23]:
from io import StringIO

def load_ids_mapping(path: str | Path) -> dict[str, pd.DataFrame]:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Mapping file not found at: {path.resolve()}")

    text = path.read_text(encoding="utf-8", errors="ignore").splitlines()

    # Split into blocks separated by blank lines or a single comma line
    blocks = []
    cur = []
    for line in text:
        s = line.strip()
        if s in {"", ","}:
            if cur:
                blocks.append("\n".join(cur))
                cur = []
            continue
        cur.append(line)
    if cur:
        blocks.append("\n".join(cur))

    out: dict[str, pd.DataFrame] = {}
    for block in blocks:
        dfm = pd.read_csv(StringIO(block), dtype="string")
        if dfm.shape[1] >= 2 and "description" in dfm.columns:
            key_col = dfm.columns[0]
            dfm = dfm.dropna(subset=[key_col]).copy()
            dfm[key_col] = dfm[key_col].astype("string").str.strip()
            dfm["description"] = dfm["description"].astype("string").str.strip()
            out[key_col] = dfm[[key_col, "description"]]
    return out

id_maps = load_ids_mapping(MAP_PATH)
print("Found mappings:", list(id_maps.keys()))
print({k: v.shape for k, v in id_maps.items()})


Found mappings: ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
{'admission_type_id': (8, 2), 'discharge_disposition_id': (30, 2), 'admission_source_id': (25, 2)}


In [11]:
%%writefile code.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, Literal, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)

READMITTED_CLASSES_3: Tuple[str, str, str] = ("NO", "<30", ">30")


class TargetEngineeringError(ValueError):
    """Raised when the readmitted target contains unexpected/invalid values."""


def _normalize_readmitted_value(v: object) -> Optional[str]:
    """
    Normalize a single readmitted value:
    - None/NaN -> None
    - strings -> stripped, uppercased (except <30 and >30 remain as-is after upper)
    """
    if v is None:
        return None
    # pandas NA / numpy nan
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass

    s = str(v).strip()
    if s == "":
        return None
    return s.upper()


def validate_readmitted_values(
    values: Union[pd.Series, Iterable[object]],
    allowed: Sequence[str] = READMITTED_CLASSES_3,
    *,
    allow_na: bool = False,
) -> None:
    """
    Validate that all non-missing readmitted values are inside 'allowed'.

    Raises:
        TargetEngineeringError if an unexpected value is found.
    """
    allowed_set = {a.upper() for a in allowed}
    if isinstance(values, pd.Series):
        raw_iter = values.tolist()
    else:
        raw_iter = list(values)

    unexpected = set()
    for v in raw_iter:
        nv = _normalize_readmitted_value(v)
        if nv is None:
            if not allow_na:
                # Missing values are unexpected unless allow_na=True
                unexpected.add(None)
            continue
        if nv not in allowed_set:
            unexpected.add(nv)

    if unexpected:
        raise TargetEngineeringError(
            f"Unexpected readmitted values found: {sorted([x for x in unexpected if x is not None])}"
            + (" (and missing values)" if None in unexpected else "")
            + f". Allowed values: {list(allowed)}."
        )


def binarize_readmitted(
    readmitted: pd.Series,
    *,
    positive_value: str = "<30",
    negative_values: Sequence[str] = ("NO", ">30"),
    output_name: str = "readmitted_30d",
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.Series:
    """
    Convert {NO, <30, >30} into binary {1 if <30, 0 otherwise}.

    Args:
        readmitted: pandas Series with original readmitted values.
        positive_value: value mapped to 1.
        negative_values: values mapped to 0.
        output_name: name for the output series.
        unknown_policy:
            - "error": raise if any unexpected/NA values appear
            - "nan": map unexpected/NA to <NA> (nullable integer)

    Returns:
        pd.Series of dtype int8 (or nullable Int8 if unknown_policy="nan").
    """
    pos = positive_value.upper()
    neg = tuple(v.upper() for v in negative_values)

    # Normalize series
    norm = readmitted.map(_normalize_readmitted_value)

    mapping: Dict[str, int] = {pos: 1, **{v: 0 for v in neg}}

    if unknown_policy == "error":
        validate_readmitted_values(norm, allowed=(pos, *neg), allow_na=False)
        out = norm.map(mapping).astype(np.int8)
        out.name = output_name
        return out

    if unknown_policy == "nan":
        # allow NA/unknown -> <NA>
        out = norm.map(mapping)
        out = out.astype("Int8")  # nullable integer supports <NA>
        out.name = output_name
        return out

    raise ValueError(f"unknown_policy must be 'error' or 'nan', got: {unknown_policy}")


def keep_multiclass_readmitted(
    readmitted: pd.Series,
    *,
    output_name: str = "readmitted_3class",
    allowed: Sequence[str] = READMITTED_CLASSES_3,
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.Series:
    """
    Keep the original 3-class label, optionally validating values.

    Returns:
        pd.Series of dtype 'category' with categories in the allowed order,
        or with missing if unknown_policy="nan".
    """
    norm = readmitted.map(_normalize_readmitted_value)

    if unknown_policy == "error":
        validate_readmitted_values(norm, allowed=allowed, allow_na=False)
    elif unknown_policy == "nan":
        # allow NA/unknown; do not raise
        pass
    else:
        raise ValueError(f"unknown_policy must be 'error' or 'nan', got: {unknown_policy}")

    cat = pd.Categorical(norm, categories=[a.upper() for a in allowed], ordered=False)
    out = pd.Series(cat, index=readmitted.index, name=output_name)
    return out


def engineer_targets(
    df: pd.DataFrame,
    *,
    source_col: str = "readmitted",
    binary_col: str = "readmitted_30d",
    multiclass_col: str = "readmitted_3class",
    add_multiclass: bool = True,
    drop_source: bool = False,
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.DataFrame:
    """
    Add engineered target columns to a dataframe:
      - binary: 1 if <30 else 0
      - optional multiclass: categorical {NO, <30, >30}

    This keeps Section 2 self-contained and reproducible.

    Returns:
        A copy of df with new columns added (and optionally source_col dropped).
    """
    if source_col not in df.columns:
        raise KeyError(f"Column '{source_col}' not found in df. Available: {list(df.columns)}")

    out = df.copy()
    out[binary_col] = binarize_readmitted(
        out[source_col],
        output_name=binary_col,
        unknown_policy=unknown_policy,
    )

    if add_multiclass:
        out[multiclass_col] = keep_multiclass_readmitted(
            out[source_col],
            output_name=multiclass_col,
            unknown_policy=unknown_policy,
        )

    if drop_source:
        out.drop(columns=[source_col], inplace=True)

    return out


@dataclass(frozen=True)
class BinaryConfusionTerms:
    tp: int
    fp: int
    fn: int
    tn: int


def confusion_terms_binary(
    y_true: Union[pd.Series, np.ndarray, Sequence[int]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[int]],
    *,
    positive_label: int = 1,
) -> BinaryConfusionTerms:
    """
    Return TP/FP/FN/TN for a binary task once the positive class is defined.

    This directly supports the Section 2 narrative about TP/FP/FN/TN meaning.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    # cm layout with labels [0,1] is:
    # [[TN, FP],
    #  [FN, TP]]
    tn, fp, fn, tp = cm.ravel()
    return BinaryConfusionTerms(tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn))


def metrics_binary(
    y_true: Union[pd.Series, np.ndarray, Sequence[int]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[int]],
    y_score: Optional[Union[pd.Series, np.ndarray, Sequence[float]]] = None,
) -> Dict[str, Optional[float]]:
    """
    Convenience metric bundle for binary framing (Section 2 awareness):
      - accuracy
      - f1
      - balanced_accuracy
      - roc_auc (if y_score is provided)
    """
    res: Dict[str, Optional[float]] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, pos_label=1)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "roc_auc": None,
    }
    if y_score is not None:
        res["roc_auc"] = float(roc_auc_score(y_true, y_score))
    return res


def metrics_multiclass(
    y_true: Union[pd.Series, np.ndarray, Sequence[str]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[str]],
    y_proba: Optional[np.ndarray] = None,
    *,
    labels: Sequence[str] = READMITTED_CLASSES_3,
) -> Dict[str, Optional[float]]:
    """
    Metric bundle for the advanced extension (3-class framing):
      - macro_f1
      - weighted_f1
      - balanced_accuracy (macro recall)
      - ovr_auc_macro (if y_proba is provided with shape [n_samples, n_classes])

    Notes:
      - For AUC, we use one-vs-rest multi-class AUC (macro).
      - Requires y_proba columns correspond to 'labels' in the same order.
    """
    labels_up = [l.upper() for l in labels]

    yt = pd.Series(y_true).map(_normalize_readmitted_value)
    yp = pd.Series(y_pred).map(_normalize_readmitted_value)

    res: Dict[str, Optional[float]] = {
        "macro_f1": float(f1_score(yt, yp, labels=labels_up, average="macro")),
        "weighted_f1": float(f1_score(yt, yp, labels=labels_up, average="weighted")),
        "balanced_accuracy": float(balanced_accuracy_score(yt, yp)),
        "ovr_auc_macro": None,
    }

    if y_proba is not None:
        y_proba = np.asarray(y_proba)
        if y_proba.ndim != 2 or y_proba.shape[1] != len(labels_up):
            raise ValueError(
                f"y_proba must have shape [n_samples, {len(labels_up)}] matching labels order {labels_up}."
            )
        res["ovr_auc_macro"] = float(
            roc_auc_score(yt, y_proba, multi_class="ovr", average="macro", labels=labels_up)
        )

    return res


Overwriting code.py


In [25]:
%%writefile test_target_engineering.py
import os
import unittest
import importlib.util

import numpy as np
import pandas as pd

# Robust import of /content/.../code.py without colliding with the stdlib 'code' module
def import_module_from_path(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot import module from {file_path}")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


PROJECT_DIR = os.path.dirname(os.path.abspath(__file__))
CODE_PATH = os.path.join(PROJECT_DIR, "code.py")
mod = import_module_from_path("project_code", CODE_PATH)


class TestTargetEngineering(unittest.TestCase):
    def test_binarize_basic_mapping(self):
        s = pd.Series(["NO", "<30", ">30", "NO", ">30", "<30"])
        y = mod.binarize_readmitted(s, unknown_policy="error")
        self.assertEqual(list(y.astype(int)), [0, 1, 0, 0, 0, 1])
        self.assertEqual(y.name, "readmitted_30d")
        self.assertTrue(str(y.dtype).lower() in ("int8", "int64", "int32"))

    def test_binarize_handles_whitespace_and_case(self):
        s = pd.Series([" no ", " <30", ">30 ", "No", "<30", " >30"])
        y = mod.binarize_readmitted(s, unknown_policy="error")
        self.assertEqual(list(y.astype(int)), [0, 1, 0, 0, 1, 0])

    def test_binarize_unknown_raises(self):
        s = pd.Series(["NO", "MAYBE", "<30"])
        with self.assertRaises(mod.TargetEngineeringError):
            _ = mod.binarize_readmitted(s, unknown_policy="error")

    def test_binarize_unknown_to_nan(self):
        s = pd.Series(["NO", "MAYBE", "<30", None])
        y = mod.binarize_readmitted(s, unknown_policy="nan")
        # Expect: NO->0, MAYBE->NA, <30->1, None->NA
        self.assertEqual(int(y.iloc[0]), 0)
        self.assertTrue(pd.isna(y.iloc[1]))
        self.assertEqual(int(y.iloc[2]), 1)
        self.assertTrue(pd.isna(y.iloc[3]))
        self.assertEqual(str(y.dtype), "Int8")

    def test_engineer_targets_adds_columns(self):
        df = pd.DataFrame(
            {
                "readmitted": ["NO", "<30", ">30"],
                "age": ["[50-60)", "[60-70)", "[40-50)"],
            }
        )
        out = mod.engineer_targets(df, add_multiclass=True, drop_source=False, unknown_policy="error")
        self.assertIn("readmitted_30d", out.columns)
        self.assertIn("readmitted_3class", out.columns)
        self.assertIn("readmitted", out.columns)
        self.assertEqual(list(out["readmitted_30d"].astype(int)), [0, 1, 0])
        # multiclass should be categorical with expected categories
        self.assertTrue(pd.api.types.is_categorical_dtype(out["readmitted_3class"]))

    def test_confusion_terms_binary(self):
        y_true = [1, 1, 0, 0, 1, 0]
        y_pred = [1, 0, 0, 1, 1, 0]
        terms = mod.confusion_terms_binary(y_true, y_pred)
        # Manually:
        # TP: positions 0 and 4 => 2
        # FN: position 1 => 1
        # FP: position 3 => 1
        # TN: positions 2 and 5 => 2
        self.assertEqual((terms.tp, terms.fn, terms.fp, terms.tn), (2, 1, 1, 2))

    def test_metrics_multiclass_with_auc(self):
        y_true = ["NO", "<30", ">30", "NO", "<30", ">30"]
        y_pred = ["NO", "<30", "NO", "NO", "<30", ">30"]
        # Probabilities aligned with labels order ("NO", "<30", ">30")
        y_proba = np.array(
            [
                [0.8, 0.1, 0.1],
                [0.1, 0.8, 0.1],
                [0.4, 0.2, 0.4],
                [0.7, 0.2, 0.1],
                [0.1, 0.7, 0.2],
                [0.2, 0.1, 0.7],
            ],
            dtype=float,
        )
        res = mod.metrics_multiclass(y_true, y_pred, y_proba=y_proba)
        for key in ["macro_f1", "weighted_f1", "balanced_accuracy", "ovr_auc_macro"]:
            self.assertIn(key, res)
            self.assertIsInstance(res[key], float)

    def test_keep_multiclass_error_policy(self):
        s = pd.Series(["NO", "<30", "???"])
        with self.assertRaises(mod.TargetEngineeringError):
            _ = mod.keep_multiclass_readmitted(s, unknown_policy="error")

    def test_keep_multiclass_nan_policy(self):
        s = pd.Series(["NO", "<30", "???"])
        out = mod.keep_multiclass_readmitted(s, unknown_policy="nan")
        self.assertTrue(pd.isna(out.iloc[2]))


if __name__ == "__main__":
    unittest.main(verbosity=2)


Overwriting test_target_engineering.py


In [14]:
!python -m unittest -v test_target_engineering.py


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Python313\Lib\unittest\__main__.py", line 18, in <module>
    main(module=None)
    ~~~~^^^^^^^^^^^^^
  File "c:\Python313\Lib\unittest\main.py", line 103, in __init__
    self.parseArgs(argv)
    ~~~~~~~~~~~~~~^^^^^^
  File "c:\Python313\Lib\unittest\main.py", line 142, in parseArgs
    self.createTests()
    ~~~~~~~~~~~~~~~~^^
  File "c:\Python313\Lib\unittest\main.py", line 153, in createTests
    self.test = self.testLoader.loadTestsFromNames(self.testNames,
                ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
                                                   self.module)
                                                   ^^^^^^^^^^^^
  File "c:\Python313\Lib\unittest\loader.py", line 207, in loadTestsFromNames
    suites = [self.loadTestsFromName(name, module) for name in names]
              ~~~~~~~~~~~~~~~~~~~~~~^^^

In [26]:
# 2.1 Apply ID -> description mappings (recommended for interpretability)
df = df_raw.copy()

def apply_mapping(df: pd.DataFrame, id_col: str, new_col: str) -> pd.DataFrame:
    if id_col not in df.columns or id_col not in id_maps:
        return df
    map_df = id_maps[id_col]
    mapper = map_df.set_index(id_col)["description"]
    mapper.index = mapper.index.astype("string")

    keys = df[id_col].astype("string").str.strip()
    df[new_col] = keys.map(mapper)  # unknown stays <NA>
    return df

df = apply_mapping(df, "admission_type_id", "admission_type")
df = apply_mapping(df, "discharge_disposition_id", "discharge_disposition")
df = apply_mapping(df, "admission_source_id", "admission_source")

# Optionally drop the numeric ID columns to avoid duplicated information:
df = df.drop(columns=[c for c in ["admission_type_id", "discharge_disposition_id", "admission_source_id"] if c in df.columns])

# 2.2 Engineer targets (binary + optional multiclass)
df = project_code.engineer_targets(
    df,
    source_col="readmitted",
    binary_col="readmitted_30d",
    multiclass_col="readmitted_3class",
    add_multiclass=True,
    drop_source=False,
    unknown_policy="error",  # fail fast if something unexpected appears
)

# Quick sanity check
df[["readmitted", "readmitted_30d", "readmitted_3class"]].head(), df["readmitted"].value_counts()


(  readmitted  readmitted_30d readmitted_3class
 0         NO               0                NO
 1        >30               0               >30
 2         NO               0                NO
 3         NO               0                NO
 4         NO               0                NO,
 readmitted
 NO     54864
 >30    35545
 <30    11357
 Name: count, dtype: Int64)

In [27]:
%%writefile section3_dataset_understanding.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd


DEFAULT_MISSING_TOKENS: Tuple[str, ...] = (
    "", "?", "NA", "N/A", "NULL", "NAN", "UNKNOWN", "UNKNOWN/INVALID"
)

DEFAULT_ID_COLUMNS: Tuple[str, ...] = (
    "id",
    "url",
    "encounter_id",
    "patient_nbr",
    "patient_id",
    "paper_id",
)

DEFAULT_TEXT_COLUMNS: Tuple[str, ...] = (
    "title",
    "abstract",
    "text",
)

DEFAULT_INTERVAL_COLUMNS: Tuple[str, ...] = (
    "year",
)


def _is_missing_value(x: Any, *, treat_empty_as_missing: bool = True) -> bool:
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        pass
    if treat_empty_as_missing and isinstance(x, str) and x.strip() == "":
        return True
    return False


def _normalize_token(s: str) -> str:
    return s.strip().upper()


def _count_missing_like_tokens(
    series: pd.Series,
    *,
    missing_tokens: Sequence[str] = DEFAULT_MISSING_TOKENS,
) -> Dict[str, int]:
    """
    Count occurrences of missing/unknown encodings in an object/string-like column.
    Counting is case-insensitive and strips whitespace.
    """
    if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
        return {}

    toks = {_normalize_token(t) for t in missing_tokens}
    counts: Dict[str, int] = {t: 0 for t in toks}

    for v in series.astype("object").tolist():
        if v is None:
            continue
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass

        if isinstance(v, str):
            key = _normalize_token(v)
        else:
            key = _normalize_token(str(v))

        if key in counts:
            counts[key] += 1

    # Keep only tokens that actually appear (>0)
    return {k: v for k, v in counts.items() if v > 0}


def _unique_ratio(series: pd.Series) -> float:
    n = len(series)
    if n == 0:
        return 0.0
    return float(series.nunique(dropna=True)) / float(n)


def _avg_text_length(series: pd.Series) -> float:
    vals = []
    for v in series.tolist():
        if _is_missing_value(v, treat_empty_as_missing=True):
            continue
        vals.append(len(str(v)))
    if not vals:
        return 0.0
    return float(np.mean(vals))


def infer_kind_and_scale(
    series: pd.Series,
    col: str,
    *,
    text_columns: Sequence[str] = DEFAULT_TEXT_COLUMNS,
    interval_columns: Sequence[str] = DEFAULT_INTERVAL_COLUMNS,
    ordinal_columns: Sequence[str] = (),
) -> Tuple[str, str]:
    """
    Returns (kind, scale) where:
      - kind  in {"numeric","categorical","text"}
      - scale in {"nominal","ordinal","interval","ratio","text"}
    """
    col_low = col.strip().lower()

    # Forced text columns by name (common in this project)
    if col_low in {c.lower() for c in text_columns}:
        return "text", "text"

    # Numeric
    if pd.api.types.is_numeric_dtype(series.dtype) and not pd.api.types.is_bool_dtype(series.dtype):
        if col_low in {c.lower() for c in interval_columns}:
            return "numeric", "interval"
        return "numeric", "ratio"

    # Bool
    if pd.api.types.is_bool_dtype(series.dtype):
        return "categorical", "nominal"

    # Otherwise object/string => decide text vs categorical by heuristic
    avg_len = _avg_text_length(series)
    if avg_len >= 30.0:
        return "text", "text"

    # Ordinal hint by name override
    if col_low in {c.lower() for c in ordinal_columns}:
        return "categorical", "ordinal"

    return "categorical", "nominal"


def infer_role(
    df: pd.DataFrame,
    col: str,
    kind: str,
    *,
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
    derived_text_column: str = "text",
    id_like_unique_ratio_threshold: float = 0.98,
) -> str:
    """
    Returns a role label to help the report/checklist:
      - "identifier"
      - "raw_text_feature"
      - "derived_feature"
      - "metadata_feature"
      - "numeric_feature"
      - "categorical_feature"
    """
    col_low = col.strip().lower()
    s = df[col]

    if col_low in {c.lower() for c in id_columns}:
        return "identifier"

    # Name-based derived text (common: text = title + abstract)
    if col_low == derived_text_column.lower():
        return "derived_feature"

    # ID-like heuristic (very high uniqueness ratio)
    if _unique_ratio(s) >= id_like_unique_ratio_threshold and kind != "numeric":
        return "identifier"

    # Heuristic: metadata-like
    if col_low in {"year", "venue"}:
        return "metadata_feature"

    if kind == "text":
        return "raw_text_feature"
    if kind == "numeric":
        return "numeric_feature"
    return "categorical_feature"


@dataclass(frozen=True)
class DatasetUnderstandingReport:
    data_dictionary: pd.DataFrame
    missing_summary: pd.DataFrame
    missing_tokens_found: pd.DataFrame
    duplicates_summary: pd.DataFrame
    leakage_risks: pd.DataFrame
    quality_risks: pd.DataFrame


def build_data_dictionary(
    df: pd.DataFrame,
    *,
    text_columns: Sequence[str] = DEFAULT_TEXT_COLUMNS,
    interval_columns: Sequence[str] = DEFAULT_INTERVAL_COLUMNS,
    ordinal_columns: Sequence[str] = (),
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]
        kind, scale = infer_kind_and_scale(
            s,
            col,
            text_columns=text_columns,
            interval_columns=interval_columns,
            ordinal_columns=ordinal_columns,
        )
        role = infer_role(df, col, kind, id_columns=id_columns)

        miss = int(s.isna().sum())
        empty = 0
        if pd.api.types.is_object_dtype(s.dtype) or pd.api.types.is_string_dtype(s.dtype):
            empty = int((s.astype("object").map(lambda x: isinstance(x, str) and x.strip() == "")).sum())

        nun = int(s.nunique(dropna=True))
        ur = float(nun) / float(n) if n else 0.0

        # Example values (up to 5) excluding missing/empty
        examples = []
        for v in s.tolist():
            if _is_missing_value(v, treat_empty_as_missing=True):
                continue
            examples.append(v)
            if len(examples) >= 5:
                break

        rows.append(
            {
                "column": col,
                "pandas_dtype": str(s.dtype),
                "kind": kind,               # numeric / categorical / text
                "scale": scale,             # nominal / ordinal / interval / ratio / text
                "role": role,               # identifier / feature etc
                "n_rows": n,
                "n_unique": nun,
                "unique_ratio": ur,
                "missing_count": miss,
                "missing_rate": (miss / n) if n else 0.0,
                "empty_string_count": empty,
                "examples": examples,
            }
        )

    return pd.DataFrame(rows).sort_values(["role", "kind", "column"]).reset_index(drop=True)


def summarize_missingness(
    df: pd.DataFrame,
    *,
    treat_empty_as_missing: bool = True,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]
        na = int(s.isna().sum())
        empty = 0
        if treat_empty_as_missing and (pd.api.types.is_object_dtype(s.dtype) or pd.api.types.is_string_dtype(s.dtype)):
            empty = int((s.astype("object").map(lambda x: isinstance(x, str) and x.strip() == "")).sum())

        miss_total = na + empty
        rows.append(
            {
                "column": col,
                "na_count": na,
                "empty_string_count": empty,
                "missing_total": miss_total,
                "missing_rate": (miss_total / n) if n else 0.0,
            }
        )

    return pd.DataFrame(rows).sort_values("missing_rate", ascending=False).reset_index(drop=True)


def detect_missing_tokens(
    df: pd.DataFrame,
    *,
    missing_tokens: Sequence[str] = DEFAULT_MISSING_TOKENS,
) -> pd.DataFrame:
    """
    Detects tokens like '?', 'N/A', 'Unknown', etc., per column (string/object columns).
    Returns a long-form DataFrame: column, token, count
    """
    rows: List[Dict[str, Any]] = []
    for col in df.columns:
        series = df[col]
        counts = _count_missing_like_tokens(series, missing_tokens=missing_tokens)
        for token, cnt in sorted(counts.items()):
            rows.append({"column": col, "token": token, "count": int(cnt)})
    return pd.DataFrame(rows)


def duplicates_report(
    df: pd.DataFrame,
    *,
    key_columns: Sequence[str] = ("url",),
    content_columns: Sequence[str] = ("title", "abstract"),
) -> pd.DataFrame:
    """
    Returns a compact table of duplicate signals:
      - duplicates by key_columns (e.g., URL duplicates)
      - duplicates by content_columns (title+abstract duplicates)
    """
    rows: List[Dict[str, Any]] = []

    def _dup_stats(subset: Sequence[str], name: str) -> None:
        present = [c for c in subset if c in df.columns]
        if not present:
            rows.append(
                {
                    "scope": name,
                    "subset": list(subset),
                    "available_subset": [],
                    "duplicate_rows": 0,
                    "duplicate_groups": 0,
                }
            )
            return

        dup_mask = df.duplicated(subset=present, keep=False)
        duplicate_rows = int(dup_mask.sum())
        # number of duplicated keys/groups (excluding non-duplicated)
        duplicate_groups = int(df.loc[dup_mask, present].drop_duplicates().shape[0])

        rows.append(
            {
                "scope": name,
                "subset": list(subset),
                "available_subset": present,
                "duplicate_rows": duplicate_rows,
                "duplicate_groups": duplicate_groups,
            }
        )

    _dup_stats(key_columns, "key_duplicates")
    _dup_stats(content_columns, "content_duplicates")

    return pd.DataFrame(rows)


def leakage_risk_report(
    df: pd.DataFrame,
    *,
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
    id_like_unique_ratio_threshold: float = 0.98,
) -> pd.DataFrame:
    """
    Flags columns that are likely to cause leakage / trivial memorization in supervised setups
    (or trivial retrieval shortcuts in IR): identifier-like columns, near-unique columns, etc.
    """
    risks: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]
        col_low = col.strip().lower()
        ur = _unique_ratio(s)
        nun = int(s.nunique(dropna=True))

        # 1) Name-based ID columns
        if col_low in {c.lower() for c in id_columns}:
            risks.append(
                {
                    "column": col,
                    "risk_type": "identifier_like",
                    "severity": "high",
                    "reason": "Column name matches a known identifier field (e.g., 'url'/'id').",
                    "unique_ratio": ur,
                    "n_unique": nun,
                    "n_rows": n,
                }
            )
            continue

        # 2) ID-like by uniqueness ratio (very close to 1)
        if ur >= id_like_unique_ratio_threshold and not pd.api.types.is_numeric_dtype(s.dtype):
            risks.append(
                {
                    "column": col,
                    "risk_type": "identifier_like",
                    "severity": "medium",
                    "reason": f"Very high uniqueness ratio (>= {id_like_unique_ratio_threshold}).",
                    "unique_ratio": ur,
                    "n_unique": nun,
                    "n_rows": n,
                }
            )

    return pd.DataFrame(risks).sort_values(["severity", "unique_ratio"], ascending=[True, False]).reset_index(drop=True)


def quality_risk_report(
    df: pd.DataFrame,
    *,
    missing_rate_warn: float = 0.20,
    high_cardinality_warn: float = 0.50,
    treat_empty_as_missing: bool = True,
) -> pd.DataFrame:
    """
    Heuristic quality risks: high missingness, high cardinality, constant columns.
    """
    risks: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]

        # Missingness risk
        miss_rate = float(
            (_is_missing_value_count(s, treat_empty_as_missing=treat_empty_as_missing) / n) if n else 0.0
        )
        if miss_rate >= missing_rate_warn:
            risks.append(
                {
                    "column": col,
                    "risk_type": "high_missingness",
                    "severity": "medium",
                    "metric": "missing_rate",
                    "value": miss_rate,
                    "threshold": missing_rate_warn,
                    "note": "Consider explicit handling in preprocessing (drop/impute/tokenize missing).",
                }
            )

        # Cardinality risk (mostly for categorical/text columns)
        ur = float(s.nunique(dropna=True) / n) if n else 0.0
        if ur >= high_cardinality_warn and not pd.api.types.is_numeric_dtype(s.dtype):
            risks.append(
                {
                    "column": col,
                    "risk_type": "high_cardinality",
                    "severity": "low",
                    "metric": "unique_ratio",
                    "value": ur,
                    "threshold": high_cardinality_warn,
                    "note": "May lead to sparse features / overfitting if one-hot encoded.",
                }
            )

        # Constant column risk
        nun = int(s.nunique(dropna=True))
        if nun <= 1:
            risks.append(
                {
                    "column": col,
                    "risk_type": "constant_or_almost_constant",
                    "severity": "low",
                    "metric": "n_unique",
                    "value": nun,
                    "threshold": 1,
                    "note": "Usually safe to drop (no predictive signal).",
                }
            )

    return pd.DataFrame(risks).sort_values(["severity", "risk_type", "column"]).reset_index(drop=True)


def _is_missing_value_count(series: pd.Series, *, treat_empty_as_missing: bool) -> int:
    na = int(series.isna().sum())
    if not treat_empty_as_missing:
        return na
    if pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype):
        empty = int((series.astype("object").map(lambda x: isinstance(x, str) and x.strip() == "")).sum())
        return na + empty
    return na


def profile_dataset(
    df_or_records: Union[pd.DataFrame, Sequence[Dict[str, Any]]],
    *,
    expected_keys: Sequence[str] = ("title", "abstract", "url", "venue", "year"),
    text_columns: Sequence[str] = DEFAULT_TEXT_COLUMNS,
    interval_columns: Sequence[str] = DEFAULT_INTERVAL_COLUMNS,
    ordinal_columns: Sequence[str] = (),
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
    missing_tokens: Sequence[str] = DEFAULT_MISSING_TOKENS,
) -> DatasetUnderstandingReport:
    """
    End-to-end helper for Section 3:
      - Builds data dictionary (types/scales/roles)
      - Summarizes missingness + detects missing tokens
      - Detects duplicates
      - Flags leakage risks
      - Flags quality risks

    Accepts either a DataFrame or the raw list[dict] records.
    """
    if isinstance(df_or_records, pd.DataFrame):
        df = df_or_records.copy()
    else:
        # records -> dataframe
        if not isinstance(df_or_records, (list, tuple)):
            raise TypeError(f"df_or_records must be a DataFrame or list/tuple of dicts, got {type(df_or_records)}")
        rows = []
        for i, rec in enumerate(df_or_records):
            if not isinstance(rec, dict):
                raise TypeError(f"Each record must be a dict. Found {type(rec)} at index {i}.")
            rows.append({k: rec.get(k, None) for k in expected_keys})
        df = pd.DataFrame(rows)

    data_dict = build_data_dictionary(
        df,
        text_columns=text_columns,
        interval_columns=interval_columns,
        ordinal_columns=ordinal_columns,
        id_columns=id_columns,
    )
    missing_summary = summarize_missingness(df, treat_empty_as_missing=True)
    missing_tokens_found = detect_missing_tokens(df, missing_tokens=missing_tokens)
    duplicates_summary = duplicates_report(df)
    leakage_risks = leakage_risk_report(df, id_columns=id_columns)
    quality_risks = quality_risk_report(df)

    return DatasetUnderstandingReport(
        data_dictionary=data_dict,
        missing_summary=missing_summary,
        missing_tokens_found=missing_tokens_found,
        duplicates_summary=duplicates_summary,
        leakage_risks=leakage_risks,
        quality_risks=quality_risks,
    )


Overwriting section3_dataset_understanding.py


In [28]:
%%writefile test_section3_dataset_understanding.py

import unittest
import pandas as pd

import section3_dataset_understanding as s3


class TestSection3DatasetUnderstanding(unittest.TestCase):
    def _tiny_df(self):
        df = pd.DataFrame(
            {
                "title": ["Paper A", "Paper B", "Paper B"],
                "abstract": ["Text A", "Text B", "Text B"],
                "url": ["u1", "u2", "u2"],   # duplicate to test duplicate detection
                "venue": ["EMNLP", "EMNLP", "?"],  # token-like missing/unknown
                "year": [2016, 2017, 2017],
            }
        )
        df["text"] = df["title"] + " " + df["abstract"]  # derived
        return df

    def test_build_data_dictionary_infers_expected_roles_and_scales(self):
        df = self._tiny_df()
        dd = s3.build_data_dictionary(df)

        cols = set(dd["column"].tolist())
        self.assertTrue({"title", "abstract", "url", "venue", "year", "text"}.issubset(cols))

        year_row = dd.loc[dd["column"] == "year"].iloc[0]
        self.assertEqual(year_row["kind"], "numeric")
        self.assertEqual(year_row["scale"], "interval")  # year should be interval by default
        self.assertIn(year_row["role"], {"metadata_feature", "numeric_feature"})

        url_row = dd.loc[dd["column"] == "url"].iloc[0]
        self.assertEqual(url_row["role"], "identifier")

        text_row = dd.loc[dd["column"] == "text"].iloc[0]
        self.assertEqual(text_row["kind"], "text")
        self.assertEqual(text_row["role"], "derived_feature")

        title_row = dd.loc[dd["column"] == "title"].iloc[0]
        self.assertEqual(title_row["kind"], "text")
        self.assertEqual(title_row["role"], "raw_text_feature")

    def test_detect_missing_tokens_finds_question_mark(self):
        df = self._tiny_df()
        mt = s3.detect_missing_tokens(df, missing_tokens=("?", "N/A"))

        # We expect venue has "?" exactly once
        found = mt[(mt["column"] == "venue") & (mt["token"] == "?")]
        self.assertEqual(int(found["count"].iloc[0]), 1)

    def test_missing_summary_counts_empty_strings(self):
        df = self._tiny_df()
        df.loc[1, "abstract"] = "   "  # empty after strip
        ms = s3.summarize_missingness(df, treat_empty_as_missing=True)

        abs_row = ms.loc[ms["column"] == "abstract"].iloc[0]
        self.assertGreaterEqual(int(abs_row["empty_string_count"]), 1)
        self.assertGreaterEqual(int(abs_row["missing_total"]), 1)

    def test_duplicates_report_detects_key_duplicates(self):
        df = self._tiny_df()
        rep = s3.duplicates_report(df, key_columns=("url",), content_columns=("title", "abstract"))

        key_row = rep.loc[rep["scope"] == "key_duplicates"].iloc[0]
        self.assertEqual(key_row["available_subset"], ["url"])
        self.assertGreaterEqual(int(key_row["duplicate_rows"]), 2)  # u2 repeated => two rows duplicated

    def test_leakage_risk_report_flags_url_as_identifier_like(self):
        df = self._tiny_df()
        lr = s3.leakage_risk_report(df, id_columns=("url", "id"))

        self.assertTrue("column" in lr.columns)
        self.assertTrue(any(lr["column"] == "url"))

    def test_profile_dataset_accepts_records_list(self):
        records = [
            {"title": "T1", "abstract": "A1", "url": "u1", "venue": "EMNLP", "year": 2016},
            {"title": "T2", "abstract": "A2", "url": "u2", "venue": "EMNLP", "year": 2017},
        ]
        report = s3.profile_dataset(records)
        self.assertTrue(hasattr(report, "data_dictionary"))
        self.assertTrue(hasattr(report, "missing_summary"))
        self.assertTrue(hasattr(report, "duplicates_summary"))


if __name__ == "__main__":
    unittest.main(verbosity=2)


Overwriting test_section3_dataset_understanding.py


In [29]:
!python -m unittest -v test_section3_dataset_understanding.py


test_build_data_dictionary_infers_expected_roles_and_scales (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_build_data_dictionary_infers_expected_roles_and_scales) ... ok
test_detect_missing_tokens_finds_question_mark (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_detect_missing_tokens_finds_question_mark) ... ok
test_duplicates_report_detects_key_duplicates (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_duplicates_report_detects_key_duplicates) ... ok
test_leakage_risk_report_flags_url_as_identifier_like (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_leakage_risk_report_flags_url_as_identifier_like) ... ok
test_missing_summary_counts_empty_strings (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.test_missing_summary_counts_empty_strings) ... ok
test_profile_dataset_accepts_records_list (test_section3_dataset_understanding.TestSection3DatasetUnderstanding.

In [30]:
import section3_dataset_understanding as s3

# Build the report object (tables you can directly use in the PDF report)
rep = s3.profile_dataset(df)

# Display key tables
rep.data_dictionary.head(20), rep.missing_summary.head(20)
rep.leakage_risks, rep.quality_risks.head(30)


KeyboardInterrupt: 

In [20]:
!python -m pip install -q pandas numpy scikit-learn matplotlib



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
%%writefile section4_eda_imbalance.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn import metrics
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split


# ---------------------------------------------------------------------
# 4.1  Class imbalance after binarization
# ---------------------------------------------------------------------


@dataclass(frozen=True)
class ClassImbalanceSummary:
    """Compact summary of class imbalance for the binary target.

    The positive class is the early readmission (<30 days), i.e. y=1.
    This supports the discussion of why accuracy can be misleading and
    why precision/recall/F1 (for the minority class) are preferred
    under imbalance (see PDF: confusion matrix + metrics). 
    """
    N: int
    n_pos: int
    n_neg: int
    pos_rate: float
    neg_rate: float

    def as_dict(self) -> Dict[str, float]:
        return {
            "N": self.N,
            "n_pos": self.n_pos,
            "n_neg": self.n_neg,
            "pos_rate": self.pos_rate,
            "neg_rate": self.neg_rate,
        }


def compute_class_imbalance(
    y: Iterable[int],
    *,
    positive_label: int = 1,
    negative_label: int = 0,
) -> ClassImbalanceSummary:
    """Quantify class imbalance for a binary target y={0,1}.

    Parameters
    ----------
    y:
        Iterable with the binary labels (after binarization).
    positive_label:
        Label considered "positive" (early readmission).
    negative_label:
        Label considered "negative".

    Returns
    -------
    ClassImbalanceSummary
        N, n_pos, n_neg and prevalence rates.
    """
    y_series = pd.Series(list(y))
    N = int(len(y_series))
    counts = y_series.value_counts(dropna=False).to_dict()
    n_pos = int(counts.get(positive_label, 0))
    n_neg = int(counts.get(negative_label, 0))
    pos_rate = float(n_pos / N) if N else 0.0
    neg_rate = float(n_neg / N) if N else 0.0
    return ClassImbalanceSummary(N=N, n_pos=n_pos, n_neg=n_neg,
                                 pos_rate=pos_rate, neg_rate=neg_rate)


# ---------------------------------------------------------------------
# 4.2  Missingness per feature and per record
# ---------------------------------------------------------------------

MISSING_TOKENS: Tuple[str, ...] = (
    "", "?", "NA", "N/A", "NULL", "NAN", "UNKNOWN", "UNKNOWN/INVALID"
)


def is_missing(x: Any, missing_tokens: Sequence[str] = MISSING_TOKENS) -> bool:
    """Unified predicate for 'missing' used in Section 4.

    Treats as missing:
    - None / NaN (according to pandas.isna)
    - empty strings (after strip)
    - tokens like '?', 'NA', 'UNKNOWN', 'UNKNOWN/INVALID', etc.
    """
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        # If pandas cannot decide, ignore and continue
        pass

    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return True
        s_up = s.upper()
        tokens_up = {t.upper() for t in missing_tokens}
        return s_up in tokens_up

    return False


def _series_missing_mask(series: pd.Series,
                         missing_tokens: Sequence[str]) -> pd.Series:
    """Return a boolean mask indicating missing values in a Series."""
    return series.map(lambda v: is_missing(v, missing_tokens=missing_tokens))


def missingness_by_feature(
    df: pd.DataFrame,
    *,
    feature_cols: Optional[Sequence[str]] = None,
    missing_tokens: Sequence[str] = MISSING_TOKENS,
) -> pd.DataFrame:
    """Column-level missingness summary.

    Parameters
    ----------
    df:
        Full dataframe including target and features.
    feature_cols:
        Columns to treat as features. If None, all columns except common
        target columns are used.
    missing_tokens:
        Encodings to be treated as missing (in addition to NaN / empty).

    Returns
    -------
    DataFrame
        Index = column name, columns:
        - missing_rate
        - missing_count
    """
    if feature_cols is None:
        excluded = {"y", "readmitted", "readmitted_30d", "readmitted_3class"}
        feature_cols = [c for c in df.columns if c not in excluded]

    n = len(df)
    if n == 0:
        raise ValueError("DataFrame is empty; cannot summarize missingness.")

    def _col_missing_rate(col: pd.Series) -> float:
        return float(_series_missing_mask(col, missing_tokens).mean())

    missing_rate_by_col = (
        df[feature_cols]
        .apply(_col_missing_rate)
        .sort_values(ascending=False)
        .to_frame("missing_rate")
    )
    missing_rate_by_col["missing_count"] = (
        (missing_rate_by_col["missing_rate"] * n).round().astype(int)
    )
    return missing_rate_by_col


@dataclass(frozen=True)
class MissingnessByRowResult:
    """Result of row-level missingness analysis."""
    missing_per_row: pd.Series
    missing_rate_per_row: pd.Series
    summary_counts: Dict[str, float]
    high_missing_fraction: float

    def as_dict(self) -> Dict[str, Any]:
        return {
            "missing_per_row_summary": self.summary_counts,
            "high_missing_fraction": self.high_missing_fraction,
        }

def _df_elementwise_map(df: pd.DataFrame, func):
    return df.map(func) if hasattr(df, "map") else df.applymap(func)


def missingness_by_row(
    df: pd.DataFrame,
    *,
    feature_cols: Optional[Sequence[str]] = None,
    missing_tokens: Sequence[str] = MISSING_TOKENS,
    high_missing_threshold: float = 0.30,
) -> MissingnessByRowResult:
    """Row-level missingness (per encounter/patient record).

    Parameters
    ----------
    df:
        DataFrame with at least the feature columns.
    feature_cols:
        Subset of df columns to treat as features. If None, all except target.
    missing_tokens:
        Tokens to treat as missing.
    high_missing_threshold:
        Threshold on missing *rate* to consider a row 'highly incomplete'.

    Returns
    -------
    MissingnessByRowResult
        - missing_per_row: number of missing values in each row.
        - missing_rate_per_row: fraction of missing per row.
        - summary_counts: describe() of missing_per_row (as dict).
        - high_missing_fraction: proportion of rows with missing_rate >= threshold.
    """
    if feature_cols is None:
        excluded = {"y", "readmitted", "readmitted_30d", "readmitted_3class"}
        feature_cols = [c for c in df.columns if c not in excluded]

    if not feature_cols:
        raise ValueError("No feature columns provided for missingness_by_row.")

    miss_matrix = _df_elementwise_map(
        df[feature_cols],
        lambda v: is_missing(v, missing_tokens=missing_tokens),
    )
    missing_per_row = miss_matrix.sum(axis=1)
    missing_rate_per_row = missing_per_row / float(len(feature_cols))

    summary_counts = missing_per_row.describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    ).to_dict()
    high_missing_fraction = float((missing_rate_per_row >= high_missing_threshold).mean())

    return MissingnessByRowResult(
        missing_per_row=missing_per_row,
        missing_rate_per_row=missing_rate_per_row,
        summary_counts=summary_counts,
        high_missing_fraction=high_missing_fraction,
    )


# ---------------------------------------------------------------------
# 4.3  Distributions and high-cardinality categoricals
# ---------------------------------------------------------------------


def topk_table(
    series: pd.Series,
    *,
    k: int = 20,
    missing_tokens: Sequence[str] = MISSING_TOKENS,
    missing_label: str = "__MISSING__",
) -> Tuple[pd.DataFrame, int, float]:
    """Frequency table for high-cardinality categorical variables.

    Parameters
    ----------
    series:
        Categorical column (e.g., diag_1, medical_specialty, payer_code).
    k:
        Top-k categories to display.
    missing_tokens:
        Encodings considered missing.
    missing_label:
        Label used to group all missing values into a single bucket.

    Returns
    -------
    (top_df, n_unique, coverage)
        - top_df: DataFrame with columns ['count', 'rate'] for top-k items.
        - n_unique: total number of distinct categories (including missing_label).
        - coverage: fraction of rows covered by the top-k categories.
    """
    s = series.copy()
    s = s.where(~s.map(lambda v: is_missing(v, missing_tokens=missing_tokens)),
                other=missing_label)

    vc = s.value_counts(dropna=False)
    top = vc.head(k).to_frame("count")
    if len(s) > 0:
        top["rate"] = top["count"] / float(len(s))
        coverage = float(top["count"].sum() / float(len(s)))
    else:
        top["rate"] = 0.0
        coverage = 0.0

    return top, int(vc.shape[0]), coverage


# ---------------------------------------------------------------------
# 4.4  Clinically meaningful errors (FN vs FP) – dummy baseline
# ---------------------------------------------------------------------


@dataclass(frozen=True)
class DummyBaselineResult:
    """Result of a most_frequent DummyClassifier baseline.

    This baseline is meant to *illustrate* the effect of class imbalance:
    it usually predicts only the majority class, leading to apparently
    reasonable accuracy but zero recall and F1 for the minority class, as
    discussed in the PDF for imbalanced problems. 
    """
    confusion_terms: Dict[str, int]
    metrics: Dict[str, float]

    def as_dict(self) -> Dict[str, Dict[str, float]]:
        return {
            "confusion_terms": self.confusion_terms,
            "metrics": self.metrics,
        }


def run_dummy_most_frequent_baseline(
    df: pd.DataFrame,
    *,
    feature_cols: Sequence[str],
    target_col: str = "y",
    test_size: float = 0.2,
    random_state: int = 42,
) -> DummyBaselineResult:
    """Run a 'most_frequent' DummyClassifier to illustrate imbalance.

    Parameters
    ----------
    df:
        DataFrame with features and binary target.
    feature_cols:
        Columns used as input X (raw; preprocessing comes later).
    target_col:
        Name of the binary target column (e.g., 'readmitted_30d' or 'y').
    test_size:
        Fraction reserved for the illustrative test split.
    random_state:
        Seed for the train/test partition (kept fixed for reproducibility).

    Returns
    -------
    DummyBaselineResult
        Confusion-matrix terms TN/FP/FN/TP and metrics:
        accuracy, precision, recall, F1.
    """
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found in df.")

    X = df[feature_cols]
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )

    clf = DummyClassifier(strategy="most_frequent")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    tn, fp, fn, tp = metrics.confusion_matrix(y_test, y_pred).ravel()

    metrics_dict = {
        "accuracy": float(metrics.accuracy_score(y_test, y_pred)),
        "precision": float(metrics.precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(metrics.recall_score(y_test, y_pred, zero_division=0)),
        "f1": float(metrics.f1_score(y_test, y_pred, zero_division=0)),
    }

    confusion_terms = {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    return DummyBaselineResult(confusion_terms=confusion_terms, metrics=metrics_dict)


Overwriting section4_eda_imbalance.py


In [33]:
%%writefile test_section4_eda_imbalance.py
import unittest
import numpy as np
import pandas as pd

import section4_eda_imbalance as s4


class TestSection4EDAImbalance(unittest.TestCase):
    def setUp(self):
        # Pequeño dataframe con números, categóricas y target binario
        self.df = pd.DataFrame(
            {
                "feat1": [1, 2, 3, 4],
                "feat2": ["a", "?", "", "b"],       # '?' y '' se consideran missing
                "feat3": [np.nan, "x", "y", "z"],  # NaN se considera missing
                "y": [0, 0, 1, 0],
            }
        )

    # ---- 4.1: class imbalance -----------------------------------------

    def test_compute_class_imbalance_basic(self):
        summary = s4.compute_class_imbalance(self.df["y"])
        self.assertEqual(summary.N, 4)
        self.assertEqual(summary.n_pos, 1)
        self.assertEqual(summary.n_neg, 3)
        self.assertAlmostEqual(summary.pos_rate, 0.25)
        self.assertAlmostEqual(summary.neg_rate, 0.75)
        d = summary.as_dict()
        self.assertIn("n_pos", d)
        self.assertIn("pos_rate", d)

    # ---- 4.2: missingness predicates ----------------------------------

    def test_is_missing_handles_nan_and_tokens(self):
        self.assertTrue(s4.is_missing(None))
        self.assertTrue(s4.is_missing(np.nan))
        self.assertTrue(s4.is_missing("   "))          # vacío tras strip
        self.assertTrue(s4.is_missing("?"))
        self.assertTrue(s4.is_missing(" unknown "))    # token normalizado
        self.assertFalse(s4.is_missing("value"))

    def test_missingness_by_feature_rates_and_counts(self):
        mf = s4.missingness_by_feature(
            self.df,
            feature_cols=["feat1", "feat2", "feat3"],
        )
        # feat1 no tiene missing
        self.assertEqual(int(mf.loc["feat1", "missing_count"]), 0)
        self.assertAlmostEqual(float(mf.loc["feat1", "missing_rate"]), 0.0)

        # feat2: '?' y '' -> 2 de 4 => 0.5
        self.assertEqual(int(mf.loc["feat2", "missing_count"]), 2)
        self.assertAlmostEqual(float(mf.loc["feat2", "missing_rate"]), 0.5)

        # feat3: un NaN -> 1 de 4 => 0.25
        self.assertEqual(int(mf.loc["feat3", "missing_count"]), 1)
        self.assertAlmostEqual(float(mf.loc["feat3", "missing_rate"]), 0.25)

    def test_missingness_by_row_summary_and_high_missing_fraction(self):
        res = s4.missingness_by_row(
            self.df,
            feature_cols=["feat1", "feat2", "feat3"],
            high_missing_threshold=0.30,
        )

        # Comprobamos longitudes
        self.assertEqual(len(res.missing_per_row), 4)
        self.assertEqual(len(res.missing_rate_per_row), 4)

        # Cuentas esperadas por fila:
        # fila0: NaN en feat3 -> 1
        # fila1: '?' en feat2 -> 1
        # fila2: '' en feat2 -> 1
        # fila3: sin missing -> 0
        self.assertListEqual(res.missing_per_row.tolist(), [1, 1, 1, 0])

        # 3 de 4 filas con ratio >= 1/3 (~0.33) > 0.30
        self.assertAlmostEqual(res.high_missing_fraction, 0.75)

        d = res.as_dict()
        self.assertIn("missing_per_row_summary", d)
        self.assertIn("high_missing_fraction", d)

    # ---- 4.3: top-k tables for high-cardinality categoricals ----------

    def test_topk_table_collapses_missing_and_computes_coverage(self):
        top, nunique, coverage = s4.topk_table(self.df["feat2"], k=2)
        # categorías: 'a', 'b', '__MISSING__'
        self.assertEqual(nunique, 3)
        # top-2 cubre '__MISSING__'(2) + 'a'(1) = 3/4 => 0.75
        self.assertAlmostEqual(coverage, 0.75)

        # la categoría '__MISSING__' debe estar en la tabla
        self.assertIn("__MISSING__",
                      top.index.astype(str).tolist())

    # ---- 4.4: dummy baseline under imbalance --------------------------

    def test_run_dummy_most_frequent_baseline_behaviour(self):
        # Dataset más grande e imbalance más claro
        df_large = pd.DataFrame(
            {
                "f1": np.arange(50),
                "y": [0] * 40 + [1] * 10,  # 80% negativos
            }
        )

        result = s4.run_dummy_most_frequent_baseline(
            df_large,
            feature_cols=["f1"],
            target_col="y",
            test_size=0.2,
            random_state=0,
        )

        cm = result.confusion_terms
        metrics_dict = result.metrics

        # Estrategia 'most_frequent' debe predecir solo la clase mayoritaria (0)
        self.assertEqual(cm["tp"], 0)
        self.assertEqual(cm["fp"], 0)
        self.assertGreater(cm["tn"], 0)
        self.assertGreater(cm["fn"], 0)

        # Bajo desequilibrio: accuracy > 0, pero recall y F1 del positivo = 0
        self.assertGreater(metrics_dict["accuracy"], 0.0)
        self.assertAlmostEqual(metrics_dict["recall"], 0.0)
        self.assertAlmostEqual(metrics_dict["f1"], 0.0)

        d = result.as_dict()
        self.assertIn("confusion_terms", d)
        self.assertIn("metrics", d)


if __name__ == "__main__":
    unittest.main(verbosity=2)


Overwriting test_section4_eda_imbalance.py


In [23]:
import section4_eda_imbalance as s4

# 4.1 Class imbalance (binary)
imb = s4.compute_class_imbalance(df["readmitted_30d"].astype(int))
imb.as_dict()


{'N': 101766,
 'n_pos': 11357,
 'n_neg': 90409,
 'pos_rate': 0.11159915885462728,
 'neg_rate': 0.8884008411453728}

In [24]:
# 4.2 Missingness by feature (using the unified missing-token logic)
mf = s4.missingness_by_feature(df)
mf.head(25)


,missing_rate,missing_count
weight,0.968585,98569
max_glu_serum,0.947468,96420
A1Cresult,0.832773,84748
medical_specialty,0.490822,49949
payer_code,0.395574,40256
admission_source,0.066633,6781
admission_type,0.051992,5291
discharge_disposition,0.036269,3691
race,0.022336,2273
diag_3,0.013983,1423


In [25]:
# 4.2 Missingness by row (how incomplete are records?)
mr = s4.missingness_by_row(df, high_missing_threshold=0.30)
mr.as_dict()


c:\Users\Cris-SX\Desktop\UA\CHAD-MASTER-IA\TAU\tau-final-project\code\lvl1\lvl2\tau-final-code\section4_eda_imbalance.py:217: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  miss_matrix = df[feature_cols].applymap(


{'missing_per_row_summary': {'count': 101766.0,
  'mean': 3.830188864650276,
  'std': 0.9344269760640691,
  'min': 1.0,
  '50%': 4.0,
  '75%': 4.0,
  '90%': 5.0,
  '95%': 5.0,
  '99%': 6.0,
  'max': 9.0},
 'high_missing_fraction': 0.0}

In [26]:
# 4.3 Top-K category distributions for known high-cardinality columns
for col in ["medical_specialty", "diag_1", "diag_2", "diag_3", "payer_code"]:
    if col in df.columns:
        top, nunique, coverage = s4.topk_table(df[col], k=15)
        print(f"\n=== {col} | nunique={nunique} | top15 coverage={coverage:.3f} ===")
        display(top)



=== medical_specialty | nunique=73 | top15 coverage=0.955 ===


,count,rate
medical_specialty,,
__MISSING__,49949,0.490822
InternalMedicine,14635,0.14381
Emergency/Trauma,7565,0.074337
Family/GeneralPractice,7440,0.073109
Cardiology,5352,0.052591
Surgery-General,3099,0.030452
Nephrology,1613,0.01585
Orthopedics,1400,0.013757
Orthopedics-Reconstructive,1233,0.012116



=== diag_1 | nunique=717 | top15 coverage=0.443 ===


,count,rate
diag_1,,
428,6862,0.067429
414,6581,0.064668
786,4016,0.039463
410,3614,0.035513
486,3508,0.034471
427,2766,0.02718
491,2275,0.022355
715,2151,0.021137
682,2042,0.020066



=== diag_2 | nunique=749 | top15 coverage=0.511 ===


,count,rate
diag_2,,
276,6752,0.066348
428,6662,0.065464
250,6071,0.059656
427,5036,0.049486
401,3736,0.036712
496,3305,0.032476
599,3288,0.032309
403,2823,0.02774
414,2650,0.02604



=== diag_3 | nunique=790 | top15 coverage=0.527 ===


,count,rate
diag_3,,
250,11555,0.113545
401,8289,0.081452
276,5175,0.050852
428,4577,0.044976
427,3955,0.038864
414,3664,0.036004
496,2605,0.025598
403,2357,0.023161
585,1992,0.019574



=== payer_code | nunique=18 | top15 coverage=0.999 ===


,count,rate
payer_code,,
__MISSING__,40256,0.395574
MC,32439,0.318761
HM,6274,0.061651
SP,5007,0.049201
BC,4655,0.045742
MD,3532,0.034707
CP,2533,0.02489
UN,2448,0.024055
CM,1937,0.019034


In [27]:
# 4.4 Dummy "most_frequent" baseline on a simple train/test split (illustrative only)
excluded = {"readmitted", "readmitted_30d", "readmitted_3class"}
feature_cols = [c for c in df.columns if c not in excluded]

dummy_res = s4.run_dummy_most_frequent_baseline(
    df,
    feature_cols=feature_cols,
    target_col="readmitted_30d",
    test_size=0.2,
    random_state=42,
)
dummy_res.as_dict()


{'confusion_terms': {'tn': 18083, 'fp': 0, 'fn': 2271, 'tp': 0},
 'metrics': {'accuracy': 0.8884248796305394,
  'precision': 0.0,
  'recall': 0.0,
  'f1': 0.0}}

In [34]:
!python -m unittest -v test_section4_eda_imbalance.py


test_compute_class_imbalance_basic (test_section4_eda_imbalance.TestSection4EDAImbalance.test_compute_class_imbalance_basic) ... ok
test_is_missing_handles_nan_and_tokens (test_section4_eda_imbalance.TestSection4EDAImbalance.test_is_missing_handles_nan_and_tokens) ... ok
test_missingness_by_feature_rates_and_counts (test_section4_eda_imbalance.TestSection4EDAImbalance.test_missingness_by_feature_rates_and_counts) ... ok
test_missingness_by_row_summary_and_high_missing_fraction (test_section4_eda_imbalance.TestSection4EDAImbalance.test_missingness_by_row_summary_and_high_missing_fraction) ... ok
test_run_dummy_most_frequent_baseline_behaviour (test_section4_eda_imbalance.TestSection4EDAImbalance.test_run_dummy_most_frequent_baseline_behaviour) ... ok
test_topk_table_collapses_missing_and_computes_coverage (test_section4_eda_imbalance.TestSection4EDAImbalance.test_topk_table_collapses_missing_and_computes_coverage) ... ok

-----------------------------------------------------------------

In [35]:
%%writefile section5_preprocessing_pipeline.py

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

import math

# ---------------------------------------------------------------------
# Section 5 (PDF-aligned):
# - Numeric values: scaling/normalization (StandardScaler / MinMaxScaler)
# - Categorical values: binarization (dummies / one-hot)
# - Missing/unknown: remove variable/sample or impute (Simple/KNN/Iterative)
# - No leakage: all learned steps must be fit inside CV folds via Pipeline
# ---------------------------------------------------------------------

DEFAULT_MISSING_TOKENS: Tuple[str, ...] = (
    "",
    "?",
    "NA",
    "N/A",
    "NULL",
    "NAN",
    "UNKNOWN",
    "UNKNOWN/INVALID",
)

DEFAULT_ID_COLS: Tuple[str, ...] = ("encounter_id", "patient_nbr")
DEFAULT_TARGET_COLS: Tuple[str, ...] = ("readmitted", "readmitted_30d", "readmitted_3class", "y")

# Ordinal default: only AGE is treated as ordinal (safe + clinically meaningful order).
AGE_ORDER: Tuple[str, ...] = (
    "[0-10)",
    "[10-20)",
    "[20-30)",
    "[30-40)",
    "[40-50)",
    "[50-60)",
    "[60-70)",
    "[70-80)",
    "[80-90)",
    "[90-100)",
)

# Integer-coded fields that are nominal categories (avoid imposing fake geometry)
DEFAULT_FORCE_CATEGORICAL: Tuple[str, ...] = (
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
)

# Count-like numeric fields (plus weight if present)
DEFAULT_NUMERIC_HINTS: Tuple[str, ...] = (
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "weight",
)


def _normalize_token(x: str) -> str:
    return x.strip().upper()


def _missing_token_set(tokens: Sequence[str]) -> set:
    return {_normalize_token(t) for t in tokens}

class MissingTokenCleaner(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        missing_tokens=("?", "UNKNOWN/INVALID", ""),
        *,
        case_insensitive: bool = True,
        strip: bool = True,
        treat_empty_as_missing: bool = True,
    ):
        self.missing_tokens = tuple(missing_tokens)
        self.case_insensitive = case_insensitive
        self.strip = strip
        self.treat_empty_as_missing = treat_empty_as_missing

    def fit(self, X, y=None):
        toks = []
        for t in self.missing_tokens:
            s = str(t)
            s = s.strip() if self.strip else s
            s = s.upper() if self.case_insensitive else s
            toks.append(s)
        self._missing_set_ = set(toks)
        return self

    def transform(self, X):
        # Soporta DataFrame y arrays
        if isinstance(X, pd.DataFrame):
            out = X.copy()
            missing_set = getattr(self, "_missing_set_", None)
            if missing_set is None:
                self.fit(X)
                missing_set = self._missing_set_

            for c in out.columns:
                s = out[c]
                if not (pd.api.types.is_object_dtype(s.dtype) or pd.api.types.is_string_dtype(s.dtype)):
                    continue

                def _clean(v):
                    if v is None:
                        return np.nan
                    try:
                        if pd.isna(v):
                            return np.nan
                    except Exception:
                        pass

                    if isinstance(v, str):
                        vv = v.strip() if self.strip else v
                        if self.treat_empty_as_missing and vv == "":
                            return np.nan
                        key = vv.upper() if self.case_insensitive else vv
                        return np.nan if key in missing_set else vv

                    vv = str(v)
                    vv = vv.strip() if self.strip else vv
                    key = vv.upper() if self.case_insensitive else vv
                    return np.nan if key in missing_set else v

                out[c] = s.map(_clean)

            return out

        # array-like (n_samples, n_features)
        arr = np.asarray(X, dtype=object)
        missing_set = getattr(self, "_missing_set_", None)
        if missing_set is None:
            # fit “perezoso”
            self.fit(pd.DataFrame(arr))
            missing_set = self._missing_set_

        def _clean_scalar(v):
            if v is None:
                return np.nan
            try:
                if pd.isna(v):
                    return np.nan
            except Exception:
                pass
            if isinstance(v, str):
                vv = v.strip() if self.strip else v
                if self.treat_empty_as_missing and vv == "":
                    return np.nan
                key = vv.upper() if self.case_insensitive else vv
                return np.nan if key in missing_set else vv
            vv = str(v)
            vv = vv.strip() if self.strip else vv
            key = vv.upper() if self.case_insensitive else vv
            return np.nan if key in missing_set else v

        vfunc = np.vectorize(_clean_scalar, otypes=[object])
        return vfunc(arr)



class NumericCoercer(BaseEstimator, TransformerMixin):
    """Coerce selected columns to numeric (float), invalid parsing -> NaN.
    """

    def __init__(self, numeric_cols: Sequence[str]):
        self.numeric_cols = numeric_cols  # DO NOT copy/convert (clone compatibility)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            return X
        df = X.copy()
        for col in self.numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        return df

class CategoricalCaster(BaseEstimator, TransformerMixin):
    """
    Cast categorical values to strings (keeping missing as np.nan).
    Prevents mixed types (e.g., int-coded categories + string missing filler).
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_arr = self._to_2d_array(X).astype("object", copy=True)
        out = np.empty_like(X_arr, dtype="object")

        for i in range(X_arr.shape[0]):
            for j in range(X_arr.shape[1]):
                v = X_arr[i, j]
                if v is None:
                    out[i, j] = np.nan
                    continue
                try:
                    if pd.isna(v):
                        out[i, j] = np.nan
                        continue
                except Exception:
                    pass
                out[i, j] = str(v)

        return out

    @staticmethod
    def _to_2d_array(X):
        if isinstance(X, pd.DataFrame):
            return X.to_numpy(dtype="object")
        if isinstance(X, pd.Series):
            return X.to_frame().to_numpy(dtype="object")
        X_arr = np.asarray(X, dtype="object")
        if X_arr.ndim == 1:
            X_arr = X_arr.reshape(-1, 1)
        return X_arr
from sklearn.base import BaseEstimator, TransformerMixin

class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        min_count: int = 10,
        *,
        min_frequency: float | None = None,
        other_label: str = "__OTHER__",
        missing_label: str = "__MISSING__",
    ):
        self.min_count = int(min_count)
        self.min_frequency = min_frequency
        self.other_label = other_label
        self.missing_label = missing_label

    def fit(self, X, y=None):
        arr = np.asarray(X, dtype=object)
        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)

        n_rows = arr.shape[0]

        # If min_frequency is provided, convert it to an absolute threshold per fit()
        freq_count = 0
        if self.min_frequency is not None:
            if not (0.0 < float(self.min_frequency) <= 1.0):
                raise ValueError(f"min_frequency must be in (0,1], got {self.min_frequency}")
            freq_count = int(math.ceil(float(self.min_frequency) * n_rows))

        effective_min = max(1, self.min_count, freq_count)

        allowed = []
        for j in range(arr.shape[1]):
            col = arr[:, j]

            col2 = np.array(
                [
                    self.missing_label
                    if (v is None or pd.isna(v))
                    else v
                    for v in col
                ],
                dtype=object,
            )

            vc = pd.Series(col2).value_counts(dropna=False)
            keep = set(vc[vc >= effective_min].index.tolist())

            # Always allow missing bucket
            keep.add(self.missing_label)

            allowed.append(keep)

        self.allowed_categories_ = allowed
        self.effective_min_count_ = effective_min
        return self

    def transform(self, X):
        arr = np.asarray(X, dtype=object)
        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)

        if not hasattr(self, "allowed_categories_"):
            self.fit(arr)

        out = arr.copy()
        for j in range(out.shape[1]):
            keep = self.allowed_categories_[j]

            def _map(v):
                if v is None or pd.isna(v):
                    return self.missing_label
                if v == self.missing_label:
                    return self.missing_label
                return v if v in keep else self.other_label

            out[:, j] = np.vectorize(_map, otypes=[object])(out[:, j])

        return out

class ToNumeric(BaseEstimator, TransformerMixin):
    """
    Convert input columns to numeric (float) using pandas.to_numeric(errors='coerce').
    This is mandatory for numeric-like string columns (e.g., 'weight' = "70") so that
    median imputation + scaling work reliably.
    """

    def __init__(self, dtype: str = "float64"):
        self.dtype = dtype

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # DataFrame path (best case)
        if isinstance(X, pd.DataFrame):
            return X.apply(pd.to_numeric, errors="coerce").to_numpy(dtype=self.dtype)

        # numpy / array-like path
        arr = np.asarray(X, dtype=object)
        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)

        out = np.empty(arr.shape, dtype=self.dtype)
        for j in range(arr.shape[1]):
            out[:, j] = pd.to_numeric(pd.Series(arr[:, j]), errors="coerce").to_numpy(dtype=self.dtype)
        return out

def make_onehot_encoder(*, sparse_output: bool):
    # sklearn >= 1.2 uses sparse_output; older uses sparse
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=sparse_output)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=sparse_output)

def _make_one_hot_encoder(*, sparse: bool, handle_unknown: str = "ignore") -> OneHotEncoder:
    # sklearn compatibility (sparse vs sparse_output)
    try:
        return OneHotEncoder(handle_unknown=handle_unknown, sparse_output=sparse)
    except TypeError:
        return OneHotEncoder(handle_unknown=handle_unknown, sparse=sparse)


def _make_ordinal_encoder(categories: List[List[str]]) -> OrdinalEncoder:
    # sklearn compatibility: handle_unknown might not exist in older versions
    try:
        return OrdinalEncoder(
            categories=categories,
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )
    except TypeError:
        return OrdinalEncoder(categories=categories)


@dataclass(frozen=True)
class DiabetesPreprocessConfig:
    # Missingness tokens and core columns
    missing_tokens: Tuple[str, ...] = DEFAULT_MISSING_TOKENS
    id_cols: Tuple[str, ...] = DEFAULT_ID_COLS
    target_cols: Tuple[str, ...] = DEFAULT_TARGET_COLS

    # Ordinal specs
    ordinal_cols: Mapping[str, Sequence[str]] = field(default_factory=lambda: {"age": AGE_ORDER})

    # Force integer-coded categoricals
    force_categorical_cols: Tuple[str, ...] = DEFAULT_FORCE_CATEGORICAL

    # Numeric imputation + scaling
    numeric_impute_strategy: str = "median"   # PDF: mean/median are standard
    scale_numeric: bool = True               # PDF: scaling/normalization recommended for numeric for many models
    scaler: str = "standard"                 # 'standard' or 'minmax' (PDF mentions both)

    # Categorical handling
    cat_impute_strategy: str = "constant"    # constant -> explicit "__MISSING__" bucket
    cat_impute_fill_value: str = "__MISSING__"
    rare_min_count: int = 50
    rare_min_frequency: Optional[float] = None
    rare_other_label: str = "__OTHER__"

    # Output format
    onehot_sparse: bool = True


@dataclass(frozen=True)
class FeatureGroups:
    numeric: List[str]
    ordinal: List[str]
    categorical: List[str]
    dropped: List[str]


def infer_feature_groups(df: pd.DataFrame, config: DiabetesPreprocessConfig) -> FeatureGroups:
    cols = list(df.columns)

    id_set = set(config.id_cols)
    target_set = set(config.target_cols)

    dropped = [c for c in cols if c in id_set]
    usable = [c for c in cols if c not in id_set and c not in target_set]

    ordinal = [c for c in usable if c in set(config.ordinal_cols.keys())]
    remaining = [c for c in usable if c not in set(ordinal)]

    force_cat = set(config.force_categorical_cols)

    # Numeric:
    # (1) known count-like fields present
    # (2) detected numeric dtypes, excluding forced categoricals
    numeric: List[str] = [c for c in remaining if c in set(DEFAULT_NUMERIC_HINTS)]
    numeric_set = set(numeric)

    for c in remaining:
        if c in numeric_set or c in force_cat:
            continue
        if pd.api.types.is_numeric_dtype(df[c].dtype) and not pd.api.types.is_bool_dtype(df[c].dtype):
            numeric.append(c)
            numeric_set.add(c)

    # Everything else -> categorical (includes forced categoricals)
    categorical = [c for c in remaining if c not in numeric_set]
    return FeatureGroups(
        numeric=list(numeric),
        ordinal=list(ordinal),
        categorical=list(categorical),
        dropped=list(dropped),
    )


def build_preprocessor(df: pd.DataFrame, config: Optional[DiabetesPreprocessConfig] = None) -> Pipeline:
    """
    Single-Source-of-Truth preprocessing pipeline.

    Guarantees:
    - Unified missingness: tokens -> NaN
    - Categorical: one-hot (handle_unknown='ignore') + optional rare-category grouping
    - Numeric: median imputation + (optional) scaling/normalization
    - Ordinal: explicit ordered encoding (age)
    - Leakage-safe when used inside CV: all learned steps are fit on training folds only.
    """
    if config is None:
        config = DiabetesPreprocessConfig()

    groups = infer_feature_groups(df, config)

    # Numeric pipeline
    num_steps = [
        ("to_numeric", ToNumeric()),  # <<< AQUÍ VA
        ("imputer", SimpleImputer(strategy=config.numeric_impute_strategy)),
    ]

    if config.scale_numeric:
        scaler_name = config.scaler.lower()
        if scaler_name == "standard":
            num_steps.append(("scaler", StandardScaler()))
        elif scaler_name == "minmax":
            num_steps.append(("scaler", MinMaxScaler()))
        else:
            raise ValueError(f"Unknown scaler: {config.scaler}. Use 'standard' or 'minmax'.")

    num_pipe = Pipeline(steps=num_steps)

    # Ordinal pipeline (age)
    if groups.ordinal:
        categories: List[List[str]] = [list(config.ordinal_cols[c]) for c in groups.ordinal]
        ord_pipe = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value=config.cat_impute_fill_value)),
                ("ordinal", _make_ordinal_encoder(categories=categories)),
            ]
        )
    else:
        ord_pipe = "drop"

    # Categorical pipeline
    if config.cat_impute_strategy == "constant":
        cat_imputer = SimpleImputer(strategy="constant", fill_value=config.cat_impute_fill_value)
    elif config.cat_impute_strategy == "most_frequent":
        cat_imputer = SimpleImputer(strategy="most_frequent")
    else:
        raise ValueError("cat_impute_strategy must be 'constant' or 'most_frequent'.")

    onehot = make_onehot_encoder(sparse_output=config.onehot_sparse)

    cat_pipe = Pipeline(
        steps=[
            ("cast_to_str", CategoricalCaster()),
            ("imputer", cat_imputer),
            ("rare", RareCategoryGrouper(
                min_count=config.rare_min_count,
                min_frequency=config.rare_min_frequency,
                other_label=config.rare_other_label,
                missing_label=config.cat_impute_fill_value,
            )),
            ("onehot", onehot),
        ]
    )

    transformers = []
    if groups.numeric:
        transformers.append(("num", num_pipe, groups.numeric))
    if groups.ordinal:
        transformers.append(("ord", ord_pipe, groups.ordinal))
    if groups.categorical:
        transformers.append(("cat", cat_pipe, groups.categorical))

    ct = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    sparse_threshold=0.0 if not config.onehot_sparse else 0.3,
)

    return Pipeline(
        steps=[
            ("clean_missing_tokens", MissingTokenCleaner(missing_tokens=config.missing_tokens)),
            ("coerce_numeric", NumericCoercer(numeric_cols=groups.numeric)),
            ("features", ct),
        ]
    )


Overwriting section5_preprocessing_pipeline.py


In [36]:
%%writefile test_section5_preprocessing_pipeline.py

import unittest

import math


import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin


import section5_preprocessing_pipeline as s5


class TestSection5PreprocessingPipeline(unittest.TestCase):
    def _toy_df(self):
        # Mini dataset that matches the real schema patterns:
        # - ID columns to drop
        # - ordinal age
        # - numeric counts + numeric-like strings (weight)
        # - int-coded categorical (admission_type_id)
        # - missing tokens ("?", "Unknown/Invalid")
        # - rare categorical values
        df = pd.DataFrame(
            {
                "encounter_id": [1, 2, 3, 4, 5, 6],
                "patient_nbr": [10, 11, 12, 13, 14, 15],
                "age": ["[0-10)", "[10-20)", "?", "[30-40)", "[40-50)", "[50-60)"],
                "weight": ["?", "80", "90", "?", "Unknown/Invalid", "70"],
                "num_lab_procedures": [41, 59, 11, 44, np.nan, 50],
                "admission_type_id": [6, 1, 1, 1, 6, 6],
                "medical_specialty": ["Pediatrics-Endocrinology", "?", "Cardiology", "Cardiology", "RareSpec", "Cardiology"],
                "diag_1": ["250.83", "276", "648", "8", "777", "8"],
                "readmitted_30d": [0, 1, 0, 0, 1, 0],
            }
        )
        return df

    def test_missing_token_cleaner_replaces_tokens_with_nan(self):
        df = self._toy_df()
        cleaner = s5.MissingTokenCleaner(missing_tokens=("?", "UNKNOWN/INVALID"))
        out = cleaner.transform(df[["medical_specialty", "weight", "age"]])

        # '?' and 'Unknown/Invalid' should become NaN
        self.assertTrue(pd.isna(out.loc[1, "medical_specialty"]))
        self.assertTrue(pd.isna(out.loc[0, "weight"]))
        self.assertTrue(pd.isna(out.loc[4, "weight"]))
        self.assertTrue(pd.isna(out.loc[2, "age"]))

    def test_infer_feature_groups_expected(self):
        df = self._toy_df()
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=False, rare_min_count=2)

        groups = s5.infer_feature_groups(df, cfg)

        # IDs dropped
        self.assertIn("encounter_id", groups.dropped)
        self.assertIn("patient_nbr", groups.dropped)

        # age is ordinal
        self.assertIn("age", groups.ordinal)

        # numeric includes weight + num_lab_procedures
        self.assertIn("weight", groups.numeric)
        self.assertIn("num_lab_procedures", groups.numeric)

        # admission_type_id forced categorical (not numeric)
        self.assertIn("admission_type_id", groups.categorical)
        self.assertNotIn("admission_type_id", groups.numeric)

    def test_preprocessor_fit_transform_no_crash_and_no_nan(self):
        df = self._toy_df()
        X = df.drop(columns=["readmitted_30d"])
        y = df["readmitted_30d"]

        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=False, rare_min_count=2)
        pre = s5.build_preprocessor(df, cfg)

        pre.fit(X, y)
        Xt = pre.transform(X)

        self.assertEqual(Xt.shape[0], len(df))
        # After imputation/encoding there should be no NaNs (Ordinal unknown becomes -1, not NaN)
        self.assertFalse(np.isnan(np.asarray(Xt)).any())

    def test_unseen_category_does_not_break_transform(self):
        df = self._toy_df()
        train = df.iloc[:4].copy()
        test = df.iloc[4:].copy()

        X_train = train.drop(columns=["readmitted_30d"])
        y_train = train["readmitted_30d"]
        X_test = test.drop(columns=["readmitted_30d"])

        # Force rare grouping to be aggressive, so unseen/rare go to __OTHER__
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=False, rare_min_count=2)
        pre = s5.build_preprocessor(train, cfg)

        pre.fit(X_train, y_train)
        _ = pre.transform(X_test)  # should not raise

        self.assertTrue(True)

    def test_rare_category_grouper_maps_rare_to_other(self):
        X = np.array(
            [
                ["A"],
                ["A"],
                ["B"],  # rare if min_count=2
                ["__MISSING__"],
            ],
            dtype=object,
        )
        g = s5.RareCategoryGrouper(min_count=2, other_label="__OTHER__", missing_label="__MISSING__")
        g.fit(X)
        Xt = g.transform(np.array([["B"], ["A"], ["C"], [np.nan]], dtype=object))

        # B is rare -> __OTHER__; C unseen -> __OTHER__; nan -> __MISSING__
        self.assertEqual(Xt[0, 0], "__OTHER__")
        self.assertEqual(Xt[1, 0], "A")
        self.assertEqual(Xt[2, 0], "__OTHER__")
        self.assertEqual(Xt[3, 0], "__MISSING__")

    def test_pipeline_works_inside_cross_validation(self):
        df = self._toy_df()
        X = df.drop(columns=["readmitted_30d"])
        y = df["readmitted_30d"]

        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=False, rare_min_count=2)
        pre = s5.build_preprocessor(df, cfg)

        pipe = Pipeline(
            steps=[
                ("pre", pre),
                ("clf", LogisticRegression(max_iter=1000)),
            ]
        )

        # ROC-AUC requires BOTH classes in every test fold.
        # With tiny toy data, choose n_splits <= min(class_count) to avoid undefined AUC.
        counts = pd.Series(y).value_counts()
        min_class = int(counts.min())
        if min_class < 2:
            self.skipTest("Need at least 2 samples per class to compute ROC-AUC in CV.")

        from sklearn.model_selection import StratifiedKFold

        n_splits = min(3, min_class)  # ensures each fold has at least 1 sample of each class
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)

        scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
        self.assertEqual(len(scores), n_splits)
        self.assertTrue(np.isfinite(scores).all())



Overwriting test_section5_preprocessing_pipeline.py


In [37]:
!python -m unittest -v test_section5_preprocessing_pipeline.py 


test_infer_feature_groups_expected (test_section5_preprocessing_pipeline.TestSection5PreprocessingPipeline.test_infer_feature_groups_expected) ... ok
test_missing_token_cleaner_replaces_tokens_with_nan (test_section5_preprocessing_pipeline.TestSection5PreprocessingPipeline.test_missing_token_cleaner_replaces_tokens_with_nan) ... ok
test_pipeline_works_inside_cross_validation (test_section5_preprocessing_pipeline.TestSection5PreprocessingPipeline.test_pipeline_works_inside_cross_validation) ... ok
test_preprocessor_fit_transform_no_crash_and_no_nan (test_section5_preprocessing_pipeline.TestSection5PreprocessingPipeline.test_preprocessor_fit_transform_no_crash_and_no_nan) ... ok
test_rare_category_grouper_maps_rare_to_other (test_section5_preprocessing_pipeline.TestSection5PreprocessingPipeline.test_rare_category_grouper_maps_rare_to_other) ... ok
test_unseen_category_does_not_break_transform (test_section5_preprocessing_pipeline.TestSection5PreprocessingPipeline.test_unseen_category_doe

In [32]:
import section5_preprocessing_pipeline as s5

cfg = s5.DiabetesPreprocessConfig(
    onehot_sparse=True,     # recommended for ~100k rows + many categoricals
    rare_min_count=50,      # group very rare categories into __OTHER__
)

groups = s5.infer_feature_groups(df, cfg)
groups


FeatureGroups(numeric=['weight', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses'], ordinal=['age'], categorical=['race', 'gender', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'admission_type', 'discharge_disposition', 'admission_source'], dropped=['encounter_id', 'patient_nbr'])

In [33]:
pre = s5.build_preprocessor(df, cfg)

X = df.drop(columns=[c for c in ["readmitted", "readmitted_30d", "readmitted_3class"] if c in df.columns])
y = df["readmitted_30d"].astype(int)

pre.fit(X, y)
Xt = pre.transform(X)

type(Xt), Xt.shape


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


(scipy.sparse._csr.csr_matrix, (101766, 786))

In [38]:
%%writefile section6_baseline_system.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

import section5_preprocessing_pipeline as s5


# ---------------------------------------------------------------------
# Section 6 — Baseline System (DummyClassifier sanity-check)
# ---------------------------------------------------------------------
# Required by the practice:
#   - Baseline ROC-AUC from a DummyClassifier
# Also report clinically interpretable metrics:
#   - Precision / Recall / F1 (minority class = 1)
#
# We evaluate using Stratified K-Fold CV (same philosophy as later sections),
# and we keep random_state fixed for reproducibility and fair comparisons.
# ---------------------------------------------------------------------


@dataclass(frozen=True)
class BaselineCVResult:
    strategy: str
    n_splits: int
    fold_scores: Dict[str, np.ndarray]
    mean_scores: Dict[str, float]
    std_scores: Dict[str, float]

    def as_row(self) -> Dict[str, float]:
        row: Dict[str, float] = {"strategy": self.strategy, "n_splits": float(self.n_splits)}
        for k, v in self.mean_scores.items():
            row[f"{k}_mean"] = float(v)
        for k, v in self.std_scores.items():
            row[f"{k}_std"] = float(v)
        return row


def _validate_binary_target(y: pd.Series) -> pd.Series:
    if y.isna().any():
        raise ValueError("Target contains missing values; baseline evaluation requires a clean binary target.")
    uniq = set(pd.Series(y).astype(int).unique().tolist())
    if not uniq.issubset({0, 1}):
        raise ValueError(f"Target must be binary in {{0,1}}, found values: {sorted(list(uniq))}")
    return pd.Series(y).astype(int)


def _default_feature_frame(df: pd.DataFrame, *, target_col: str, config: s5.DiabetesPreprocessConfig) -> pd.DataFrame:
    drop_cols = [c for c in config.target_cols if c in df.columns]
    if target_col in df.columns and target_col not in drop_cols:
        drop_cols.append(target_col)
    X = df.drop(columns=drop_cols, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError("No feature columns available after dropping target columns.")
    return X


def _scorers_binary() -> Dict[str, object]:
    # zero_division=0 avoids warnings when a baseline predicts no positives
    return {
        "roc_auc": "roc_auc",
        "accuracy": make_scorer(accuracy_score),
        "precision": make_scorer(precision_score, zero_division=0),
        "recall": make_scorer(recall_score, zero_division=0),
        "f1": make_scorer(f1_score, zero_division=0),
    }

def evaluate_dummy_baseline_cv(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    strategy: str = "most_frequent",
    n_splits: int = 10,
    cv_random_state: int = 42,
    dummy_random_state: int = 42,
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    cv_splits: Optional[Sequence[Tuple[np.ndarray, np.ndarray]]] = None,
) -> BaselineCVResult:
    """
    Evaluate a DummyClassifier baseline under Stratified K-Fold CV.

    NEW (Section 7 requirement):
      - If cv_splits is provided, those exact folds are used (identical partitions across experiments).
      - Otherwise, a StratifiedKFold is created as before.
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found. Available: {list(df.columns)}")

    y = _validate_binary_target(df[target_col])
    X = _default_feature_frame(df, target_col=target_col, config=preprocess_config)

    pre = s5.build_preprocessor(df, preprocess_config)

    clf = DummyClassifier(strategy=strategy, random_state=dummy_random_state)
    pipe = Pipeline(steps=[("pre", pre), ("dummy", clf)])

    if cv_splits is not None:
        cv_used = list(cv_splits)
        n_splits_effective = len(cv_used)
    else:
        cv_used = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=cv_random_state)
        n_splits_effective = n_splits

    scoring = _scorers_binary()

    out = cross_validate(
        pipe,
        X,
        y,
        cv=cv_used,
        scoring=scoring,
        return_train_score=False,
        n_jobs=None,
    )

    fold_scores: Dict[str, np.ndarray] = {}
    for key in scoring.keys():
        fold_scores[key] = np.asarray(out[f"test_{key}"], dtype=float)

    mean_scores = {k: float(np.mean(v)) for k, v in fold_scores.items()}
    std_scores = {k: float(np.std(v, ddof=1)) if len(v) > 1 else 0.0 for k, v in fold_scores.items()}

    return BaselineCVResult(
        strategy=strategy,
        n_splits=n_splits_effective,
        fold_scores=fold_scores,
        mean_scores=mean_scores,
        std_scores=std_scores,
    )


def run_dummy_baselines(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    strategies: Sequence[str] = ("most_frequent", "stratified"),
    n_splits: int = 10,
    cv_random_state: int = 42,
    dummy_random_state: int = 42,
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    cv_splits: Optional[Sequence[Tuple[np.ndarray, np.ndarray]]] = None,
) -> pd.DataFrame:
    """
    Convenience helper: evaluate multiple DummyClassifier baselines
    and return a compact results table (mean ± std across folds).

    NEW:
      - cv_splits can be passed to force identical folds across baselines and later models.
    """
    rows: List[Dict[str, float]] = []
    for strat in strategies:
        res = evaluate_dummy_baseline_cv(
            df,
            target_col=target_col,
            strategy=strat,
            n_splits=n_splits,
            cv_random_state=cv_random_state,
            dummy_random_state=dummy_random_state,
            preprocess_config=preprocess_config,
            cv_splits=cv_splits,
        )
        rows.append(res.as_row())

    table = pd.DataFrame(rows)
    preferred = [
        "strategy",
        "n_splits",
        "roc_auc_mean",
        "roc_auc_std",
        "precision_mean",
        "precision_std",
        "recall_mean",
        "recall_std",
        "f1_mean",
        "f1_std",
        "accuracy_mean",
        "accuracy_std",
    ]
    cols = [c for c in preferred if c in table.columns] + [c for c in table.columns if c not in preferred]
    return table[cols]


Overwriting section6_baseline_system.py


In [40]:
%%writefile test_section6_baseline_system.py
import unittest
import numpy as np
import pandas as pd

import section6_baseline_system as s6
import section5_preprocessing_pipeline as s5


class TestSection6BaselineSystem(unittest.TestCase):
    def _toy_diabetes_like_df(self, n: int = 120) -> pd.DataFrame:
        rng = np.random.RandomState(0)

        # Fixed imbalance: 20% positives (early readmission)
        y = np.array([0] * int(n * 0.8) + [1] * (n - int(n * 0.8)), dtype=int)
        rng.shuffle(y)

        ages = list(s5.AGE_ORDER)
        df = pd.DataFrame(
            {
                "encounter_id": np.arange(n),
                "patient_nbr": np.arange(10_000, 10_000 + n),
                "age": rng.choice(ages, size=n),
                "weight": rng.choice(["?", "70", "80", "90"], size=n),
                "num_lab_procedures": rng.poisson(lam=40, size=n),
                "admission_type_id": rng.choice([1, 2, 6], size=n),
                "medical_specialty": rng.choice(["Cardiology", "InternalMedicine", "?", "RareSpec"], size=n),
                "diag_1": rng.choice(["250.83", "276", "648", "8"], size=n),
                "readmitted_30d": y,
            }
        )
        return df

    def test_evaluate_most_frequent_baseline_expected_behavior(self):
        df = self._toy_diabetes_like_df(n=120)

        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=False, rare_min_count=1)

        res = s6.evaluate_dummy_baseline_cv(
            df,
            target_col="readmitted_30d",
            strategy="most_frequent",
            n_splits=4,
            cv_random_state=0,
            dummy_random_state=0,
            preprocess_config=cfg,
        )

        self.assertEqual(res.strategy, "most_frequent")
        self.assertEqual(res.n_splits, 4)

        # Fold arrays exist and have correct length
        for k in ["roc_auc", "precision", "recall", "f1", "accuracy"]:
            self.assertIn(k, res.fold_scores)
            self.assertEqual(len(res.fold_scores[k]), 4)
            self.assertTrue(np.isfinite(res.fold_scores[k]).all())

        # Most-frequent baseline predicts only the majority class => recall/F1 for positives should be 0
        self.assertAlmostEqual(res.mean_scores["recall"], 0.0, places=12)
        self.assertAlmostEqual(res.mean_scores["f1"], 0.0, places=12)

        # Constant scoring => ROC-AUC should be chance-level (~0.5)
        self.assertAlmostEqual(res.mean_scores["roc_auc"], 0.5, places=12)

    def test_evaluate_stratified_baseline_runs_and_scores_are_valid(self):
        df = self._toy_diabetes_like_df(n=120)
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=False, rare_min_count=1)

        res = s6.evaluate_dummy_baseline_cv(
            df,
            target_col="readmitted_30d",
            strategy="stratified",
            n_splits=4,
            cv_random_state=0,
            dummy_random_state=0,
            preprocess_config=cfg,
        )

        # Basic sanity: fold scores exist and are finite
        for k in ["roc_auc", "precision", "recall", "f1", "accuracy"]:
            self.assertIn(k, res.fold_scores)
            self.assertEqual(len(res.fold_scores[k]), 4)
            self.assertTrue(np.isfinite(res.fold_scores[k]).all())

        # Mean scores must be within [0,1]
        for k, v in res.mean_scores.items():
            self.assertGreaterEqual(v, 0.0)
            self.assertLessEqual(v, 1.0)

        # Chance-level discrimination is *approximately* 0.5 for a random baseline.
        # Do NOT assert exact 0.5 due to finite-sample variability.
        roc = res.mean_scores["roc_auc"]
        self.assertGreaterEqual(roc, 0.35)
        self.assertLessEqual(roc, 0.65)


    def test_run_dummy_baselines_returns_table_with_expected_columns(self):
        df = self._toy_diabetes_like_df(n=120)
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=False, rare_min_count=1)

        table = s6.run_dummy_baselines(
            df,
            target_col="readmitted_30d",
            strategies=("most_frequent", "stratified"),
            n_splits=4,
            cv_random_state=0,
            dummy_random_state=0,
            preprocess_config=cfg,
        )

        self.assertEqual(table.shape[0], 2)
        self.assertIn("strategy", table.columns)
        self.assertIn("roc_auc_mean", table.columns)
        self.assertIn("f1_mean", table.columns)
        self.assertTrue(set(table["strategy"].tolist()) == {"most_frequent", "stratified"})


if __name__ == "__main__":
    unittest.main(verbosity=2)


Overwriting test_section6_baseline_system.py


In [41]:
!python -m unittest -v test_section6_baseline_system.py


test_evaluate_most_frequent_baseline_expected_behavior (test_section6_baseline_system.TestSection6BaselineSystem.test_evaluate_most_frequent_baseline_expected_behavior) ... ok
test_evaluate_stratified_baseline_runs_and_scores_are_valid (test_section6_baseline_system.TestSection6BaselineSystem.test_evaluate_stratified_baseline_runs_and_scores_are_valid) ... ok
test_run_dummy_baselines_returns_table_with_expected_columns (test_section6_baseline_system.TestSection6BaselineSystem.test_run_dummy_baselines_returns_table_with_expected_columns) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.830s

OK


In [37]:
import section6_baseline_system as s6

baseline_table = s6.run_dummy_baselines(
    df,
    target_col="readmitted_30d",
    strategies=("most_frequent", "stratified"),
    n_splits=10,
    cv_random_state=42,
    dummy_random_state=42,
    preprocess_config=cfg,
)
baseline_table


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median

,strategy,n_splits,roc_auc_mean,roc_auc_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,accuracy_mean,accuracy_std
0,most_frequent,10.0,0.500000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.888401,0.000043
1,stratified,10.0,0.502107,0.004082,0.115465,0.00749,0.111825,0.007248,0.113616,0.007367,0.805279,0.001621


In [42]:
%%writefile section7_validation_protocol.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import hashlib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold


Split = Tuple[np.ndarray, np.ndarray]


# ---------------------------------------------------------------------
# Section 7 — Single, reusable 10-fold CV protocol (identical folds)
# ---------------------------------------------------------------------


def make_stratified_cv_splits(
    y: Iterable[int],
    *,
    n_splits: int = 10,
    shuffle: bool = True,
    random_state: int = 42,
) -> List[Split]:
    """
    Create one fixed set of Stratified K-Fold splits (train_idx, test_idx),
    to be REUSED across all experiments (baseline, models, imbalance methods,
    selection, PCA, tuning outer loop).

    This is the "identical folds across comparisons" contract required by the practice.
    """
    y_arr = np.asarray(list(y), dtype=int)
    if y_arr.ndim != 1:
        raise ValueError("y must be a 1D iterable of binary labels.")
    if len(y_arr) == 0:
        raise ValueError("y is empty.")
    uniq = set(np.unique(y_arr).tolist())
    if not uniq.issubset({0, 1}):
        raise ValueError(f"y must be binary in {{0,1}}. Found: {sorted(list(uniq))}")

    cv = StratifiedKFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
    splits: List[Split] = []
    X_dummy = np.zeros((len(y_arr), 1), dtype=float)  # splitter needs X, but doesn't use it
    for tr, te in cv.split(X_dummy, y_arr):
        splits.append((np.asarray(tr, dtype=np.int64), np.asarray(te, dtype=np.int64)))

    validate_cv_splits(splits, n_samples=len(y_arr), n_splits_expected=n_splits)
    return splits


def validate_cv_splits(
    splits: Sequence[Split],
    *,
    n_samples: int,
    n_splits_expected: Optional[int] = None,
) -> None:
    """
    Sanity checks:
      - correct number of folds (if provided)
      - each test fold disjoint
      - union of test indices covers all samples exactly once
      - train/test disjoint per fold
    """
    if n_samples <= 0:
        raise ValueError("n_samples must be > 0.")
    if n_splits_expected is not None and len(splits) != int(n_splits_expected):
        raise ValueError(f"Expected {n_splits_expected} folds, got {len(splits)}.")

    seen_test = np.zeros(n_samples, dtype=int)

    for i, (tr, te) in enumerate(splits):
        tr = np.asarray(tr, dtype=np.int64)
        te = np.asarray(te, dtype=np.int64)

        if tr.ndim != 1 or te.ndim != 1:
            raise ValueError(f"Fold {i}: indices must be 1D arrays.")
        if len(te) == 0:
            raise ValueError(f"Fold {i}: empty test fold.")
        if np.intersect1d(tr, te).size != 0:
            raise ValueError(f"Fold {i}: train/test overlap detected.")

        if tr.min() < 0 or te.min() < 0 or tr.max() >= n_samples or te.max() >= n_samples:
            raise ValueError(f"Fold {i}: indices out of bounds for n_samples={n_samples}.")

        # track test coverage
        for idx in te:
            seen_test[idx] += 1

    if not np.all(seen_test == 1):
        bad = np.where(seen_test != 1)[0]
        raise ValueError(
            "Test folds must cover every sample exactly once. "
            f"Violations at indices: {bad[:20].tolist()} (showing up to 20)."
        )


def cv_splits_fingerprint(splits: Sequence[Split]) -> str:
    """
    Stable hash identifying the exact fold partitioning.
    Useful to assert fold identity across experiments and for reproducibility logs.
    """
    h = hashlib.sha256()
    h.update(str(len(splits)).encode("utf-8"))
    for tr, te in splits:
        tr = np.asarray(tr, dtype=np.int64)
        te = np.asarray(te, dtype=np.int64)
        h.update(tr.tobytes())
        h.update(b"|")
        h.update(te.tobytes())
        h.update(b";")
    return h.hexdigest()


def _final_estimator(pipeline_or_estimator):
    if hasattr(pipeline_or_estimator, "steps") and pipeline_or_estimator.steps:
        return pipeline_or_estimator.steps[-1][1]
    return pipeline_or_estimator


def _get_score_vector(pipeline_or_estimator, X, *, positive_label: int = 1) -> np.ndarray:
    """
    Get continuous scores for ROC-AUC:
      - predict_proba[:, pos] if available
      - else decision_function if available
      - else raise (to avoid meaningless AUC from hard labels)
    """
    if hasattr(pipeline_or_estimator, "predict_proba"):
        proba = pipeline_or_estimator.predict_proba(X)
        proba = np.asarray(proba)
        if proba.ndim != 2 or proba.shape[1] < 2:
            raise ValueError("predict_proba output must be [n_samples, 2+] for binary ROC-AUC.")
        est = _final_estimator(pipeline_or_estimator)
        classes = getattr(est, "classes_", None)
        if classes is None:
            # fallback: assume column 1 is positive
            pos_idx = 1
        else:
            classes = np.asarray(classes)
            if positive_label in set(classes.tolist()):
                pos_idx = int(np.where(classes == positive_label)[0][0])
            else:
                pos_idx = 1
        return proba[:, pos_idx].astype(float)

    if hasattr(pipeline_or_estimator, "decision_function"):
        s = pipeline_or_estimator.decision_function(X)
        return np.asarray(s, dtype=float).ravel()

    raise ValueError("Estimator must expose predict_proba or decision_function to compute ROC-AUC.")


@dataclass(frozen=True)
class CVEvaluationResult:
    n_splits: int
    split_fingerprint: str
    fold_indices: List[Split]
    fold_scores: Dict[str, np.ndarray]
    mean_scores: Dict[str, float]
    std_scores: Dict[str, float]
    oof_pred: Optional[np.ndarray] = None
    oof_score: Optional[np.ndarray] = None


def evaluate_binary_pipeline_cv(
    pipeline,
    X,
    y: Iterable[int],
    *,
    cv_splits: Sequence[Split],
    positive_label: int = 1,
    metrics: Sequence[str] = ("roc_auc", "accuracy", "precision", "recall", "f1", "balanced_accuracy"),
    return_oof: bool = True,
) -> CVEvaluationResult:
    """
    Manual CV loop to guarantee:
      - identical folds (cv_splits provided)
      - per-fold metric vectors (needed later for Wilcoxon)
      - optional out-of-fold predictions/scores (useful for error analysis / XAI)
    """
    y_arr = np.asarray(list(y), dtype=int)
    n = len(y_arr)
    validate_cv_splits(cv_splits, n_samples=n)
    fp = cv_splits_fingerprint(cv_splits)

    wanted = set(metrics)
    allowed = {"roc_auc", "accuracy", "precision", "recall", "f1", "balanced_accuracy"}
    if not wanted.issubset(allowed):
        raise ValueError(f"Unsupported metrics requested: {sorted(list(wanted - allowed))}")

    fold_scores: Dict[str, List[float]] = {m: [] for m in metrics}

    oof_pred = np.full(n, fill_value=-1, dtype=int) if return_oof else None
    oof_score = np.full(n, fill_value=np.nan, dtype=float) if return_oof else None

    for tr, te in cv_splits:
        tr = np.asarray(tr, dtype=np.int64)
        te = np.asarray(te, dtype=np.int64)

        model = clone(pipeline)
        model.fit(_safe_index(X, tr), y_arr[tr])

        y_pred = np.asarray(model.predict(_safe_index(X, te)), dtype=int)

        if "roc_auc" in wanted:
            y_score = _get_score_vector(model, _safe_index(X, te), positive_label=positive_label)
            fold_scores["roc_auc"].append(float(roc_auc_score(y_arr[te], y_score)))
        else:
            y_score = None

        if "accuracy" in wanted:
            fold_scores["accuracy"].append(float(accuracy_score(y_arr[te], y_pred)))
        if "precision" in wanted:
            fold_scores["precision"].append(float(precision_score(y_arr[te], y_pred, pos_label=positive_label, zero_division=0)))
        if "recall" in wanted:
            fold_scores["recall"].append(float(recall_score(y_arr[te], y_pred, pos_label=positive_label, zero_division=0)))
        if "f1" in wanted:
            fold_scores["f1"].append(float(f1_score(y_arr[te], y_pred, pos_label=positive_label, zero_division=0)))
        if "balanced_accuracy" in wanted:
            fold_scores["balanced_accuracy"].append(float(balanced_accuracy_score(y_arr[te], y_pred)))

        if return_oof:
            oof_pred[te] = y_pred
            if y_score is not None:
                oof_score[te] = np.asarray(y_score, dtype=float)

    fold_scores_arr: Dict[str, np.ndarray] = {k: np.asarray(v, dtype=float) for k, v in fold_scores.items()}
    mean_scores = {k: float(np.mean(v)) for k, v in fold_scores_arr.items()}
    std_scores = {k: float(np.std(v, ddof=1)) if len(v) > 1 else 0.0 for k, v in fold_scores_arr.items()}

    return CVEvaluationResult(
        n_splits=len(cv_splits),
        split_fingerprint=fp,
        fold_indices=[(np.asarray(tr, dtype=np.int64), np.asarray(te, dtype=np.int64)) for tr, te in cv_splits],
        fold_scores=fold_scores_arr,
        mean_scores=mean_scores,
        std_scores=std_scores,
        oof_pred=oof_pred,
        oof_score=oof_score,
    )


def _safe_index(X, idx: np.ndarray):
    """
    Index X robustly whether it's a pandas DataFrame/Series or a numpy array / sparse matrix.
    """
    if isinstance(X, (pd.DataFrame, pd.Series)):
        return X.iloc[idx]
    return X[idx]


Overwriting section7_validation_protocol.py


In [43]:
%%writefile test_section7_validation_protocol.py
import unittest
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

import section7_validation_protocol as s7
import section5_preprocessing_pipeline as s5


def _splits_equal(a, b) -> bool:
    if len(a) != len(b):
        return False
    for (tr1, te1), (tr2, te2) in zip(a, b):
        if not (np.array_equal(tr1, tr2) and np.array_equal(te1, te2)):
            return False
    return True


class TestSection7ValidationProtocol(unittest.TestCase):
    def _toy_diabetes_like_df(self, n: int = 200) -> pd.DataFrame:
        rng = np.random.RandomState(0)

        # 20% positives, 80% negatives
        y = np.array([0] * int(n * 0.8) + [1] * (n - int(n * 0.8)), dtype=int)
        rng.shuffle(y)

        ages = list(s5.AGE_ORDER)
        df = pd.DataFrame(
            {
                "encounter_id": np.arange(n),
                "patient_nbr": np.arange(10_000, 10_000 + n),
                "age": rng.choice(ages, size=n),
                "weight": rng.choice(["?", "70", "80", "90", "Unknown/Invalid"], size=n),
                "num_lab_procedures": rng.poisson(lam=40, size=n),
                "admission_type_id": rng.choice([1, 2, 6], size=n),
                "medical_specialty": rng.choice(["Cardiology", "InternalMedicine", "?", "RareSpec"], size=n),
                "diag_1": rng.choice(["250.83", "276", "648", "8"], size=n),
                "readmitted_30d": y,
            }
        )
        return df

    def test_make_stratified_cv_splits_reproducible_and_valid(self):
        df = self._toy_diabetes_like_df(n=200)
        y = df["readmitted_30d"].astype(int).tolist()

        splits1 = s7.make_stratified_cv_splits(y, n_splits=10, random_state=42)
        splits2 = s7.make_stratified_cv_splits(y, n_splits=10, random_state=42)

        self.assertTrue(_splits_equal(splits1, splits2))
        # validate should not raise
        s7.validate_cv_splits(splits1, n_samples=len(y), n_splits_expected=10)

    def test_cv_fingerprint_changes_with_seed(self):
        df = self._toy_diabetes_like_df(n=200)
        y = df["readmitted_30d"].astype(int).tolist()

        s0 = s7.make_stratified_cv_splits(y, n_splits=10, random_state=0)
        s1 = s7.make_stratified_cv_splits(y, n_splits=10, random_state=1)

        fp0 = s7.cv_splits_fingerprint(s0)
        fp1 = s7.cv_splits_fingerprint(s1)

        self.assertNotEqual(fp0, fp1)

    def test_evaluate_binary_pipeline_cv_returns_fold_scores_and_oof(self):
        df = self._toy_diabetes_like_df(n=200)
        y = df["readmitted_30d"].astype(int)
        X = df.drop(columns=["readmitted_30d"])

        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=False, rare_min_count=1)
        pre = s5.build_preprocessor(df, cfg)

        pipe = Pipeline(
            steps=[
                ("pre", pre),
                ("clf", LogisticRegression(max_iter=1000)),
            ]
        )

        splits = s7.make_stratified_cv_splits(y, n_splits=10, random_state=42)
        res = s7.evaluate_binary_pipeline_cv(pipe, X, y, cv_splits=splits, return_oof=True)

        self.assertEqual(res.n_splits, 10)
        self.assertEqual(len(res.fold_scores["roc_auc"]), 10)
        self.assertTrue(np.isfinite(res.fold_scores["roc_auc"]).all())

        # means in [0,1]
        for v in res.mean_scores.values():
            self.assertGreaterEqual(v, 0.0)
            self.assertLessEqual(v, 1.0)

        # OOF outputs cover all rows
        self.assertIsNotNone(res.oof_pred)
        self.assertIsNotNone(res.oof_score)
        self.assertEqual(len(res.oof_pred), len(df))
        self.assertEqual(len(res.oof_score), len(df))
        self.assertTrue(np.isfinite(res.oof_score).all())

    def test_two_models_share_identical_folds(self):
        df = self._toy_diabetes_like_df(n=200)
        y = df["readmitted_30d"].astype(int)
        X = df.drop(columns=["readmitted_30d"])

        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=False, rare_min_count=1)
        pre = s5.build_preprocessor(df, cfg)

        splits = s7.make_stratified_cv_splits(y, n_splits=10, random_state=42)

        pipe_lr = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
        pipe_dt = Pipeline([("pre", pre), ("clf", DecisionTreeClassifier(random_state=0))])

        r1 = s7.evaluate_binary_pipeline_cv(pipe_lr, X, y, cv_splits=splits, return_oof=False)
        r2 = s7.evaluate_binary_pipeline_cv(pipe_dt, X, y, cv_splits=splits, return_oof=False)

        self.assertEqual(r1.split_fingerprint, r2.split_fingerprint)
        self.assertTrue(_splits_equal(r1.fold_indices, r2.fold_indices))


if __name__ == "__main__":
    unittest.main(verbosity=2)


Overwriting test_section7_validation_protocol.py


In [44]:
!python -m unittest -v test_section7_validation_protocol.py


test_cv_fingerprint_changes_with_seed (test_section7_validation_protocol.TestSection7ValidationProtocol.test_cv_fingerprint_changes_with_seed) ... ok
test_evaluate_binary_pipeline_cv_returns_fold_scores_and_oof (test_section7_validation_protocol.TestSection7ValidationProtocol.test_evaluate_binary_pipeline_cv_returns_fold_scores_and_oof) ... ok
test_make_stratified_cv_splits_reproducible_and_valid (test_section7_validation_protocol.TestSection7ValidationProtocol.test_make_stratified_cv_splits_reproducible_and_valid) ... ok
test_two_models_share_identical_folds (test_section7_validation_protocol.TestSection7ValidationProtocol.test_two_models_share_identical_folds) ... ok

----------------------------------------------------------------------
Ran 4 tests in 1.723s

OK


In [45]:
import section7_validation_protocol as s7

y = df["readmitted_30d"].astype(int).to_numpy()

splits = s7.make_stratified_cv_splits(
    y,
    n_splits=10,
    shuffle=True,
    random_state=42,   # keep fixed for reproducibility + Wilcoxon later
)

fingerprint = s7.cv_splits_fingerprint(splits)
fingerprint


'f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3d6a99a4e5b64150ef5'

In [46]:
%%writefile section8_core_modeling.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7


# ---------------------------------------------------------------------
# Section 8 — Core Modeling (Basic Level)
# - Train and analyze >= 2 different classifiers
# - Must use identical 10-fold CV splits across comparisons (Section 7)
# - Models here: Logistic Regression (linear/probabilistic) + Decision Tree (rule-based/non-linear)
#
# Notes:
# - Decision trees are commonly restricted to CSC sparse matrices in sklearn;
#   we add a lightweight converter to avoid format-related failures while keeping sparse encoding.
# - Preprocessing is the single-source-of-truth from Section 5 (no leakage: fit inside CV folds).
# ---------------------------------------------------------------------


class SparseToCSC(BaseEstimator, TransformerMixin):
    """Convert sparse matrices to CSC (DecisionTree often expects CSC when input is sparse)."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        try:
            from scipy import sparse  # scipy is a dependency of sklearn in most environments
        except Exception as e:
            raise RuntimeError(
                "scipy is required for sparse matrix conversion. "
                "Install with: pip install scipy"
            ) from e

        if sparse.issparse(X):
            return X.tocsc()
        return X


def _validate_binary_target(y: pd.Series) -> np.ndarray:
    if y.isna().any():
        raise ValueError("Target contains missing values; Section 8 requires a clean binary target.")
    y_arr = np.asarray(pd.Series(y).astype(int).tolist(), dtype=int)
    uniq = set(np.unique(y_arr).tolist())
    if not uniq.issubset({0, 1}):
        raise ValueError(f"Target must be binary in {{0,1}}, found values: {sorted(list(uniq))}")
    return y_arr


def _default_X_y(
    df: pd.DataFrame,
    *,
    target_col: str,
    preprocess_config: s5.DiabetesPreprocessConfig,
) -> Tuple[pd.DataFrame, np.ndarray]:
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found. Available: {list(df.columns)}")

    y_arr = _validate_binary_target(df[target_col])

    drop_cols = [c for c in preprocess_config.target_cols if c in df.columns]
    if target_col in df.columns and target_col not in drop_cols:
        drop_cols.append(target_col)

    X = df.drop(columns=drop_cols, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError("No feature columns available after dropping target columns.")
    return X, y_arr


@dataclass(frozen=True)
class ModelSpec:
    name: str
    estimator: Any
    require_csc: bool = False


def build_section8_model_specs(
    *,
    random_state: int = 42,
) -> List[ModelSpec]:
    """
    Two conceptually different classifiers:
      1) Logistic Regression (linear, probabilistic)
      2) Decision Tree (rule-based, non-linear)

    We pick a LR solver compatible with sparse one-hot features ("saga").
    """
    lr = LogisticRegression(
        solver="saga",
        penalty="l2",
        max_iter=2000,
        random_state=random_state,
        n_jobs=-1,
    )
    dt = DecisionTreeClassifier(
        random_state=random_state,
    )

    return [
        ModelSpec(name="logistic_regression", estimator=lr, require_csc=False),
        ModelSpec(name="decision_tree", estimator=dt, require_csc=True),
    ]


def build_section8_pipelines(
    df: pd.DataFrame,
    *,
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    model_random_state: int = 42,
) -> Dict[str, Pipeline]:
    """
    Build model pipelines = preprocessing (Section 5) + optional format adapter + classifier.
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    pre = s5.build_preprocessor(df, preprocess_config)
    specs = build_section8_model_specs(random_state=model_random_state)

    pipes: Dict[str, Pipeline] = {}
    for spec in specs:
        steps = [("pre", pre)]
        if spec.require_csc:
            steps.append(("to_csc", SparseToCSC()))
        steps.append(("clf", spec.estimator))
        pipes[spec.name] = Pipeline(steps=steps)

    return pipes


@dataclass(frozen=True)
class Section8RunResult:
    """
    Holds everything you need for the report:
      - identical-fold CV metrics (mean/std + per-fold vectors)
      - fold fingerprint for reproducibility
    """
    target_col: str
    n_splits: int
    cv_random_state: int
    split_fingerprint: str
    results_by_model: Dict[str, s7.CVEvaluationResult]
    summary_table: pd.DataFrame


def _summary_table_from_results(
    results_by_model: Dict[str, s7.CVEvaluationResult],
    *,
    model_order: Sequence[str],
    metrics: Sequence[str],
) -> pd.DataFrame:
    rows: List[Dict[str, float]] = []

    for name in model_order:
        r = results_by_model[name]
        row: Dict[str, float] = {"model": name}
        for m in metrics:
            row[f"{m}_mean"] = float(r.mean_scores[m])
            row[f"{m}_std"] = float(r.std_scores[m])
        rows.append(row)

    cols = ["model"] + [f"{m}_{s}" for m in metrics for s in ("mean", "std")]
    return pd.DataFrame(rows)[cols]


def run_section8_core_models(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    n_splits: int = 10,
    cv_random_state: int = 42,
    model_random_state: int = 42,
    metrics: Sequence[str] = ("roc_auc", "precision", "recall", "f1", "accuracy", "balanced_accuracy"),
    return_oof: bool = True,
    cv_splits: Optional[Sequence[s7.Split]] = None,
) -> Section8RunResult:
    """
    End-to-end Section 8 runner:
      - builds preprocessing + LR + DT pipelines
      - creates/reuses identical stratified splits (10-fold by default)
      - evaluates each model with the manual CV loop (Section 7)
      - returns a compact mean±std table + full per-fold vectors
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    X, y = _default_X_y(df, target_col=target_col, preprocess_config=preprocess_config)

    if cv_splits is None:
        cv_splits = s7.make_stratified_cv_splits(
            y,
            n_splits=n_splits,
            shuffle=True,
            random_state=cv_random_state,
        )
    else:
        s7.validate_cv_splits(cv_splits, n_samples=len(y), n_splits_expected=n_splits)

    fp = s7.cv_splits_fingerprint(cv_splits)

    pipes = build_section8_pipelines(
        df,
        preprocess_config=preprocess_config,
        model_random_state=model_random_state,
    )

    # Evaluate with identical folds
    results_by_model: Dict[str, s7.CVEvaluationResult] = {}
    for name, pipe in pipes.items():
        res = s7.evaluate_binary_pipeline_cv(
            pipe,
            X,
            y,
            cv_splits=cv_splits,
            metrics=metrics,
            return_oof=return_oof,
        )
        results_by_model[name] = res

    model_order = list(pipes.keys())
    table = _summary_table_from_results(results_by_model, model_order=model_order, metrics=metrics)

    return Section8RunResult(
        target_col=target_col,
        n_splits=n_splits,
        cv_random_state=cv_random_state,
        split_fingerprint=fp,
        results_by_model=results_by_model,
        summary_table=table,
    )


Overwriting section8_core_modeling.py


In [47]:
%%writefile test_section8_core_modeling.py
import unittest
import numpy as np
import pandas as pd

import section5_preprocessing_pipeline as s5
import section8_core_modeling as s8
import section7_validation_protocol as s7


class TestSection8CoreModeling(unittest.TestCase):
    def _toy_diabetes_like_df(self, n: int = 300) -> pd.DataFrame:
        rng = np.random.RandomState(0)

        # 20% positives, 80% negatives
        y = np.array([0] * int(n * 0.8) + [1] * (n - int(n * 0.8)), dtype=int)
        rng.shuffle(y)

        ages = list(s5.AGE_ORDER)
        df = pd.DataFrame(
            {
                "encounter_id": np.arange(n),
                "patient_nbr": np.arange(10_000, 10_000 + n),
                "race": rng.choice(["Caucasian", "AfricanAmerican", "Hispanic", "?"], size=n),
                "gender": rng.choice(["Male", "Female"], size=n),
                "age": rng.choice(ages, size=n),
                "weight": rng.choice(["?", "70", "80", "90", "Unknown/Invalid"], size=n),
                "admission_type_id": rng.choice([1, 2, 6], size=n),
                "discharge_disposition_id": rng.choice([1, 3, 25], size=n),
                "admission_source_id": rng.choice([1, 7], size=n),
                "time_in_hospital": rng.choice([1, 2, 3, 4, 5, 6, 7], size=n),
                "payer_code": rng.choice(["MC", "MD", "?", "SP"], size=n),
                "medical_specialty": rng.choice(
                    ["Cardiology", "InternalMedicine", "?", "RareSpecA", "RareSpecB"], size=n
                ),
                "num_lab_procedures": rng.poisson(lam=40, size=n),
                "num_procedures": rng.poisson(lam=1, size=n),
                "num_medications": rng.poisson(lam=10, size=n),
                "number_outpatient": rng.poisson(lam=1, size=n),
                "number_emergency": rng.poisson(lam=0.2, size=n),
                "number_inpatient": rng.poisson(lam=0.5, size=n),
                "diag_1": rng.choice(["250.83", "276", "648", "8", "401"], size=n),
                "diag_2": rng.choice(["?", "250.01", "403", "V27"], size=n),
                "diag_3": rng.choice(["?", "7", "9", "6"], size=n),
                "A1Cresult": rng.choice(["None", "Norm", ">7", ">8", "?"], size=n),
                "insulin": rng.choice(["No", "Steady", "Up", "Down"], size=n),
                "change": rng.choice(["Ch", "No"], size=n),
                "diabetesMed": rng.choice(["Yes", "No"], size=n),
                "readmitted_30d": y,
            }
        )
        return df

    def test_build_pipelines_contains_two_models(self):
        df = self._toy_diabetes_like_df(n=50)
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=True, rare_min_count=1)

        pipes = s8.build_section8_pipelines(df, preprocess_config=cfg, model_random_state=42)
        self.assertIn("logistic_regression", pipes)
        self.assertIn("decision_tree", pipes)

        # Decision tree pipeline should include the CSC converter step
        self.assertTrue(any(step_name == "to_csc" for step_name, _ in pipes["decision_tree"].steps))

    def test_run_section8_core_models_returns_table_and_identical_folds(self):
        df = self._toy_diabetes_like_df(n=300)
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=True, rare_min_count=1)

        # Create splits once and reuse
        y = df["readmitted_30d"].astype(int).to_numpy()
        splits = s7.make_stratified_cv_splits(y, n_splits=5, random_state=123)

        out = s8.run_section8_core_models(
            df,
            target_col="readmitted_30d",
            preprocess_config=cfg,
            n_splits=5,
            cv_random_state=123,
            model_random_state=42,
            cv_splits=splits,
            return_oof=True,
        )

        # Table shape: 2 models
        self.assertEqual(out.summary_table.shape[0], 2)
        self.assertIn("roc_auc_mean", out.summary_table.columns)
        self.assertIn("f1_mean", out.summary_table.columns)

        # Both models must share the exact same folds fingerprint
        r_lr = out.results_by_model["logistic_regression"]
        r_dt = out.results_by_model["decision_tree"]
        self.assertEqual(r_lr.split_fingerprint, r_dt.split_fingerprint)
        self.assertEqual(r_lr.split_fingerprint, out.split_fingerprint)

        # Fold scores are finite and length = n_splits
        self.assertEqual(len(r_lr.fold_scores["roc_auc"]), 5)
        self.assertTrue(np.isfinite(r_lr.fold_scores["roc_auc"]).all())
        self.assertEqual(len(r_dt.fold_scores["roc_auc"]), 5)
        self.assertTrue(np.isfinite(r_dt.fold_scores["roc_auc"]).all())

        # OOF vectors exist and cover all rows
        self.assertEqual(len(r_lr.oof_pred), len(df))
        self.assertEqual(len(r_lr.oof_score), len(df))
        self.assertTrue(np.isfinite(r_lr.oof_score).all())

    def test_raises_if_target_missing(self):
        df = self._toy_diabetes_like_df(n=50).drop(columns=["readmitted_30d"])
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=True, rare_min_count=1)
        with self.assertRaises(KeyError):
            _ = s8.run_section8_core_models(df, target_col="readmitted_30d", preprocess_config=cfg, n_splits=3)


if __name__ == "__main__":
    unittest.main(verbosity=2)


Overwriting test_section8_core_modeling.py


In [48]:
!python -m unittest -v test_section8_core_modeling.py


test_build_pipelines_contains_two_models (test_section8_core_modeling.TestSection8CoreModeling.test_build_pipelines_contains_two_models) ... ok
test_raises_if_target_missing (test_section8_core_modeling.TestSection8CoreModeling.test_raises_if_target_missing) ... ok
test_run_section8_core_models_returns_table_and_identical_folds (test_section8_core_modeling.TestSection8CoreModeling.test_run_section8_core_models_returns_table_and_identical_folds) ... ok

----------------------------------------------------------------------
Ran 3 tests in 1.867s

OK


In [44]:
import section8_core_modeling as s8

out8 = s8.run_section8_core_models(
    df,
    target_col="readmitted_30d",
    preprocess_config=cfg,
    n_splits=10,
    cv_random_state=42,
    model_random_state=42,
    cv_splits=splits,     # <<< critical: identical folds
    return_oof=True,      # useful later for error analysis / XAI
)

out8.split_fingerprint, out8.summary_table


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


KeyboardInterrupt: 

In [49]:
%%writefile section9_expanded_modeling.py
from __future__ import annotations

from dataclasses import dataclass, replace
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7

Split = s7.Split


class SparseToCSC(BaseEstimator, TransformerMixin):
    """Convert sparse matrices to CSC (tree-based estimators often prefer CSC when input is sparse)."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        try:
            from scipy import sparse
        except Exception as e:
            raise RuntimeError(
                "scipy is required for sparse matrix conversion. Install with: pip install scipy"
            ) from e

        if sparse.issparse(X):
            return X.tocsc()
        return X


def _validate_binary_target(y: pd.Series) -> np.ndarray:
    if y.isna().any():
        raise ValueError("Target contains missing values; Section 9 requires a clean binary target.")
    y_arr = np.asarray(pd.Series(y).astype(int).tolist(), dtype=int)
    uniq = set(np.unique(y_arr).tolist())
    if not uniq.issubset({0, 1}):
        raise ValueError(f"Target must be binary in {{0,1}}, found values: {sorted(list(uniq))}")
    return y_arr


def _default_X_y(
    df: pd.DataFrame,
    *,
    target_col: str,
    preprocess_config: s5.DiabetesPreprocessConfig,
) -> Tuple[pd.DataFrame, np.ndarray]:
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found. Available: {list(df.columns)}")

    y = _validate_binary_target(df[target_col])

    drop_cols = [c for c in preprocess_config.target_cols if c in df.columns]
    if target_col not in drop_cols:
        drop_cols.append(target_col)

    X = df.drop(columns=drop_cols, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError("No feature columns available after dropping target columns.")
    return X, y


@dataclass(frozen=True)
class ModelSpec:
    name: str
    pipeline: Pipeline
    track: str  # helps your report: scaled / tree / nb_nonneg / svd_dense


def _make_configs(
    base: s5.DiabetesPreprocessConfig,
) -> Dict[str, s5.DiabetesPreprocessConfig]:
    """
    Section 9 allows deterministic preprocessing tracks:
      - scaled: for scale-sensitive models (SVM, MLP)
      - tree: for tree-based models (DT, RF) (scaling not required)
      - nb_nonneg: ensures non-negative features for MultinomialNB:
          * disables ordinal encoding (age becomes one-hot categorical)
          * disables numeric scaling
    """
    cfg_scaled = replace(base, scale_numeric=True, scaler="standard")
    cfg_tree = replace(base, scale_numeric=False)

    # IMPORTANT: MultinomialNB requires non-negative inputs.
    # Our default ordinal encoding can yield -1 for unknowns, so for NB we remove ordinal encoding.
    cfg_nb = replace(cfg_tree, ordinal_cols={})

    return {"scaled": cfg_scaled, "tree": cfg_tree, "nb_nonneg": cfg_nb}


def build_section9_model_specs(
    df: pd.DataFrame,
    *,
    preprocess_config_base: Optional[s5.DiabetesPreprocessConfig] = None,
    model_random_state: int = 42,
    svd_components: int = 100,
    fast_mode: bool = False,
) -> List[ModelSpec]:
    """
    Expanded suite (>=5, includes ensembles):
      - decision_tree (anchor)
      - naive_bayes (MultinomialNB; non-negative track)
      - svm_linear (LinearSVC; margin-based)
      - mlp_svd (MLP on TruncatedSVD components; avoids densifying huge one-hot)
      - random_forest (bagging)
      - gradient_boosting_svd (boosting on SVD components; avoids sparse incompatibility)

    fast_mode reduces compute for unit tests / quick sanity runs.
    """
    if preprocess_config_base is None:
        preprocess_config_base = s5.DiabetesPreprocessConfig()

    cfgs = _make_configs(preprocess_config_base)

    pre_tree = s5.build_preprocessor(df, cfgs["tree"])
    pre_scaled = s5.build_preprocessor(df, cfgs["scaled"])
    pre_nb = s5.build_preprocessor(df, cfgs["nb_nonneg"])

    # ---- hyperparameters (safe defaults; tuning belongs in Section 14) ----
    rf_estimators = 80 if fast_mode else 300
    gb_estimators = 100 if fast_mode else 200
    mlp_max_iter = 60 if fast_mode else 200
    mlp_hidden = (32,) if fast_mode else (64,)

    # ---- pipelines ----
    dt_pipe = Pipeline(
        steps=[
            ("pre", pre_tree),
            ("to_csc", SparseToCSC()),
            ("clf", DecisionTreeClassifier(random_state=model_random_state)),
        ]
    )

    nb_pipe = Pipeline(
        steps=[
            ("pre", pre_nb),
            ("clf", MultinomialNB(alpha=1.0)),
        ]
    )

    svm_pipe = Pipeline(
        steps=[
            ("pre", pre_scaled),
            ("clf", LinearSVC(C=1.0, random_state=model_random_state)),
        ]
    )

    # MLP cannot consume sparse one-hot reliably; we compress sparse -> dense with TruncatedSVD.
    mlp_pipe = Pipeline(
        steps=[
            ("pre", pre_scaled),
            ("svd", TruncatedSVD(n_components=int(svd_components), random_state=model_random_state)),
            ("clf", MLPClassifier(
                hidden_layer_sizes=mlp_hidden,
                activation="relu",
                solver="adam",
                alpha=1e-4,
                max_iter=int(mlp_max_iter),
                random_state=model_random_state,
                early_stopping=True,
                n_iter_no_change=10,
            )),
        ]
    )

    rf_pipe = Pipeline(
        steps=[
            ("pre", pre_tree),
            ("to_csc", SparseToCSC()),
            ("clf", RandomForestClassifier(
                n_estimators=int(rf_estimators),
                random_state=model_random_state,
                n_jobs=-1,
            )),
        ]
    )

    # GradientBoostingClassifier does not accept sparse matrices; we again use SVD to get dense components.
    gb_pipe = Pipeline(
        steps=[
            ("pre", pre_tree),
            ("svd", TruncatedSVD(n_components=int(svd_components), random_state=model_random_state)),
            ("clf", GradientBoostingClassifier(
                n_estimators=int(gb_estimators),
                learning_rate=0.1,
                random_state=model_random_state,
            )),
        ]
    )

    return [
        ModelSpec("decision_tree", dt_pipe, track="tree"),
        ModelSpec("naive_bayes", nb_pipe, track="nb_nonneg"),
        ModelSpec("svm_linear", svm_pipe, track="scaled"),
        ModelSpec("mlp_svd", mlp_pipe, track="svd_dense"),
        ModelSpec("random_forest", rf_pipe, track="tree"),
        ModelSpec("gradient_boosting_svd", gb_pipe, track="svd_dense"),
    ]


@dataclass(frozen=True)
class Section9RunResult:
    target_col: str
    n_splits: int
    cv_random_state: int
    split_fingerprint: str
    results_by_model: Dict[str, s7.CVEvaluationResult]
    summary_table: pd.DataFrame


def _summary_table(
    specs: Sequence[ModelSpec],
    results_by_model: Dict[str, s7.CVEvaluationResult],
    *,
    metrics: Sequence[str],
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for spec in specs:
        r = results_by_model[spec.name]
        row: Dict[str, Any] = {"model": spec.name, "track": spec.track}
        for m in metrics:
            row[f"{m}_mean"] = float(r.mean_scores[m])
            row[f"{m}_std"] = float(r.std_scores[m])
        rows.append(row)

    cols = ["model", "track"] + [f"{m}_{s}" for m in metrics for s in ("mean", "std")]
    return pd.DataFrame(rows)[cols]


def run_section9_expanded_suite(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    preprocess_config_base: Optional[s5.DiabetesPreprocessConfig] = None,
    n_splits: int = 10,
    cv_random_state: int = 42,
    model_random_state: int = 42,
    metrics: Sequence[str] = ("roc_auc", "precision", "recall", "f1", "accuracy", "balanced_accuracy"),
    return_oof: bool = False,
    cv_splits: Optional[Sequence[Split]] = None,
    svd_components: int = 100,
    fast_mode: bool = False,
) -> Section9RunResult:
    """
    End-to-end Section 9:
      - builds 6-model suite (>=5, includes ensembles)
      - uses identical StratifiedKFold splits (must be reused across comparisons)
      - reports mean ± std across folds (table-ready)

    IMPORTANT: For Wilcoxon later, pass the SAME cv_splits you used elsewhere.
    """
    if preprocess_config_base is None:
        preprocess_config_base = s5.DiabetesPreprocessConfig()

    X, y = _default_X_y(df, target_col=target_col, preprocess_config=preprocess_config_base)

    if cv_splits is None:
        cv_splits = s7.make_stratified_cv_splits(y, n_splits=n_splits, random_state=cv_random_state)
    else:
        s7.validate_cv_splits(cv_splits, n_samples=len(y), n_splits_expected=n_splits)

    fp = s7.cv_splits_fingerprint(cv_splits)

    specs = build_section9_model_specs(
        df,
        preprocess_config_base=preprocess_config_base,
        model_random_state=model_random_state,
        svd_components=svd_components,
        fast_mode=fast_mode,
    )

    results_by_model: Dict[str, s7.CVEvaluationResult] = {}
    for spec in specs:
        res = s7.evaluate_binary_pipeline_cv(
            spec.pipeline,
            X,
            y,
            cv_splits=cv_splits,
            metrics=metrics,
            return_oof=return_oof,
        )
        results_by_model[spec.name] = res

    table = _summary_table(specs, results_by_model, metrics=metrics)

    return Section9RunResult(
        target_col=target_col,
        n_splits=n_splits,
        cv_random_state=cv_random_state,
        split_fingerprint=fp,
        results_by_model=results_by_model,
        summary_table=table,
    )


Overwriting section9_expanded_modeling.py


In [50]:
%%writefile test_section9_expanded_modeling.py
import unittest
import numpy as np
import pandas as pd

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7
import section9_expanded_modeling as s9


class TestSection9ExpandedModeling(unittest.TestCase):
    def _toy_diabetes_like_df(self, n: int = 260) -> pd.DataFrame:
        rng = np.random.RandomState(0)

        # 20% positives, 80% negatives (enough for stratified folds)
        y = np.array([0] * int(n * 0.8) + [1] * (n - int(n * 0.8)), dtype=int)
        rng.shuffle(y)

        ages = list(s5.AGE_ORDER)
        df = pd.DataFrame(
            {
                "encounter_id": np.arange(n),
                "patient_nbr": np.arange(10_000, 10_000 + n),
                "race": rng.choice(["Caucasian", "AfricanAmerican", "Hispanic", "?"], size=n),
                "gender": rng.choice(["Male", "Female"], size=n),
                "age": rng.choice(ages + ["?"], size=n),
                "weight": rng.choice(["?", "70", "80", "90", "Unknown/Invalid"], size=n),
                "admission_type_id": rng.choice([1, 2, 6], size=n),
                "discharge_disposition_id": rng.choice([1, 3, 25], size=n),
                "admission_source_id": rng.choice([1, 7], size=n),
                "time_in_hospital": rng.choice([1, 2, 3, 4, 5, 6, 7], size=n),
                "payer_code": rng.choice(["MC", "MD", "?", "SP"], size=n),
                "medical_specialty": rng.choice(
                    ["Cardiology", "InternalMedicine", "?", "RareSpecA", "RareSpecB"], size=n
                ),
                "num_lab_procedures": rng.poisson(lam=40, size=n),
                "num_procedures": rng.poisson(lam=1, size=n),
                "num_medications": rng.poisson(lam=10, size=n),
                "number_outpatient": rng.poisson(lam=1, size=n),
                "number_emergency": rng.poisson(lam=0.2, size=n),
                "number_inpatient": rng.poisson(lam=0.5, size=n),
                "diag_1": rng.choice(["250.83", "276", "648", "8", "401"], size=n),
                "diag_2": rng.choice(["?", "250.01", "403", "V27"], size=n),
                "diag_3": rng.choice(["?", "7", "9", "6"], size=n),
                "A1Cresult": rng.choice(["None", "Norm", ">7", ">8", "?"], size=n),
                "insulin": rng.choice(["No", "Steady", "Up", "Down"], size=n),
                "change": rng.choice(["Ch", "No"], size=n),
                "diabetesMed": rng.choice(["Yes", "No"], size=n),
                "readmitted_30d": y,
            }
        )
        return df

    def test_build_model_specs_contains_six_models(self):
        df = self._toy_diabetes_like_df(n=120)
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=True, rare_min_count=1)

        specs = s9.build_section9_model_specs(
            df,
            preprocess_config_base=cfg,
            model_random_state=0,
            svd_components=10,
            fast_mode=True,
        )
        names = [s.name for s in specs]
        self.assertEqual(len(names), 6)
        for must in [
            "decision_tree",
            "naive_bayes",
            "svm_linear",
            "mlp_svd",
            "random_forest",
            "gradient_boosting_svd",
        ]:
            self.assertIn(must, names)

    def test_run_section9_produces_table_and_identical_folds(self):
        df = self._toy_diabetes_like_df(n=260)
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=True, rare_min_count=1)

        y = df["readmitted_30d"].astype(int).to_numpy()
        splits = s7.make_stratified_cv_splits(y, n_splits=5, random_state=123)

        out = s9.run_section9_expanded_suite(
            df,
            target_col="readmitted_30d",
            preprocess_config_base=cfg,
            n_splits=5,
            cv_random_state=123,
            model_random_state=0,
            cv_splits=splits,
            svd_components=10,
            fast_mode=True,
            return_oof=False,
        )

        # 6 models in the summary table
        self.assertEqual(out.summary_table.shape[0], 6)
        self.assertIn("roc_auc_mean", out.summary_table.columns)
        self.assertIn("f1_mean", out.summary_table.columns)

        # All models share the same fold fingerprint
        fps = set()
        for name, res in out.results_by_model.items():
            fps.add(res.split_fingerprint)
            self.assertEqual(len(res.fold_scores["roc_auc"]), 5)
            self.assertTrue(np.isfinite(res.fold_scores["roc_auc"]).all())
        self.assertEqual(len(fps), 1)
        self.assertEqual(out.split_fingerprint, list(fps)[0])


if __name__ == "__main__":
    unittest.main(verbosity=2)


Overwriting test_section9_expanded_modeling.py


In [51]:
!python -m unittest -v test_section9_expanded_modeling.py


test_build_model_specs_contains_six_models (test_section9_expanded_modeling.TestSection9ExpandedModeling.test_build_model_specs_contains_six_models) ... ok
test_run_section9_produces_table_and_identical_folds (test_section9_expanded_modeling.TestSection9ExpandedModeling.test_run_section9_produces_table_and_identical_folds) ... ok

----------------------------------------------------------------------
Ran 2 tests in 5.125s

OK


In [48]:
!python -m pip install -q pandas numpy scikit-learn scipy


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# =========================
# SECTION 9 — RUN ON REAL CSV
# Path relative to hospital_readmission.ipynb:
CSV_PATH = "database/diabetic_data.csv"
# =========================

import pandas as pd

# 1) Load the REAL data
df_raw = pd.read_csv(CSV_PATH)

# 2) Engineer the binary target required by Sections 7–9 (NO/>30 -> 0, <30 -> 1)
# IMPORTANT: your "engineer_targets" function is defined in code.py,
# but in notebooks it is typically imported as "project_code" (as you did above).
# If you haven't imported it yet, run the import cell you already have for code.py.

df = project_code.engineer_targets(
    df_raw,
    source_col="readmitted",
    binary_col="readmitted_30d",
    multiclass_col="readmitted_3class",
    add_multiclass=True,
    drop_source=False,
    unknown_policy="error",
)

# 3) Build the preprocessing config (same as Section 5)
import section5_preprocessing_pipeline as s5

cfg = s5.DiabetesPreprocessConfig(
    onehot_sparse=True,   # recommended for ~100k rows + many categoricals
    rare_min_count=50,    # group rare categories -> __OTHER__
)

# 4) Create the IDENTICAL 10-fold splits ONCE (required for fair comparison + Wilcoxon later)
import section7_validation_protocol as s7

y = df["readmitted_30d"].astype(int).to_numpy()
splits = s7.make_stratified_cv_splits(
    y,
    n_splits=10,
    shuffle=True,
    random_state=42,
)

print("CV fingerprint:", s7.cv_splits_fingerprint(splits))

# 5) Run SECTION 9 expanded suite on the real dataset
import section9_expanded_modeling as s9

out9 = s9.run_section9_expanded_suite(
    df,
    target_col="readmitted_30d",
    preprocess_config_base=cfg,
    n_splits=10,
    cv_random_state=42,
    model_random_state=42,
    cv_splits=splits,       # <<< CRITICAL: reuse identical folds
    svd_components=100,     # safe default (SVD for MLP/GB)
    fast_mode=False,
    return_oof=False,       # can be True, but it increases memory/time
)

# 6) Show the results table (mean ± std across folds)
out9.summary_table


CV fingerprint: f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3d6a99a4e5b64150ef5


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [1]:
%%writefile section10_imbalance_handling.py

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple, Literal

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_sample_weight

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7

Split = s7.Split


# ---------------------------------------------------------------------
# Section 10 — Class Imbalance Handling Experiments
#
# Goals (rubric):
# - Apply and compare ≥1 explicit imbalance method (we implement 4):
#   (A) cost-sensitive learning via sample weights ("class_weight_balanced")
#   (B) random oversampling ("random_over")
#   (C) random undersampling ("random_under")
#   (D) threshold moving to maximize F1 on training fold ("threshold_f1")
#
# Contract:
# - SAME fixed CV splits (cv_splits) across all experiments (Section 7).
# - NO leakage: resampling + threshold selection happen inside each fold using TRAIN only.
# - Report ROC-AUC + precision/recall/F1 (minority class behavior), plus accuracy/balanced_accuracy.
# ---------------------------------------------------------------------


ImbalanceMethod = Literal[
    "none",
    "class_weight_balanced",
    "random_over",
    "random_under",
    "threshold_f1",
]


@dataclass(frozen=True)
class ImbalanceExperimentSpec:
    model: str
    method: ImbalanceMethod
    # For sampling methods:
    sampling_ratio: float = 1.0  # minority/majority after sampling (1.0 = balanced)
    random_state: int = 42


@dataclass(frozen=True)
class Section10EvaluationResult:
    spec: ImbalanceExperimentSpec
    n_splits: int
    split_fingerprint: str
    fold_scores: Dict[str, np.ndarray]
    mean_scores: Dict[str, float]
    std_scores: Dict[str, float]
    fold_thresholds: Optional[np.ndarray] = None  # only for threshold_f1

    def as_row(self) -> Dict[str, Any]:
        row: Dict[str, Any] = {
            "model": self.spec.model,
            "method": self.spec.method,
            "sampling_ratio": float(self.spec.sampling_ratio),
            "n_splits": int(self.n_splits),
            "split_fingerprint": self.split_fingerprint,
        }
        if self.fold_thresholds is not None:
            row["threshold_mean"] = float(np.mean(self.fold_thresholds))
            row["threshold_std"] = float(np.std(self.fold_thresholds, ddof=1)) if len(self.fold_thresholds) > 1 else 0.0

        for m, v in self.mean_scores.items():
            row[f"{m}_mean"] = float(v)
        for m, v in self.std_scores.items():
            row[f"{m}_std"] = float(v)
        return row


# ------------------------------ helpers ------------------------------


def _ensure_1d_binary(y: Iterable[int]) -> np.ndarray:
    y_arr = np.asarray(list(y), dtype=int).ravel()
    if y_arr.size == 0:
        raise ValueError("y is empty.")
    uniq = set(np.unique(y_arr).tolist())
    if not uniq.issubset({0, 1}):
        raise ValueError(f"Binary target must be in {{0,1}}. Found: {sorted(list(uniq))}")
    return y_arr


def _safe_index(X, idx: np.ndarray):
    # Works for pandas, numpy, scipy sparse
    if isinstance(X, (pd.DataFrame, pd.Series)):
        return X.iloc[idx]
    return X[idx]


def _get_score_vector(estimator, X, *, positive_label: int = 1) -> np.ndarray:
    """
    Continuous scores for ROC-AUC + threshold moving:
    - predict_proba[:, pos] if available
    - else decision_function
    """
    if hasattr(estimator, "predict_proba"):
        proba = estimator.predict_proba(X)
        proba = np.asarray(proba)
        if proba.ndim != 2 or proba.shape[1] < 2:
            raise ValueError("predict_proba must return [n_samples, 2+] for binary tasks.")
        # find column for positive_label if possible
        classes = getattr(estimator, "classes_", None)
        if classes is None:
            pos_idx = 1
        else:
            classes = np.asarray(classes)
            if positive_label in set(classes.tolist()):
                pos_idx = int(np.where(classes == positive_label)[0][0])
            else:
                pos_idx = 1
        return proba[:, pos_idx].astype(float)

    if hasattr(estimator, "decision_function"):
        s = estimator.decision_function(X)
        return np.asarray(s, dtype=float).ravel()

    raise ValueError("Estimator must provide predict_proba or decision_function for ROC-AUC/thresholding.")


def _to_csc_if_sparse(X):
    try:
        from scipy import sparse
    except Exception as e:
        raise RuntimeError("scipy is required for sparse matrix operations. Install with: pip install scipy") from e
    if sparse.issparse(X):
        return X.tocsc()
    return X


def random_oversample(X, y: np.ndarray, *, ratio: float = 1.0, random_state: int = 42):
    """
    Randomly oversample the minority class with replacement to reach:
        n_minority_new ~= ratio * n_majority
    """
    y = _ensure_1d_binary(y)
    if not (ratio > 0.0):
        raise ValueError("ratio must be > 0.")

    rng = np.random.RandomState(int(random_state))
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    if len(idx0) == 0 or len(idx1) == 0:
        return X, y

    # identify minority/majority
    if len(idx1) <= len(idx0):
        idx_min, idx_maj = idx1, idx0
        min_label = 1
    else:
        idx_min, idx_maj = idx0, idx1
        min_label = 0

    n_maj = len(idx_maj)
    n_min = len(idx_min)
    target_min = int(np.round(float(ratio) * n_maj))
    if target_min <= n_min:
        # already at/above target
        new_idx = np.concatenate([idx_maj, idx_min])
    else:
        extra = rng.choice(idx_min, size=(target_min - n_min), replace=True)
        new_idx = np.concatenate([idx_maj, idx_min, extra])

    rng.shuffle(new_idx)
    X_new = _safe_index(X, new_idx)
    y_new = y[new_idx]
    return X_new, y_new


def random_undersample(X, y: np.ndarray, *, ratio: float = 1.0, random_state: int = 42):
    """
    Randomly undersample the majority class without replacement to reach:
        n_majority_new ~= n_minority / ratio
    where ratio = minority/majority after sampling (ratio=1 => balanced).
    """
    y = _ensure_1d_binary(y)
    if not (ratio > 0.0):
        raise ValueError("ratio must be > 0.")

    rng = np.random.RandomState(int(random_state))
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    if len(idx0) == 0 or len(idx1) == 0:
        return X, y

    # identify minority/majority
    if len(idx1) <= len(idx0):
        idx_min, idx_maj = idx1, idx0
    else:
        idx_min, idx_maj = idx0, idx1

    n_min = len(idx_min)
    n_maj = len(idx_maj)

    # want: n_min / n_maj_new ~= ratio  => n_maj_new ~= n_min / ratio
    target_maj = int(np.round(float(n_min) / float(ratio)))
    target_maj = max(1, min(target_maj, n_maj))

    keep_maj = rng.choice(idx_maj, size=target_maj, replace=False)
    new_idx = np.concatenate([idx_min, keep_maj])
    rng.shuffle(new_idx)

    X_new = _safe_index(X, new_idx)
    y_new = y[new_idx]
    return X_new, y_new


def best_threshold_max_f1(
    y_true: np.ndarray,
    scores: np.ndarray,
    *,
    positive_label: int = 1,
    n_grid: int = 200,
) -> float:
    """
    Choose threshold that maximizes F1 on the provided (y_true, scores).
    Uses quantile grid for stability and speed.
    """
    y_true = _ensure_1d_binary(y_true)
    scores = np.asarray(scores, dtype=float).ravel()
    if scores.size != y_true.size:
        raise ValueError("scores and y_true must have the same length.")

    # Build candidate thresholds
    if scores.size <= 500:
        candidates = np.unique(scores)
    else:
        qs = np.linspace(0.0, 1.0, int(n_grid))
        candidates = np.unique(np.quantile(scores, qs))

    best_t = float(candidates[0])
    best_f1 = -1.0
    best_recall = -1.0

    for t in candidates:
        y_pred = (scores >= t).astype(int)
        f1 = float(f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0))
        if f1 > best_f1:
            best_f1 = f1
            best_t = float(t)
            best_recall = float(recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0))
        elif f1 == best_f1:
            # tie-break: prefer higher recall, then lower threshold (catch more positives)
            rec = float(recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0))
            if rec > best_recall or (rec == best_recall and float(t) < best_t):
                best_t = float(t)
                best_recall = rec

    return best_t


def _compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray) -> Dict[str, float]:
    return {
        "roc_auc": float(roc_auc_score(y_true, y_score)),
        "precision": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


def _supports_sample_weight(estimator) -> bool:
    # Conservative check: many sklearn estimators accept it; this avoids hard failures.
    import inspect
    try:
        sig = inspect.signature(estimator.fit)
        return "sample_weight" in sig.parameters
    except Exception:
        return False


# -------------------------- model factory ----------------------------


def build_section10_estimator(model: str, *, random_state: int = 42, fast_mode: bool = False):
    """
    Minimal, rubric-safe set for Section 10:
    - logistic_regression (probabilistic; classic for class weights + threshold moving)
    - decision_tree (PDF notes sensitivity to imbalance)
    - random_forest (ensemble/bagging; can respond well to imbalance methods)
    """
    model = str(model).strip().lower()
    if model == "logistic_regression":
        from sklearn.linear_model import LogisticRegression
        return LogisticRegression(
            solver="saga",
            penalty="l2",
            max_iter=2000,
            random_state=int(random_state),
            n_jobs=-1,
        )

    if model == "decision_tree":
        from sklearn.tree import DecisionTreeClassifier
        return DecisionTreeClassifier(random_state=int(random_state))

    if model == "random_forest":
        from sklearn.ensemble import RandomForestClassifier
        n_estimators = 80 if fast_mode else 300
        return RandomForestClassifier(
            n_estimators=int(n_estimators),
            random_state=int(random_state),
            n_jobs=-1,
        )

    if model == "linear_svm":
        from sklearn.svm import LinearSVC
        return LinearSVC(C=1.0, random_state=int(random_state))

    raise ValueError(f"Unknown model '{model}'. Use: logistic_regression, decision_tree, random_forest, linear_svm.")


# ----------------------------- main CV -------------------------------


def evaluate_section10_experiment_cv(
    df: pd.DataFrame,
    *,
    target_col: str,
    preprocess_config: Optional[s5.DiabetesPreprocessConfig],
    cv_splits: Sequence[Split],
    spec: ImbalanceExperimentSpec,
    positive_label: int = 1,
    fast_mode: bool = False,
) -> Section10EvaluationResult:
    """
    Manual fold loop to guarantee:
    - resampling inside TRAIN only
    - threshold selection inside TRAIN only
    - identical folds reused across all experiments
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found in df.")

    y = _ensure_1d_binary(df[target_col].astype(int).to_numpy())

    drop_cols = [c for c in preprocess_config.target_cols if c in df.columns]
    if target_col not in drop_cols:
        drop_cols.append(target_col)
    X = df.drop(columns=drop_cols, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError("No feature columns available after dropping target columns.")

    s7.validate_cv_splits(cv_splits, n_samples=len(y))
    fp = s7.cv_splits_fingerprint(cv_splits)

    # Preprocessor template (cloned each fold)
    pre_template = s5.build_preprocessor(df, preprocess_config)

    fold_metrics: Dict[str, List[float]] = {k: [] for k in ["roc_auc", "precision", "recall", "f1", "accuracy", "balanced_accuracy"]}
    fold_thresholds: List[float] = []

    for tr, te in cv_splits:
        tr = np.asarray(tr, dtype=np.int64)
        te = np.asarray(te, dtype=np.int64)

        X_tr_raw = _safe_index(X, tr)
        y_tr = y[tr]
        X_te_raw = _safe_index(X, te)
        y_te = y[te]

        # Fit preprocessing on TRAIN only
        pre = clone(pre_template)
        pre.fit(X_tr_raw, y_tr)
        X_tr = pre.transform(X_tr_raw)
        X_te = pre.transform(X_te_raw)

        # Apply imbalance method inside the fold (TRAIN only)
        sample_weight = None
        if spec.method == "class_weight_balanced":
            sample_weight = compute_sample_weight(class_weight="balanced", y=y_tr).astype(float)

        if spec.method == "random_over":
            X_tr, y_tr = random_oversample(X_tr, y_tr, ratio=spec.sampling_ratio, random_state=spec.random_state)
            sample_weight = None  # do not mix by default

        if spec.method == "random_under":
            X_tr, y_tr = random_undersample(X_tr, y_tr, ratio=spec.sampling_ratio, random_state=spec.random_state)
            sample_weight = None

        # Build estimator for this fold
        est = build_section10_estimator(spec.model, random_state=spec.random_state, fast_mode=fast_mode)

        # Some tree models prefer CSC if sparse
        if spec.model in {"decision_tree", "random_forest"}:
            X_tr_fit = _to_csc_if_sparse(X_tr)
            X_te_fit = _to_csc_if_sparse(X_te)
        else:
            X_tr_fit, X_te_fit = X_tr, X_te

        # Fit with or without sample_weight
        if sample_weight is not None:
            if not _supports_sample_weight(est):
                raise RuntimeError(f"Estimator '{spec.model}' does not support sample_weight; cannot run class_weight_balanced.")
            est.fit(X_tr_fit, y_tr, sample_weight=sample_weight)
        else:
            est.fit(X_tr_fit, y_tr)

        # Scores for ROC-AUC and (optionally) threshold moving
        tr_scores = _get_score_vector(est, X_tr_fit, positive_label=positive_label)
        te_scores = _get_score_vector(est, X_te_fit, positive_label=positive_label)

        if spec.method == "threshold_f1":
            thr = best_threshold_max_f1(y_tr, tr_scores, positive_label=positive_label)
            fold_thresholds.append(float(thr))
            y_pred = (te_scores >= thr).astype(int)
        else:
            # default classifier decision (usually 0.5 for proba models, 0 for margin-based)
            y_pred = np.asarray(est.predict(X_te_fit), dtype=int)

        m = _compute_metrics(y_te, y_pred, te_scores)
        for k in fold_metrics.keys():
            fold_metrics[k].append(float(m[k]))

    fold_scores = {k: np.asarray(v, dtype=float) for k, v in fold_metrics.items()}
    mean_scores = {k: float(np.mean(v)) for k, v in fold_scores.items()}
    std_scores = {k: float(np.std(v, ddof=1)) if len(v) > 1 else 0.0 for k, v in fold_scores.items()}

    thr_arr = np.asarray(fold_thresholds, dtype=float) if spec.method == "threshold_f1" else None

    return Section10EvaluationResult(
        spec=spec,
        n_splits=len(cv_splits),
        split_fingerprint=fp,
        fold_scores=fold_scores,
        mean_scores=mean_scores,
        std_scores=std_scores,
        fold_thresholds=thr_arr,
    )


def run_section10_imbalance_experiments(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    cv_splits: Sequence[Split],
    models: Sequence[str] = ("logistic_regression", "decision_tree", "random_forest"),
    methods: Sequence[ImbalanceMethod] = ("none", "class_weight_balanced", "random_over", "random_under", "threshold_f1"),
    sampling_ratio: float = 1.0,
    random_state: int = 42,
    fast_mode: bool = False,
) -> Tuple[pd.DataFrame, Dict[Tuple[str, str], Section10EvaluationResult]]:
    """
    Returns:
      - results_table: mean ± std per (model, method) for ROC-AUC + precision/recall/F1 (+ accuracy, balanced_accuracy)
      - results_map: raw per-fold vectors (needed later for Wilcoxon / deeper analysis)
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    rows: List[Dict[str, Any]] = []
    results_map: Dict[Tuple[str, str], Section10EvaluationResult] = {}

    for model in models:
        for method in methods:
            spec = ImbalanceExperimentSpec(
                model=str(model),
                method=method,
                sampling_ratio=float(sampling_ratio),
                random_state=int(random_state),
            )
            res = evaluate_section10_experiment_cv(
                df,
                target_col=target_col,
                preprocess_config=preprocess_config,
                cv_splits=cv_splits,
                spec=spec,
                fast_mode=fast_mode,
            )
            rows.append(res.as_row())
            results_map[(spec.model, spec.method)] = res

    table = pd.DataFrame(rows)

    # Nice, report-ready column ordering
    metric_order = ["roc_auc", "precision", "recall", "f1", "accuracy", "balanced_accuracy"]
    cols = (
        ["model", "method", "sampling_ratio", "n_splits", "split_fingerprint"]
        + (["threshold_mean", "threshold_std"] if "threshold_mean" in table.columns else [])
        + [f"{m}_mean" for m in metric_order]
        + [f"{m}_std" for m in metric_order]
    )
    cols = [c for c in cols if c in table.columns] + [c for c in table.columns if c not in cols]
    table = table[cols].sort_values(["model", "method"]).reset_index(drop=True)

    return table, results_map


Overwriting section10_imbalance_handling.py


In [2]:
%%writefile test_section10_imbalance_handling.py

import unittest
import numpy as np
import pandas as pd

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7
import section10_imbalance_handling as s10


class TestSection10ImbalanceHandling(unittest.TestCase):
    def _toy_diabetes_like_df(self, n: int = 300) -> pd.DataFrame:
        rng = np.random.RandomState(0)

        # 20% positives, 80% negatives
        y = np.array([0] * int(n * 0.8) + [1] * (n - int(n * 0.8)), dtype=int)
        rng.shuffle(y)

        ages = list(s5.AGE_ORDER)
        df = pd.DataFrame(
            {
                "encounter_id": np.arange(n),
                "patient_nbr": np.arange(10_000, 10_000 + n),
                "race": rng.choice(["Caucasian", "AfricanAmerican", "Hispanic", "?"], size=n),
                "gender": rng.choice(["Male", "Female"], size=n),
                "age": rng.choice(ages + ["?"], size=n),
                "weight": rng.choice(["?", "70", "80", "90", "Unknown/Invalid"], size=n),
                "admission_type_id": rng.choice([1, 2, 6], size=n),
                "discharge_disposition_id": rng.choice([1, 3, 25], size=n),
                "admission_source_id": rng.choice([1, 7], size=n),
                "time_in_hospital": rng.choice([1, 2, 3, 4, 5, 6, 7], size=n),
                "payer_code": rng.choice(["MC", "MD", "?", "SP"], size=n),
                "medical_specialty": rng.choice(
                    ["Cardiology", "InternalMedicine", "?", "RareSpecA", "RareSpecB"], size=n
                ),
                "num_lab_procedures": rng.poisson(lam=40, size=n),
                "num_procedures": rng.poisson(lam=1, size=n),
                "num_medications": rng.poisson(lam=10, size=n),
                "number_outpatient": rng.poisson(lam=1, size=n),
                "number_emergency": rng.poisson(lam=0.2, size=n),
                "number_inpatient": rng.poisson(lam=0.5, size=n),
                "diag_1": rng.choice(["250.83", "276", "648", "8", "401"], size=n),
                "diag_2": rng.choice(["?", "250.01", "403", "V27"], size=n),
                "diag_3": rng.choice(["?", "7", "9", "6"], size=n),
                "A1Cresult": rng.choice(["None", "Norm", ">7", ">8", "?"], size=n),
                "insulin": rng.choice(["No", "Steady", "Up", "Down"], size=n),
                "change": rng.choice(["Ch", "No"], size=n),
                "diabetesMed": rng.choice(["Yes", "No"], size=n),
                "readmitted_30d": y,
            }
        )
        return df

    def test_random_over_and_under_sampling_balance_counts(self):
        X = np.arange(20).reshape(-1, 1)
        y = np.array([0] * 16 + [1] * 4, dtype=int)  # minority=1

        Xo, yo = s10.random_oversample(X, y, ratio=1.0, random_state=0)
        c0, c1 = int((yo == 0).sum()), int((yo == 1).sum())
        self.assertEqual(c0, c1)

        Xu, yu = s10.random_undersample(X, y, ratio=1.0, random_state=0)
        c0u, c1u = int((yu == 0).sum()), int((yu == 1).sum())
        self.assertEqual(c0u, c1u)

    def test_best_threshold_max_f1_matches_bruteforce_on_small_case(self):
        y = np.array([0, 0, 1, 1], dtype=int)
        scores = np.array([0.1, 0.4, 0.35, 0.8], dtype=float)

        thr = s10.best_threshold_max_f1(y, scores, n_grid=50)

        # brute force over unique scores
        best = None
        best_f1 = -1
        for t in np.unique(scores):
            yp = (scores >= t).astype(int)
            f1 = float(s10.f1_score(y, yp, pos_label=1, zero_division=0))  # sklearn metric via module import
            if f1 > best_f1:
                best_f1 = f1
                best = float(t)

        self.assertAlmostEqual(float(thr), float(best), places=12)

    def test_run_section10_experiments_runs_and_returns_table(self):
        df = self._toy_diabetes_like_df(n=260)
        y = df["readmitted_30d"].astype(int).to_numpy()

        splits = s7.make_stratified_cv_splits(y, n_splits=5, random_state=123)
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=True, rare_min_count=1)

        table, results_map = s10.run_section10_imbalance_experiments(
            df,
            target_col="readmitted_30d",
            preprocess_config=cfg,
            cv_splits=splits,
            models=("logistic_regression", "decision_tree"),
            methods=("none", "class_weight_balanced", "random_over", "threshold_f1"),
            sampling_ratio=1.0,
            random_state=42,
            fast_mode=True,   # keep unit test fast
        )

        # 2 models * 4 methods = 8 rows
        self.assertEqual(table.shape[0], 8)
        self.assertIn("roc_auc_mean", table.columns)
        self.assertIn("f1_mean", table.columns)
        self.assertIn("recall_mean", table.columns)

        # results_map contains per-fold vectors
        self.assertEqual(len(results_map), 8)

        for (model, method), res in results_map.items():
            self.assertEqual(res.n_splits, 5)
            self.assertEqual(len(res.fold_scores["roc_auc"]), 5)
            self.assertTrue(np.isfinite(res.fold_scores["roc_auc"]).all())
            # Means should be within [0,1] for these metrics
            for m in ["roc_auc", "precision", "recall", "f1", "accuracy", "balanced_accuracy"]:
                self.assertGreaterEqual(res.mean_scores[m], 0.0)
                self.assertLessEqual(res.mean_scores[m], 1.0)

            if method == "threshold_f1":
                self.assertIsNotNone(res.fold_thresholds)
                self.assertEqual(len(res.fold_thresholds), 5)


if __name__ == "__main__":
    unittest.main(verbosity=2)


Overwriting test_section10_imbalance_handling.py


In [3]:
!python -m unittest -v test_section10_imbalance_handling.py


test_best_threshold_max_f1_matches_bruteforce_on_small_case (test_section10_imbalance_handling.TestSection10ImbalanceHandling.test_best_threshold_max_f1_matches_bruteforce_on_small_case) ... ok
test_random_over_and_under_sampling_balance_counts (test_section10_imbalance_handling.TestSection10ImbalanceHandling.test_random_over_and_under_sampling_balance_counts) ... ok
test_run_section10_experiments_runs_and_returns_table (test_section10_imbalance_handling.TestSection10ImbalanceHandling.test_run_section10_experiments_runs_and_returns_table) ... ok

----------------------------------------------------------------------
Ran 3 tests in 9.481s

OK


In [37]:
%%writefile section11_feature_engineering.py
from __future__ import annotations

from dataclasses import dataclass
from functools import lru_cache
from typing import List, Sequence, Tuple

import numpy as np
import pandas as pd


# -------------------------
# Missing token handling
# -------------------------
DEFAULT_MISSING_TOKENS: Tuple[str, ...] = (
    "", "?", "NA", "N/A", "NULL", "NAN", "UNKNOWN", "UNKNOWN/INVALID"
)

def _norm_token(x: object) -> str:
    return str(x).strip().upper()

_DEFAULT_MISSING_TOKENS_UP = frozenset(_norm_token(t) for t in DEFAULT_MISSING_TOKENS)

@lru_cache(maxsize=64)
def _missing_token_set(tokens: Tuple[str, ...]) -> frozenset[str]:
    return frozenset(_norm_token(t) for t in tokens)

def is_missing_token(x: object, *, missing_tokens: Sequence[str] = DEFAULT_MISSING_TOKENS) -> bool:
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        pass

    s = str(x)
    if s.strip() == "":
        return True

    # Fast path for default tokens
    up = _norm_token(s)
    if missing_tokens is DEFAULT_MISSING_TOKENS:
        return up in _DEFAULT_MISSING_TOKENS_UP

    toks = _missing_token_set(tuple(missing_tokens))
    return up in toks


# Medication columns in the UCI diabetes dataset (intersection with df.columns for robustness)
DEFAULT_MED_COLS: Tuple[str, ...] = (
    "metformin","repaglinide","nateglinide","chlorpropamide","glimepiride","acetohexamide",
    "glipizide","glyburide","tolbutamide","pioglitazone","rosiglitazone","acarbose","miglitol",
    "troglitazone","tolazamide","examide","citoglipton","insulin",
    "glyburide-metformin","glipizide-metformin","glimepiride-pioglitazone",
    "metformin-rosiglitazone","metformin-pioglitazone",
)

DIAG_COLS_DEFAULT: Tuple[str, ...] = ("diag_1", "diag_2", "diag_3")


# -------------------------
# Helpers
# -------------------------
def _to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def _df_elementwise_map(df: pd.DataFrame, func):
    # pandas>=2.1: DataFrame.map exists; older: use applymap
    return df.map(func) if hasattr(df, "map") else df.applymap(func)


# -------------------------
# Age: interval -> midpoint
# -------------------------
def age_midpoint(age_str: object) -> float:
    """
    Convert age bucket like "[50-60)" -> 55.0.
    Missing/unknown -> np.nan.
    """
    if is_missing_token(age_str):
        return np.nan
    s = str(age_str).strip()
    if not s:
        return np.nan

    # Expect formats like "[a-b)" or "(a-b]" etc.
    if "-" not in s:
        return np.nan

    try:
        inside = s.strip("[]()")
        a, b = inside.split("-", 1)
        a = float(a)
        b = float(b)
        return (a + b) / 2.0
    except Exception:
        return np.nan


# -------------------------
# Fixed binning (no leakage)
# -------------------------
def bin_numeric_fixed(
    x: pd.Series,
    bins: Sequence[float],
    labels: Sequence[str],
    *,
    include_lowest: bool = True,
    right: bool = True,
) -> pd.Series:
    """
    Deterministic, fixed-threshold binning via pd.cut.
    Keeps reproducibility and avoids data-driven thresholds (leakage).
    """
    x_num = pd.to_numeric(x, errors="coerce")
    return pd.cut(
        x_num,
        bins=bins,
        labels=labels,
        include_lowest=include_lowest,
        right=right,
        ordered=True,
    )


# -------------------------
# ICD-9 grouping (high-level)
# -------------------------
def _diag_token(v: object) -> str | None:
    """Normalize and validate a diagnosis token."""
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass

    s = str(v).strip()
    if s == "":
        return None

    up = s.upper()
    if up in _DEFAULT_MISSING_TOKENS_UP:
        return None

    return up

def _extract_leading_numeric(tok: str) -> float | None:
    """
    Extract a leading numeric value from an ICD-9 token (handles '250.13', '401', '530.81').
    Returns None if it can't parse a leading numeric prefix.
    """
    # Keep digits and first dot only until a non-digit/non-dot appears.
    buf = []
    dot_used = False
    for ch in tok:
        if ch.isdigit():
            buf.append(ch)
        elif ch == "." and not dot_used:
            buf.append(ch)
            dot_used = True
        else:
            break
    if not buf:
        return None
    try:
        return float("".join(buf))
    except Exception:
        return None

def icd9_group(code: object) -> str:
    """
    Robust ICD-9 grouping:
      - V* -> supplementary_v
      - E* -> external_e
      - numeric ranges -> major ICD-9 chapter
      - 250.* -> diabetes (special case)
    """
    tok = _diag_token(code)
    if tok is None:
        return "__MISSING__"

    # Non-numeric ICD-9 families
    if tok.startswith("V"):
        return "supplementary_v"
    if tok.startswith("E"):
        return "external_e"

    num = _extract_leading_numeric(tok)
    if num is None:
        return "other"

    # Special case diabetes: 250.xx
    if 250.0 <= num < 251.0:
        return "diabetes"

    # Major ICD-9 chapters (coarse)
    if 1.0 <= num <= 139.0:
        return "infectious"
    if 140.0 <= num <= 239.0:
        return "neoplasms"
    if 240.0 <= num <= 279.0:
        return "endocrine_metabolic"
    if 280.0 <= num <= 289.0:
        return "blood"
    if 290.0 <= num <= 319.0:
        return "mental"
    if 320.0 <= num <= 389.0:
        return "nervous"
    if 390.0 <= num <= 459.0:
        return "circulatory"
    if 460.0 <= num <= 519.0:
        return "respiratory"
    if 520.0 <= num <= 579.0:
        return "digestive"
    if 580.0 <= num <= 629.0:
        return "genitourinary"
    if 630.0 <= num <= 679.0:
        return "pregnancy"
    if 680.0 <= num <= 709.0:
        return "skin"
    if 710.0 <= num <= 739.0:
        return "musculoskeletal"
    if 740.0 <= num <= 759.0:
        return "congenital"
    if 760.0 <= num <= 779.0:
        return "perinatal"
    if 780.0 <= num <= 799.0:
        return "symptoms"
    if 800.0 <= num <= 999.0:
        return "injury"

    return "other"

def add_diag_group_features(df: pd.DataFrame, diag_cols: Sequence[str] = DIAG_COLS_DEFAULT) -> pd.DataFrame:
    out = df.copy()
    present = [c for c in diag_cols if c in out.columns]
    for c in present:
        out[f"{c}_group"] = out[c].map(icd9_group)

    group_cols = [f"{c}_group" for c in present]
    if group_cols:
        # Counts / flags
        out["n_diabetes_diags"] = (out[group_cols] == "diabetes").sum(axis=1).astype(int)

        # Robust primary flag (keeps index)
        if "diag_1_group" in out.columns:
            out["primary_diag_is_diabetes"] = (out["diag_1_group"] == "diabetes").astype(int)
        else:
            out["primary_diag_is_diabetes"] = pd.Series(0, index=out.index, dtype=int)

        # Diversity of diagnosis groups (excluding missing)
        tmp = out[group_cols].replace("__MISSING__", np.nan)
        out["n_unique_diag_groups"] = tmp.nunique(axis=1, dropna=True).fillna(0).astype(int)

        # Whether any diag group equals primary (useful redundancy indicator)
        if "diag_1_group" in out.columns:
            out["n_diags_same_as_primary_group"] = (
                out[group_cols].eq(out["diag_1_group"], axis=0).sum(axis=1).astype(int)
            )

    return out


# -------------------------
# Labs & tests (A1C / glucose)
# -------------------------
_A1C_MAP = {"NONE": 0, "NORM": 1, ">7": 2, ">8": 3}
_GLU_MAP = {"NONE": 0, "NORM": 1, ">200": 2, ">300": 3}

def add_lab_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "A1Cresult" in out.columns:
        a1c_raw = out["A1Cresult"].astype("object")
        a1c_norm = a1c_raw.map(lambda v: "MISSING" if is_missing_token(v) else str(v).strip().upper())
        out["a1c_measured"] = (a1c_norm.notna() & (a1c_norm != "MISSING") & (a1c_norm != "NONE")).astype(int)
        out["a1c_high"] = a1c_norm.isin({">7", ">8"}).astype(int)
        out["a1c_level"] = a1c_norm.map(_A1C_MAP).fillna(0).astype(int)

    if "max_glu_serum" in out.columns:
        glu_raw = out["max_glu_serum"].astype("object")
        glu_norm = glu_raw.map(lambda v: "MISSING" if is_missing_token(v) else str(v).strip().upper())
        out["glu_measured"] = (glu_norm.notna() & (glu_norm != "MISSING") & (glu_norm != "NONE")).astype(int)
        out["glu_high"] = glu_norm.isin({">200", ">300"}).astype(int)
        out["glu_level"] = glu_norm.map(_GLU_MAP).fillna(0).astype(int)

    # Optional bins for lab procedure count (fixed thresholds)
    if "num_lab_procedures" in out.columns:
        out["num_lab_procedures"] = _to_num(out["num_lab_procedures"])
        out["num_lab_procedures_bin"] = bin_numeric_fixed(
            out["num_lab_procedures"],
            bins=[-np.inf, 20, 40, 60, np.inf],
            labels=["low", "medium", "high", "very_high"],
        )

    return out


# -------------------------
# Utilization / burden features
# -------------------------
def add_utilization_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    num_cols = [
        "number_outpatient", "number_emergency", "number_inpatient",
        "time_in_hospital", "num_medications", "number_diagnoses"
    ]
    for c in num_cols:
        if c in out.columns:
            out[c] = _to_num(out[c])

    if all(c in out.columns for c in ["number_outpatient", "number_emergency", "number_inpatient"]):
        out["total_visits"] = (
            out["number_outpatient"].fillna(0)
            + out["number_emergency"].fillna(0)
            + out["number_inpatient"].fillna(0)
        ).astype(float)

        out["any_emergency"] = (out["number_emergency"].fillna(0) > 0).astype(int)
        out["any_inpatient"] = (out["number_inpatient"].fillna(0) > 0).astype(int)
        out["any_outpatient"] = (out["number_outpatient"].fillna(0) > 0).astype(int)
        out["any_prior_visit"] = (out["total_visits"].fillna(0) > 0).astype(int)

        out["total_visits_bin"] = bin_numeric_fixed(
            out["total_visits"],
            bins=[-np.inf, 0, 2, 5, np.inf],
            labels=["none", "low", "medium", "high"],
        )

    if "time_in_hospital" in out.columns:
        out["stay_bin"] = bin_numeric_fixed(
            out["time_in_hospital"],
            bins=[-np.inf, 3, 7, 14, np.inf],
            labels=["short", "medium", "long", "very_long"],
        )

    if "num_medications" in out.columns:
        out["num_medications_bin"] = bin_numeric_fixed(
            out["num_medications"],
            bins=[-np.inf, 5, 10, 20, np.inf],
            labels=["low", "medium", "high", "very_high"],
        )
        out["polypharmacy"] = (out["num_medications"].fillna(0) >= 10).astype(int)

    if "number_diagnoses" in out.columns:
        out["number_diagnoses_bin"] = bin_numeric_fixed(
            out["number_diagnoses"],
            bins=[-np.inf, 3, 6, np.inf],
            labels=["low", "medium", "high"],
        )

    return out


# -------------------------
# Medication burden from per-drug columns
# -------------------------
def add_medication_burden_features(
    df: pd.DataFrame,
    *,
    med_cols: Sequence[str] = DEFAULT_MED_COLS,
) -> pd.DataFrame:
    out = df.copy()
    present = [c for c in med_cols if c in out.columns]
    if not present:
        return out

    meds = out[present].astype("object")
    meds_norm = _df_elementwise_map(
        meds, lambda v: "MISSING" if is_missing_token(v) else str(v).strip().upper()
    )

    active_mask = meds_norm.isin({"STEADY", "UP", "DOWN"})
    changed_mask = meds_norm.isin({"UP", "DOWN"})

    out["n_active_diabetes_meds"] = active_mask.sum(axis=1).astype(int)
    out["n_changed_diabetes_meds"] = changed_mask.sum(axis=1).astype(int)
    out["any_active_diabetes_med"] = (out["n_active_diabetes_meds"] > 0).astype(int)
    out["any_changed_diabetes_med"] = (out["n_changed_diabetes_meds"] > 0).astype(int)

    # insulin-specific flags
    if "insulin" in out.columns:
        insulin = out["insulin"].astype("object").map(
            lambda v: "MISSING" if is_missing_token(v) else str(v).strip().upper()
        )
        out["insulin_active"] = insulin.isin({"STEADY", "UP", "DOWN"}).astype(int)
        out["insulin_changed"] = insulin.isin({"UP", "DOWN"}).astype(int)

    return out


# -------------------------
# Simple interactions (numeric only; low risk)
# -------------------------
def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # age midpoint
    if "age" in out.columns:
        out["age_mid"] = out["age"].map(age_midpoint)
        out["elderly"] = (out["age_mid"].fillna(-1) >= 65).astype(int)

    # ensure numeric coercion for interactions
    for c in ["time_in_hospital", "num_medications", "number_diagnoses", "total_visits"]:
        if c in out.columns:
            out[c] = _to_num(out[c])

    if all(c in out.columns for c in ["time_in_hospital", "num_medications"]):
        out["stay_x_meds"] = (out["time_in_hospital"].fillna(0) * out["num_medications"].fillna(0)).astype(float)

    if all(c in out.columns for c in ["time_in_hospital", "number_diagnoses"]):
        out["stay_x_diagnoses"] = (out["time_in_hospital"].fillna(0) * out["number_diagnoses"].fillna(0)).astype(float)

    if all(c in out.columns for c in ["total_visits", "time_in_hospital"]):
        out["visits_x_stay"] = (out["total_visits"].fillna(0) * out["time_in_hospital"].fillna(0)).astype(float)

    if all(c in out.columns for c in ["age_mid", "num_medications"]):
        out["age_x_meds"] = (out["age_mid"].fillna(0) * out["num_medications"].fillna(0)).astype(float)

    if all(c in out.columns for c in ["age_mid", "total_visits"]):
        out["age_x_visits"] = (out["age_mid"].fillna(0) * out["total_visits"].fillna(0)).astype(float)

    return out


# -------------------------
# Main entry point
# -------------------------
@dataclass(frozen=True)
class FeatureEngineeringOutput:
    df: pd.DataFrame
    added_columns: List[str]


def apply_feature_engineering(
    df: pd.DataFrame,
    *,
    diag_cols: Sequence[str] = DIAG_COLS_DEFAULT,
    med_cols: Sequence[str] = DEFAULT_MED_COLS,
) -> FeatureEngineeringOutput:
    """
    Apply Section 11 transformations and return:
      - df with engineered features added
      - list of added column names (useful for report + debugging)

    IMPORTANT: run this BEFORE building the Section 5 preprocessing pipeline,
    so the new columns are included in modeling.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("apply_feature_engineering expects a pandas DataFrame.")

    before = set(df.columns)

    out = df.copy()
    out = add_diag_group_features(out, diag_cols=diag_cols)
    out = add_lab_features(out)
    out = add_utilization_features(out)
    out = add_medication_burden_features(out, med_cols=med_cols)
    out = add_interaction_features(out)

    after = set(out.columns)
    added = sorted(list(after - before))
    return FeatureEngineeringOutput(df=out, added_columns=added)


Overwriting section11_feature_engineering.py


In [38]:
%%writefile test_section11_feature_engineering.py
import numpy as np
import pandas as pd
import pytest

import section11_feature_engineering as fe


def _is_ordered_categorical(s: pd.Series) -> bool:
    return isinstance(s.dtype, pd.CategoricalDtype) and bool(s.dtype.ordered)


# -------------------------
# Missing tokens + age
# -------------------------
def test_is_missing_token_default_behaviour():
    assert fe.is_missing_token(None) is True
    assert fe.is_missing_token(np.nan) is True
    assert fe.is_missing_token("") is True
    assert fe.is_missing_token("   ") is True
    assert fe.is_missing_token("?") is True
    assert fe.is_missing_token("unknown/invalid") is True
    assert fe.is_missing_token("NaN") is True
    assert fe.is_missing_token("NULL") is True

    # Non-missing examples
    assert fe.is_missing_token("0") is False
    assert fe.is_missing_token("No") is False
    assert fe.is_missing_token("Steady") is False


def test_age_midpoint_parsing_and_missing():
    # Your implementation accepts both bracketed and plain "a-b" formats (robust parsing).
    assert fe.age_midpoint("[50-60)") == 55.0
    assert fe.age_midpoint("(0-10]") == 5.0
    assert fe.age_midpoint("50-60") == 55.0

    # Invalid / missing -> np.nan
    assert np.isnan(fe.age_midpoint("50"))          # no hyphen / not a range
    assert np.isnan(fe.age_midpoint("abc-def"))     # non-numeric
    assert np.isnan(fe.age_midpoint("?"))
    assert np.isnan(fe.age_midpoint(None))


# -------------------------
# Fixed binning
# -------------------------
def test_bin_numeric_fixed_is_deterministic_and_ordered():
    s = pd.Series([0, 1, 2, 3, np.nan, "4"])
    b = fe.bin_numeric_fixed(s, bins=[-np.inf, 0, 2, np.inf], labels=["low", "mid", "high"])
    assert _is_ordered_categorical(b)
    # Expected bins (right=True by default): (-inf,0], (0,2], (2,inf]
    assert b.iloc[0] == "low"   # 0
    assert b.iloc[1] == "mid"   # 1
    assert b.iloc[2] == "mid"   # 2
    assert b.iloc[3] == "high"  # 3
    assert pd.isna(b.iloc[4])   # nan stays nan
    assert b.iloc[5] == "high"  # "4" coerces -> 4


# -------------------------
# ICD-9 grouping
# -------------------------
@pytest.mark.parametrize(
    "code,expected",
    [
        ("250.13", "diabetes"),
        ("250", "diabetes"),
        ("250.0", "diabetes"),
        ("V45", "supplementary_v"),
        ("E812", "external_e"),
        ("401", "circulatory"),
        ("530.81", "digestive"),
        ("786", "symptoms"),
        ("999", "injury"),
        ("abc", "other"),
        ("?", "__MISSING__"),
        (None, "__MISSING__"),
    ],
)
def test_icd9_grouping(code, expected):
    assert fe.icd9_group(code) == expected


# -------------------------
# Section functions
# -------------------------
def test_add_diag_group_features_adds_expected_columns_and_aggregates():
    df = pd.DataFrame(
        {
            "diag_1": ["250.13", "401", "?"],
            "diag_2": ["401", "V45", "250"],
            "diag_3": ["530.81", "250.0", None],
        }
    )
    out = fe.add_diag_group_features(df)

    # Group columns exist
    for c in ["diag_1_group", "diag_2_group", "diag_3_group"]:
        assert c in out.columns

    # Aggregates exist
    for c in ["n_diabetes_diags", "primary_diag_is_diabetes", "n_unique_diag_groups", "n_diags_same_as_primary_group"]:
        assert c in out.columns

    # Row-level checks
    # row0: diag groups -> diabetes, circulatory, digestive => n_diabetes_diags=1, primary is diabetes
    assert int(out.loc[0, "n_diabetes_diags"]) == 1
    assert int(out.loc[0, "primary_diag_is_diabetes"]) == 1
    assert int(out.loc[0, "n_unique_diag_groups"]) == 3
    assert int(out.loc[0, "n_diags_same_as_primary_group"]) == 1  # only diag_1 matches itself

    # row1: 401 circulatory, V45 supplementary_v, 250 diabetes => n_diabetes_diags=1, primary not diabetes
    assert int(out.loc[1, "n_diabetes_diags"]) == 1
    assert int(out.loc[1, "primary_diag_is_diabetes"]) == 0

    # row2: missing, diabetes, missing => n_diabetes_diags=1, primary missing -> 0
    assert int(out.loc[2, "n_diabetes_diags"]) == 1
    assert int(out.loc[2, "primary_diag_is_diabetes"]) == 0


def test_add_lab_features_measured_high_and_levels():
    df = pd.DataFrame(
        {
            "A1Cresult": ["None", "Norm", ">7", ">8", "?", None],
            "max_glu_serum": ["None", "Norm", ">200", ">300", "?", None],
            "num_lab_procedures": [10, 25, 45, 70, None, "60"],
        }
    )
    out = fe.add_lab_features(df)

    for c in ["a1c_measured", "a1c_high", "a1c_level", "glu_measured", "glu_high", "glu_level", "num_lab_procedures_bin"]:
        assert c in out.columns

    # a1c
    assert out["a1c_measured"].tolist() == [0, 1, 1, 1, 0, 0]
    assert out["a1c_high"].tolist() == [0, 0, 1, 1, 0, 0]
    assert out["a1c_level"].tolist() == [0, 1, 2, 3, 0, 0]

    # glucose
    assert out["glu_measured"].tolist() == [0, 1, 1, 1, 0, 0]
    assert out["glu_high"].tolist() == [0, 0, 1, 1, 0, 0]
    assert out["glu_level"].tolist() == [0, 1, 2, 3, 0, 0]

    assert _is_ordered_categorical(out["num_lab_procedures_bin"])


def test_add_utilization_features_creates_expected_features():
    df = pd.DataFrame(
        {
            "number_outpatient": [0, 1, 2, None],
            "number_emergency": [0, 0, 1, 3],
            "number_inpatient": [0, 2, 0, 1],
            "time_in_hospital": [2, 5, 10, "15"],
            "num_medications": [4, 10, 21, None],
            "number_diagnoses": [2, 4, 7, "6"],
        }
    )
    out = fe.add_utilization_features(df)

    for c in [
        "total_visits",
        "any_emergency",
        "any_inpatient",
        "any_outpatient",
        "any_prior_visit",
        "total_visits_bin",
        "stay_bin",
        "num_medications_bin",
        "polypharmacy",
        "number_diagnoses_bin",
    ]:
        assert c in out.columns

    # Row 0: no visits
    assert float(out.loc[0, "total_visits"]) == 0.0
    assert int(out.loc[0, "any_prior_visit"]) == 0

    # Row 1: outpatient 1 + inpatient 2 = 3 visits
    assert float(out.loc[1, "total_visits"]) == 3.0
    assert int(out.loc[1, "any_inpatient"]) == 1
    assert int(out.loc[1, "polypharmacy"]) == 1  # num_medications == 10

    assert _is_ordered_categorical(out["total_visits_bin"])
    assert _is_ordered_categorical(out["stay_bin"])
    assert _is_ordered_categorical(out["num_medications_bin"])
    assert _is_ordered_categorical(out["number_diagnoses_bin"])


def test_add_medication_burden_features_counts_active_and_changed_and_insulin_flags():
    df = pd.DataFrame(
        {
            "metformin": ["No", "Steady", "Up", "Down", "?", None],
            "insulin": ["No", "Steady", "Down", "Up", "?", None],
            # include another med from default list to test multi-col counting
            "glipizide": ["No", "No", "Steady", "Up", "Down", "?"],
        }
    )
    out = fe.add_medication_burden_features(df)

    for c in ["n_active_diabetes_meds", "n_changed_diabetes_meds", "any_active_diabetes_med", "any_changed_diabetes_med", "insulin_active", "insulin_changed"]:
        assert c in out.columns

    # Row-wise expectations:
    # row0: No/No/No -> active=0 changed=0 insulin_active=0 insulin_changed=0
    assert int(out.loc[0, "n_active_diabetes_meds"]) == 0
    assert int(out.loc[0, "n_changed_diabetes_meds"]) == 0
    assert int(out.loc[0, "insulin_active"]) == 0
    assert int(out.loc[0, "insulin_changed"]) == 0

    # row1: Steady insulin Steady -> active=2 (metformin+insulin), glipizide No -> total active=2, changed=0
    assert int(out.loc[1, "n_active_diabetes_meds"]) == 2
    assert int(out.loc[1, "n_changed_diabetes_meds"]) == 0
    assert int(out.loc[1, "insulin_active"]) == 1
    assert int(out.loc[1, "insulin_changed"]) == 0

    # row3: metformin Down + insulin Up + glipizide Up -> active=3, changed=3, insulin_changed=1
    assert int(out.loc[3, "n_active_diabetes_meds"]) == 3
    assert int(out.loc[3, "n_changed_diabetes_meds"]) == 3
    assert int(out.loc[3, "insulin_active"]) == 1
    assert int(out.loc[3, "insulin_changed"]) == 1


def test_add_interaction_features_creates_expected_columns():
    df = pd.DataFrame(
        {
            "age": ["[60-70)", "[30-40)", "?", None],
            "time_in_hospital": [2, 5, 10, 1],
            "num_medications": [5, 10, 2, None],
            "number_diagnoses": [2, 4, 7, 1],
            "total_visits": [0, 3, 1, 2],
        }
    )
    out = fe.add_interaction_features(df)

    for c in ["age_mid", "elderly", "stay_x_meds", "stay_x_diagnoses", "visits_x_stay", "age_x_meds", "age_x_visits"]:
        assert c in out.columns

    # elderly: age midpoint >= 65
    assert int(out.loc[0, "elderly"]) == 1
    assert int(out.loc[1, "elderly"]) == 0
    assert int(out.loc[2, "elderly"]) == 0  # missing treated as -1

    # interactions are numeric and non-negative with fillna(0)
    assert float(out.loc[0, "stay_x_meds"]) == 2 * 5
    assert float(out.loc[1, "stay_x_meds"]) == 5 * 10
    assert float(out.loc[3, "stay_x_meds"]) == 1 * 0  # num_medications missing -> 0


# -------------------------
# Main entry point
# -------------------------
def test_apply_feature_engineering_does_not_mutate_input_and_reports_added_columns():
    df = pd.DataFrame(
        {
            "age": ["[60-70)", "[30-40)"],
            "diag_1": ["250.13", "401"],
            "diag_2": ["401", "V45"],
            "diag_3": ["530.81", "250"],
            "A1Cresult": ["Norm", ">8"],
            "max_glu_serum": ["None", ">300"],
            "num_lab_procedures": [10, 70],
            "number_outpatient": [0, 1],
            "number_emergency": [0, 1],
            "number_inpatient": [0, 2],
            "time_in_hospital": [2, 5],
            "num_medications": [5, 10],
            "number_diagnoses": [2, 4],
            "metformin": ["No", "Steady"],
            "insulin": ["No", "Down"],
            "glipizide": ["No", "Up"],
        }
    )
    df_before = df.copy(deep=True)

    res = fe.apply_feature_engineering(df)

    # Original unchanged
    pd.testing.assert_frame_equal(df, df_before)

    # Returned df includes original columns + engineered ones
    assert set(df.columns).issubset(set(res.df.columns))
    assert isinstance(res.added_columns, list)

    # added_columns matches actual difference
    diff = sorted(list(set(res.df.columns) - set(df.columns)))
    assert res.added_columns == diff

    # Some key engineered columns exist
    expected_some = {
        "diag_1_group", "diag_2_group", "diag_3_group",
        "n_diabetes_diags", "primary_diag_is_diabetes",
        "a1c_measured", "a1c_level", "glu_measured", "glu_level",
        "total_visits", "total_visits_bin", "stay_bin",
        "n_active_diabetes_meds", "insulin_active",
        "age_mid", "elderly", "stay_x_meds",
    }
    assert expected_some.issubset(set(res.df.columns))


def test_apply_feature_engineering_handles_missing_columns_gracefully():
    # Minimal df should not crash; only age-related features should be added.
    df = pd.DataFrame({"age": ["[20-30)", "?"]})
    res = fe.apply_feature_engineering(df)
    assert "age_mid" in res.df.columns
    assert "elderly" in res.df.columns


def test_apply_feature_engineering_type_check():
    with pytest.raises(TypeError):
        fe.apply_feature_engineering(["not", "a", "dataframe"])


Overwriting test_section11_feature_engineering.py


In [25]:
!pytest -q test_section11_feature_engineering.py


.......................                                                  [100%]
23 passed in 0.95s


In [26]:
import os, sys, importlib.util
import pandas as pd

def import_module_from_path(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {module_name} from {file_path}")

    mod = importlib.util.module_from_spec(spec)

    # ✅ CRITICAL: register BEFORE exec_module (fixes dataclasses on Py3.13)
    sys.modules.pop(module_name, None)
    sys.modules[module_name] = mod

    spec.loader.exec_module(mod)
    return mod

PROJECT_DIR = os.getcwd()
project_code = import_module_from_path("project_code", os.path.join(PROJECT_DIR, "code.py"))


# ---- Load real data ----
df_raw = pd.read_csv("database/diabetic_data.csv")

# ---- Add binary target (required by the practice) ----
df = project_code.engineer_targets(
    df_raw,
    source_col="readmitted",
    binary_col="readmitted_30d",
    multiclass_col="readmitted_3class",
    add_multiclass=True,
    drop_source=False,
    unknown_policy="error",
)

# ---- Section 11: feature engineering ----
import section11_feature_engineering as s11

fe_out = s11.apply_feature_engineering(df)
df_fe = fe_out.df

print("Added columns (Section 11):")
print(fe_out.added_columns)

# Quick sanity check: inspect some engineered fields
cols_to_preview = [
    "readmitted_30d",
    "age", "age_mid", "elderly",
    "diag_1", "diag_1_group",
    "n_diabetes_diags", "primary_diag_is_diabetes",
    "total_visits", "total_visits_bin",
    "stay_bin", "num_medications_bin",
    "n_active_diabetes_meds", "n_changed_diabetes_meds",
    "a1c_measured", "a1c_high",
    "glu_measured", "glu_high",
    "stay_x_meds", "stay_x_diagnoses"
]
cols_to_preview = [c for c in cols_to_preview if c in df_fe.columns]
df_fe[cols_to_preview].head(10)


Added columns (Section 11):
['a1c_high', 'a1c_measured', 'age_mid', 'any_emergency', 'any_inpatient', 'any_outpatient', 'diag_1_group', 'diag_2_group', 'diag_3_group', 'elderly', 'glu_high', 'glu_measured', 'insulin_active', 'insulin_changed', 'n_active_diabetes_meds', 'n_changed_diabetes_meds', 'n_diabetes_diags', 'num_lab_procedures_bin', 'num_medications_bin', 'number_diagnoses_bin', 'polypharmacy', 'primary_diag_is_diabetes', 'stay_bin', 'stay_x_diagnoses', 'stay_x_meds', 'total_visits', 'total_visits_bin', 'visits_x_stay']


,readmitted_30d,age,age_mid,elderly,diag_1,diag_1_group,n_diabetes_diags,primary_diag_is_diabetes,total_visits,total_visits_bin,stay_bin,num_medications_bin,n_active_diabetes_meds,n_changed_diabetes_meds,a1c_measured,a1c_high,glu_measured,glu_high,stay_x_meds,stay_x_diagnoses
0,0,[0-10),5.0,0,250.83,diabetes,1,1,0.0,none,short,low,0,0,0,0,0,0,1.0,1.0
1,0,[10-20),15.0,0,276,endocrine_metabolic,1,0,0.0,none,short,high,1,1,0,0,0,0,54.0,27.0
2,0,[20-30),25.0,0,648,pregnancy,1,0,3.0,medium,short,high,1,0,0,0,0,0,26.0,12.0
3,0,[30-40),35.0,0,8,infectious,1,0,0.0,none,short,high,1,1,0,0,0,0,32.0,14.0
4,0,[40-50),45.0,0,197,neoplasms,1,0,0.0,none,short,medium,2,0,0,0,0,0,8.0,5.0
5,0,[50-60),55.0,0,414,circulatory,1,0,0.0,none,short,high,1,0,0,0,0,0,48.0,27.0
6,0,[60-70),65.0,1,414,circulatory,0,0,0.0,none,medium,very_high,3,0,0,0,0,0,84.0,28.0
7,0,[70-80),75.0,1,428,circulatory,1,0,0.0,none,medium,high,1,0,0,0,0,0,60.0,40.0
8,0,[80-90),85.0,1,398,circulatory,0,0,0.0,none,long,very_high,2,0,0,0,0,0,364.0,104.0
9,0,[90-100),95.0,1,434,circulatory,0,0,0.0,none,long,high,2,0,0,0,0,0,216.0,96.0


In [27]:
# A1C sanity
df_fe["A1Cresult"].value_counts(dropna=False).head(10)
df_fe["a1c_measured"].value_counts(dropna=False)
df_fe["a1c_high"].value_counts(dropna=False)

# Glucose sanity
df_fe["max_glu_serum"].value_counts(dropna=False).head(10)
df_fe["glu_measured"].value_counts(dropna=False)
df_fe["glu_high"].value_counts(dropna=False)


glu_high
0    99017
1     2749
Name: count, dtype: int64

In [28]:
assert (df_fe.loc[df_fe["glu_high"] == 1, "glu_measured"] == 1).all()
assert (df_fe.loc[df_fe["a1c_high"] == 1, "a1c_measured"] == 1).all()


valid = df_fe["max_glu_serum"].astype(str).str.upper().str.strip()
is_missing = valid.isin(["", "?", "NA", "N/A", "NULL", "NAN", "UNKNOWN", "UNKNOWN/INVALID"])
expected_measured = ~(is_missing | (valid == "NONE"))

assert (df_fe["glu_measured"] == expected_measured).all()

raw = df_fe["A1Cresult"].astype(str).str.upper().str.strip()

is_missing = raw.isin(["", "?", "NA", "N/A", "NULL", "NAN", "UNKNOWN", "UNKNOWN/INVALID"])
expected_measured_a1c = ~(is_missing | (raw == "NONE"))

assert (df_fe["a1c_measured"] == expected_measured_a1c).all()


df_fe["glu_high"].equals(df_fe["max_glu_serum"].isin([">200", ">300"]))

df_fe["a1c_high"].equals(df_fe["A1Cresult"].isin([">7", ">8"]))


pd.crosstab(df_fe["max_glu_serum"], df_fe["glu_high"], margins=True)
pd.crosstab(df_fe["max_glu_serum"], df_fe["glu_measured"], margins=True)

pd.crosstab(df_fe["A1Cresult"], df_fe["a1c_high"], margins=True)
pd.crosstab(df_fe["A1Cresult"], df_fe["a1c_measured"], margins=True)


a1c_measured,1,All
A1Cresult,,
>7,3812,3812
>8,8216,8216
Norm,4990,4990
All,17018,17018


In [39]:
%%writefile section12_feature_selection.py
from __future__ import annotations

from dataclasses import dataclass, replace
from functools import partial
from typing import Any, Dict, Iterable, List, Literal, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from sklearn.feature_selection import SelectFromModel, SelectKBest, RFE, chi2, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7


SelectionMethod = Literal[
    "filter_chi2",        # Filter family (fast, non-negative requirement)
    "filter_mi",          # Filter family (can be slower)
    "embedded_l1",        # Embedded family (L1 Logistic)
    "wrapper_rfe_mi",     # Wrapper family (can be heavy; two-stage: MI -> RFE)
]


# ------------------------- utilities -------------------------

def _ensure_binary_target(y: Iterable[int]) -> np.ndarray:
    y_arr = np.asarray(list(y), dtype=int).ravel()
    if y_arr.size == 0:
        raise ValueError("Empty target.")
    uniq = set(np.unique(y_arr).tolist())
    if not uniq.issubset({0, 1}):
        raise ValueError(f"Target must be binary in {{0,1}}. Found: {sorted(list(uniq))}")
    return y_arr


def _default_X_y(
    df: pd.DataFrame,
    *,
    target_col: str,
    cfg: s5.DiabetesPreprocessConfig,
) -> Tuple[pd.DataFrame, np.ndarray]:
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found in df.")

    y = _ensure_binary_target(df[target_col].astype(int).to_numpy())

    drop_cols = [c for c in cfg.target_cols if c in df.columns]
    if target_col not in drop_cols:
        drop_cols.append(target_col)

    X = df.drop(columns=drop_cols, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError("No feature columns left after dropping target columns.")
    return X, y


def _logreg_l2(*, random_state: int = 42) -> LogisticRegression:
    return LogisticRegression(
        solver="saga",
        penalty="l2",
        C=1.0,
        max_iter=3000,
        random_state=int(random_state),
        n_jobs=-1,
    )


def _logreg_l1(*, C: float = 0.05, random_state: int = 42) -> LogisticRegression:
    return LogisticRegression(
        solver="saga",
        penalty="l1",
        C=float(C),
        max_iter=4000,
        random_state=int(random_state),
        n_jobs=-1,
    )


def _nonneg_config(base: s5.DiabetesPreprocessConfig) -> s5.DiabetesPreprocessConfig:
    """
    Make preprocessing non-negative so chi2 works:
    - disable scaling (StandardScaler can create negatives)
    - remove ordinal encoding (age ordinal can create -1 for unknown); treat age as categorical one-hot instead
    """
    return replace(base, scale_numeric=False, ordinal_cols={})


# ------------------------- feature names + grouping -------------------------

def _feature_names_after_preprocessing(
    pre_fitted: Pipeline,
    df_for_groups: pd.DataFrame,
    cfg: s5.DiabetesPreprocessConfig,
) -> np.ndarray:
    """
    Robust extraction of feature names from our Section 5 preprocessor
    WITHOUT requiring custom transformers to implement get_feature_names_out().
    """
    groups = s5.infer_feature_groups(df_for_groups, cfg)
    ct = pre_fitted.named_steps["features"]

    names: List[str] = []

    # Must follow the same order used in section5.build_preprocessor(): num, ord, cat
    if groups.numeric and "num" in ct.named_transformers_:
        names.extend([f"num__{c}" for c in groups.numeric])

    if groups.ordinal and "ord" in ct.named_transformers_ and ct.named_transformers_["ord"] != "drop":
        names.extend([f"ord__{c}" for c in groups.ordinal])

    if groups.categorical and "cat" in ct.named_transformers_:
        cat_pipe = ct.named_transformers_["cat"]
        onehot = cat_pipe.named_steps["onehot"]
        ohe_names = onehot.get_feature_names_out(groups.categorical)
        names.extend([f"cat__{n}" for n in ohe_names])

    return np.asarray(names, dtype=object)


def _base_column_from_feature_name(feature_name: str, base_cols: Sequence[str]) -> str:
    """
    Map expanded one-hot feature names back to original column names.
    Works with names like:
      - num__time_in_hospital
      - ord__age
      - cat__medical_specialty_Cardiology
    """
    s = str(feature_name)
    if s.startswith("num__"):
        return s[len("num__"):]
    if s.startswith("ord__"):
        return s[len("ord__"):]
    if s.startswith("cat__"):
        rest = s[len("cat__"):]
        # match the longest base col first (to handle underscores inside col names)
        for col in sorted(base_cols, key=len, reverse=True):
            if rest == col or rest.startswith(col + "_"):
                return col
        # fallback: before first underscore
        return rest.split("_", 1)[0]
    return s


def _build_interpretation_tables(
    fitted_pipe: Pipeline,
    df: pd.DataFrame,
    cfg: s5.DiabetesPreprocessConfig,
    *,
    method: SelectionMethod,
    top_n_features: int = 40,
    top_n_groups: int = 25,
) -> Dict[str, pd.DataFrame]:
    """
    Fit is already done. Extract:
      - top features (expanded)
      - top groups (original columns)
    """
    pre = fitted_pipe.named_steps["pre"]
    feature_names = _feature_names_after_preprocessing(pre, df, cfg)

    # Build full support + importance aligned to *preprocessed* feature space
    support = None
    importance = None

    if method in {"filter_chi2", "filter_mi", "embedded_l1"}:
        sel = fitted_pipe.named_steps["select"]
        support = sel.get_support() if hasattr(sel, "get_support") else None

        if method in {"filter_chi2", "filter_mi"} and hasattr(sel, "scores_"):
            importance = np.asarray(sel.scores_, dtype=float)
        elif method == "embedded_l1":
            est = getattr(sel, "estimator_", None)
            if est is not None and hasattr(est, "coef_"):
                importance = np.abs(np.asarray(est.coef_, dtype=float)).ravel()

    elif method == "wrapper_rfe_mi":
        # Two-stage: kbest then RFE
        kb = fitted_pipe.named_steps["kbest"]
        rfe = fitted_pipe.named_steps["select"]

        kb_support = kb.get_support()
        kb_idx = np.where(kb_support)[0]

        rfe_support = rfe.get_support()
        picked_in_kb = np.where(rfe_support)[0]
        picked_idx = kb_idx[picked_in_kb]

        support = np.zeros(len(feature_names), dtype=bool)
        support[picked_idx] = True

        # importance proxy from RFE ranking (smaller rank => more important)
        imp_full = np.zeros(len(feature_names), dtype=float)
        ranking = np.asarray(getattr(rfe, "ranking_", np.ones(len(kb_idx))), dtype=float)
        imp_kb = 1.0 / np.maximum(1.0, ranking)
        imp_full[kb_idx] = imp_kb
        importance = imp_full

    if support is None:
        support = np.ones(len(feature_names), dtype=bool)
    if importance is None:
        importance = np.zeros(len(feature_names), dtype=float)

    base_cols = list(s5.infer_feature_groups(df, cfg).numeric) \
             + list(s5.infer_feature_groups(df, cfg).ordinal) \
             + list(s5.infer_feature_groups(df, cfg).categorical)

    base_col = [
        _base_column_from_feature_name(fn, base_cols=base_cols) for fn in feature_names
    ]

    feat_df = pd.DataFrame(
        {
            "feature": feature_names,
            "base_column": base_col,
            "selected": support.astype(bool),
            "importance": importance.astype(float),
        }
    )

    # Sort: selected first, then importance
    feat_df_sorted = feat_df.sort_values(
        ["selected", "importance"],
        ascending=[False, False],
    )

    top_features = feat_df_sorted.head(int(top_n_features)).reset_index(drop=True)

    # Aggregate at original column level (good for the “hypothesis about selected groups”)
    grp = (
        feat_df.assign(importance_selected=feat_df["importance"] * feat_df["selected"].astype(int))
        .groupby("base_column", as_index=False)
        .agg(
            n_selected=("selected", "sum"),
            n_total=("selected", "size"),
            importance_sum=("importance_selected", "sum"),
        )
    )
    grp["selected_rate"] = grp["n_selected"] / grp["n_total"].replace(0, np.nan)
    grp = grp.sort_values(["importance_sum", "n_selected", "selected_rate"], ascending=False)
    top_groups = grp.head(int(top_n_groups)).reset_index(drop=True)

    return {"top_features": top_features, "top_groups": top_groups, "all_features_table": feat_df}


# ------------------------- pipeline builders -------------------------

def build_selection_pipeline(
    df: pd.DataFrame,
    *,
    method: SelectionMethod,
    cfg: s5.DiabetesPreprocessConfig,
    random_state: int = 42,
    k_best: int = 800,
    chi2_k: int = 800,
    l1_C: float = 0.05,
    l1_threshold: str = "median",
    l1_max_features: Optional[int] = None,
    wrapper_k_pre: int = 1200,
    wrapper_n: int = 250,
    wrapper_step: float = 0.2,
) -> Tuple[Pipeline, s5.DiabetesPreprocessConfig]:
    """
    Returns (pipeline, cfg_used).
    """
    cfg_used = cfg
    if method == "filter_chi2":
        cfg_used = _nonneg_config(cfg_used)

    pre = s5.build_preprocessor(df, cfg_used)
    clf = _logreg_l2(random_state=random_state)

    if method == "filter_chi2":
        selector = SelectKBest(score_func=chi2, k=int(chi2_k))
        pipe = Pipeline([("pre", pre), ("select", selector), ("clf", clf)])
        return pipe, cfg_used

    if method == "filter_mi":
        mi = partial(mutual_info_classif, discrete_features="auto", random_state=int(random_state))
        selector = SelectKBest(score_func=mi, k=int(k_best))
        pipe = Pipeline([("pre", pre), ("select", selector), ("clf", clf)])
        return pipe, cfg_used

    if method == "embedded_l1":
        base_est = _logreg_l1(C=l1_C, random_state=random_state)
        selector = SelectFromModel(
            estimator=base_est,
            threshold=l1_threshold,   # "median" is a good default
            max_features=l1_max_features,
        )
        pipe = Pipeline([("pre", pre), ("select", selector), ("clf", clf)])
        return pipe, cfg_used

    if method == "wrapper_rfe_mi":
        # two-stage to keep RFE feasible:
        mi = partial(mutual_info_classif, discrete_features="auto", random_state=int(random_state))
        kbest = SelectKBest(score_func=mi, k=int(wrapper_k_pre))

        rfe_est = _logreg_l2(random_state=random_state)
        rfe = RFE(estimator=rfe_est, n_features_to_select=int(wrapper_n), step=float(wrapper_step))

        pipe = Pipeline([("pre", pre), ("kbest", kbest), ("select", rfe), ("clf", clf)])
        return pipe, cfg_used

    raise ValueError(f"Unknown method: {method}")


# ------------------------- public runner -------------------------

@dataclass(frozen=True)
class Section12Result:
    split_fingerprint: str
    summary_table: pd.DataFrame
    interpretations: Dict[str, Dict[str, pd.DataFrame]]  # method -> tables


def run_section12_feature_selection(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    preprocess_config_base: Optional[s5.DiabetesPreprocessConfig] = None,
    cv_splits: Optional[Sequence[s7.Split]] = None,
    n_splits: int = 10,
    cv_random_state: int = 42,
    model_random_state: int = 42,
    methods: Sequence[SelectionMethod] = ("filter_chi2", "embedded_l1"),
    # method hyperparams:
    k_best: int = 800,
    chi2_k: int = 800,
    l1_C: float = 0.05,
    l1_threshold: str = "median",
    l1_max_features: Optional[int] = None,
    wrapper_k_pre: int = 1200,
    wrapper_n: int = 250,
    wrapper_step: float = 0.2,
    # interpretation tables:
    build_interpretation: bool = True,
    top_n_features: int = 40,
    top_n_groups: int = 25,
) -> Section12Result:
    """
    Produces:
      - CV comparison table: all-features vs selected-features (same splits)
      - Interpretation tables to support your “hypothesis about selected groups”
    """
    if preprocess_config_base is None:
        preprocess_config_base = s5.DiabetesPreprocessConfig()

    X_base, y = _default_X_y(df, target_col=target_col, cfg=preprocess_config_base)

    if cv_splits is None:
        cv_splits = s7.make_stratified_cv_splits(
            y, n_splits=int(n_splits), shuffle=True, random_state=int(cv_random_state)
        )
    else:
        s7.validate_cv_splits(cv_splits, n_samples=len(y), n_splits_expected=int(n_splits))

    fp = s7.cv_splits_fingerprint(cv_splits)

    rows: List[Dict[str, Any]] = []
    interpretations: Dict[str, Dict[str, pd.DataFrame]] = {}

    # Helper to evaluate a pipeline and write a row
    def _eval_and_row(method_name: str, variant: str, pipe: Pipeline) -> s7.CVEvaluationResult:
        res = s7.evaluate_binary_pipeline_cv(
            pipe, X_base, y,
            cv_splits=cv_splits,
            metrics=("roc_auc", "precision", "recall", "f1", "accuracy", "balanced_accuracy"),
            return_oof=False,
        )
        row = {
            "method": method_name,
            "variant": variant,  # "all_features" or "selected"
            "split_fingerprint": fp,
        }
        for m in res.mean_scores:
            row[f"{m}_mean"] = float(res.mean_scores[m])
            row[f"{m}_std"] = float(res.std_scores[m])
        rows.append(row)
        return res

    # Run per-method comparisons
    for method in methods:
        # build selected pipeline (+ cfg used)
        sel_pipe, cfg_used = build_selection_pipeline(
            df,
            method=method,
            cfg=preprocess_config_base,
            random_state=model_random_state,
            k_best=k_best,
            chi2_k=chi2_k,
            l1_C=l1_C,
            l1_threshold=l1_threshold,
            l1_max_features=l1_max_features,
            wrapper_k_pre=wrapper_k_pre,
            wrapper_n=wrapper_n,
            wrapper_step=wrapper_step,
        )

        # build matching all-features pipeline (same cfg_used!)
        pre_all = s5.build_preprocessor(df, cfg_used)
        all_pipe = Pipeline([("pre", pre_all), ("clf", _logreg_l2(random_state=model_random_state))])

        _eval_and_row(str(method), "all_features", all_pipe)
        _eval_and_row(str(method), "selected", sel_pipe)

        if build_interpretation:
            # Fit selected pipeline ON FULL DATA for stable, report-friendly “most predictive variables”
            fitted = Pipeline(sel_pipe.steps)  # clone-ish
            fitted.fit(X_base, y)
            tables = _build_interpretation_tables(
                fitted, df, cfg_used,
                method=method,
                top_n_features=top_n_features,
                top_n_groups=top_n_groups,
            )
            interpretations[str(method)] = tables

    summary = pd.DataFrame(rows).sort_values(["method", "variant"]).reset_index(drop=True)
    return Section12Result(split_fingerprint=fp, summary_table=summary, interpretations=interpretations)


Overwriting section12_feature_selection.py


In [40]:
%%writefile test_section12_feature_selection.py
import unittest
import numpy as np
import pandas as pd

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7
import section12_feature_selection as s12


class TestSection12FeatureSelection(unittest.TestCase):
    def _toy_df(self, n=220):
        rng = np.random.RandomState(0)
        y = np.array([0]*int(n*0.8) + [1]*(n-int(n*0.8)), dtype=int)
        rng.shuffle(y)

        df = pd.DataFrame({
            "encounter_id": np.arange(n),
            "patient_nbr": np.arange(10000, 10000+n),
            "age": rng.choice(list(s5.AGE_ORDER) + ["?"], size=n),
            "weight": rng.choice(["?", "70", "80", "90", "Unknown/Invalid"], size=n),
            "time_in_hospital": rng.randint(1, 8, size=n),
            "num_lab_procedures": rng.poisson(40, size=n),
            "medical_specialty": rng.choice(["Cardiology", "InternalMedicine", "?", "RareSpec"], size=n),
            "diag_1": rng.choice(["250.83", "276", "648", "8", "401"], size=n),
            "insulin": rng.choice(["No", "Steady", "Up", "Down"], size=n),
            "readmitted_30d": y,
        })
        return df

    def test_run_section12(self):
        df = self._toy_df()
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=True, rare_min_count=1)
        y = df["readmitted_30d"].astype(int).to_numpy()
        splits = s7.make_stratified_cv_splits(y, n_splits=5, random_state=42)

        out = s12.run_section12_feature_selection(
            df,
            target_col="readmitted_30d",
            preprocess_config_base=cfg,
            cv_splits=splits,
            n_splits=5,
            methods=("filter_chi2", "embedded_l1"),
            chi2_k=30,
            l1_C=0.1,
            top_n_features=10,
            top_n_groups=10,
        )

        self.assertIn("method", out.summary_table.columns)
        self.assertIn("roc_auc_mean", out.summary_table.columns)
        self.assertTrue(len(out.summary_table) == 4)  # 2 methods x (all vs selected)
        self.assertIn("filter_chi2", out.interpretations)
        self.assertIn("top_groups", out.interpretations["filter_chi2"])

if __name__ == "__main__":
    unittest.main(verbosity=2)


Overwriting test_section12_feature_selection.py


In [41]:
!python test_section12_feature_selection.py


test_run_section12 (__main__.TestSection12FeatureSelection.test_run_section12) ... C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:782: U

In [ ]:
# ================================
# SECTION 12 — WRAPPER (RFE + MI)
# FULL PREPARATION BLOCK
# ================================

# 1) Import modules needed for preprocessing + CV + feature selection
import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7
import section12_feature_selection as s12

# 2) Ensure you already loaded df (df_raw + engineer_targets)
#    If df does NOT exist, you must re-run the earlier section where df was created.

# 3) Build the preprocessing configuration (cfg)
cfg = s5.DiabetesPreprocessConfig(
    onehot_sparse=True,
    rare_min_count=50,
    scale_numeric=True,
    scaler="standard",
)

# 4) Extract target and build the FIXED 10 folds (splits)
y = df["readmitted_30d"].astype(int).to_numpy()

splits = s7.make_stratified_cv_splits(
    y,
    n_splits=10,
    random_state=42,   # MUST stay the same to keep folds identical
)

# 5) Finally — run the WRAPPER-based feature selection (optional heavy method)
out12_wrapper = s12.run_section12_feature_selection(
    df,
    target_col="readmitted_30d",
    preprocess_config_base=cfg,
    cv_splits=splits,
    n_splits=10,
    methods=("wrapper_rfe_mi",),   # ONLY wrapper method
    wrapper_k_pre=1200,            # (1) MI filter before RFE
    wrapper_n=200,                 # (2) RFE selects 200 final features
    wrapper_step=0.2,              # step size for RFE pruning
    top_n_features=40,             # how many one-hot features to show
    top_n_groups=25,               # how many grouped semantic features to show
)

# Optional: view summary table
out12_wrapper.summary_table


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [45]:
%%writefile section13_dimensionality_reduction.py

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple, Literal

import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7

Split = s7.Split
ReductionMethod = Literal["none", "svd"]


def _validate_binary_target(y: pd.Series) -> np.ndarray:
    if pd.Series(y).isna().any():
        raise ValueError("Target contains missing values; Section 13 requires a clean binary target.")
    y_arr = np.asarray(pd.Series(y).astype(int).tolist(), dtype=int)
    uniq = set(np.unique(y_arr).tolist())
    if not uniq.issubset({0, 1}):
        raise ValueError(f"Target must be binary in {{0,1}}, found values: {sorted(list(uniq))}")
    return y_arr


def _default_X_y(
    df: pd.DataFrame,
    *,
    target_col: str,
    preprocess_config: s5.DiabetesPreprocessConfig,
) -> Tuple[pd.DataFrame, np.ndarray]:
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found. Available: {list(df.columns)}")

    y = _validate_binary_target(df[target_col])

    drop_cols = [c for c in preprocess_config.target_cols if c in df.columns]
    if target_col not in drop_cols:
        drop_cols.append(target_col)

    X = df.drop(columns=drop_cols, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError("No feature columns available after dropping target columns.")
    return X, y


def _validate_components(components: Sequence[int]) -> List[int]:
    out: List[int] = []
    for c in components:
        try:
            ci = int(c)
        except Exception:
            raise ValueError(f"All n_components must be integers. Got: {c!r}")
        if ci <= 0:
            raise ValueError(f"All n_components must be > 0. Got: {ci}")
        out.append(ci)
    # de-duplicate but keep deterministic order
    seen = set()
    uniq = []
    for ci in out:
        if ci not in seen:
            uniq.append(ci)
            seen.add(ci)
    return uniq


@dataclass(frozen=True)
class DimRedVariant:
    name: str
    method: ReductionMethod
    n_components: Optional[int] = None


@dataclass(frozen=True)
class Section13RunResult:
    """
    Section 13: Dimensionality Reduction ablation.

    We implement "PCA-like" reduction for sparse one-hot data via TruncatedSVD:
    - After one-hot encoding, X is typically a high-dimensional sparse matrix.
    - Classic PCA requires centering and generally dense matrices.
    - TruncatedSVD is the standard practical alternative on sparse matrices,
      and is commonly used as a PCA analogue for sparse feature spaces.

    Returns:
      - per-variant CV results with identical folds (for fair comparisons)
      - a compact mean±std summary table for the report
    """
    target_col: str
    n_splits: int
    cv_random_state: int
    split_fingerprint: str
    variants: List[DimRedVariant]
    results_by_variant: Dict[str, s7.CVEvaluationResult]
    summary_table: pd.DataFrame


def build_section13_pipelines(
    df: pd.DataFrame,
    *,
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    model_random_state: int = 42,
    svd_components: Sequence[int] = (50, 100, 200),
) -> Tuple[List[DimRedVariant], Dict[str, Pipeline]]:
    """
    Build pipelines for:
      - no reduction (baseline)
      - TruncatedSVD(k) for each k in svd_components
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    comps = _validate_components(svd_components)

    pre = s5.build_preprocessor(df, preprocess_config)

    clf = LogisticRegression(
        solver="saga",         # sparse-friendly; also fine on dense SVD outputs
        penalty="l2",
        max_iter=3000,
        random_state=model_random_state,
        n_jobs=-1,
    )

    variants: List[DimRedVariant] = []
    pipes: Dict[str, Pipeline] = {}

    # Baseline: no reduction
    v0 = DimRedVariant(name="no_reduction", method="none", n_components=None)
    variants.append(v0)
    pipes[v0.name] = Pipeline(steps=[("pre", pre), ("clf", clf)])

    # SVD variants
    for k in comps:
        vk = DimRedVariant(name=f"svd_{k}", method="svd", n_components=int(k))
        variants.append(vk)
        pipes[vk.name] = Pipeline(
            steps=[
                ("pre", pre),
                ("svd", TruncatedSVD(n_components=int(k), random_state=model_random_state)),
                ("clf", clf),
            ]
        )

    return variants, pipes


def _summary_table(
    variants: Sequence[DimRedVariant],
    results_by_variant: Dict[str, s7.CVEvaluationResult],
    *,
    metrics: Sequence[str],
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for v in variants:
        r = results_by_variant[v.name]
        row: Dict[str, Any] = {
            "variant": v.name,
            "method": v.method,
            "n_components": (np.nan if v.n_components is None else int(v.n_components)),
        }
        for m in metrics:
            row[f"{m}_mean"] = float(r.mean_scores[m])
            row[f"{m}_std"] = float(r.std_scores[m])
        rows.append(row)

    cols = ["variant", "method", "n_components"] + [f"{m}_{s}" for m in metrics for s in ("mean", "std")]
    return pd.DataFrame(rows)[cols]


def run_section13_dimensionality_reduction(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    svd_components: Sequence[int] = (50, 100, 200),
    metrics: Sequence[str] = ("roc_auc", "precision", "recall", "f1", "accuracy", "balanced_accuracy"),
    n_splits: int = 10,
    cv_random_state: int = 42,
    model_random_state: int = 42,
    cv_splits: Optional[Sequence[Split]] = None,
    return_oof: bool = False,
) -> Section13RunResult:
    """
    End-to-end runner for Section 13 (ablation):
      - builds baseline (no reduction) + SVD(k) variants
      - evaluates each variant with the SAME cv_splits (required for fair comparison)
      - returns mean±std table + fold vectors (usable later for Wilcoxon)
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    X, y = _default_X_y(df, target_col=target_col, preprocess_config=preprocess_config)

    if cv_splits is None:
        cv_splits = s7.make_stratified_cv_splits(y, n_splits=n_splits, shuffle=True, random_state=cv_random_state)
    else:
        s7.validate_cv_splits(cv_splits, n_samples=len(y), n_splits_expected=n_splits)

    fp = s7.cv_splits_fingerprint(cv_splits)

    variants, pipes = build_section13_pipelines(
        df,
        preprocess_config=preprocess_config,
        model_random_state=model_random_state,
        svd_components=svd_components,
    )

    results_by_variant: Dict[str, s7.CVEvaluationResult] = {}
    for v in variants:
        res = s7.evaluate_binary_pipeline_cv(
            pipes[v.name],
            X,
            y,
            cv_splits=cv_splits,
            metrics=metrics,
            return_oof=return_oof,
        )
        results_by_variant[v.name] = res

    table = _summary_table(variants, results_by_variant, metrics=metrics)

    return Section13RunResult(
        target_col=target_col,
        n_splits=n_splits,
        cv_random_state=cv_random_state,
        split_fingerprint=fp,
        variants=list(variants),
        results_by_variant=results_by_variant,
        summary_table=table,
    )


def pick_best_variant_by_auc(out: Section13RunResult) -> str:
    """Convenience: return the variant name with best mean ROC-AUC."""
    if "roc_auc_mean" not in out.summary_table.columns:
        raise ValueError("summary_table has no roc_auc_mean column. Did you include roc_auc in metrics?")
    best_idx = int(out.summary_table["roc_auc_mean"].astype(float).idxmax())
    return str(out.summary_table.loc[best_idx, "variant"])


Writing section13_dimensionality_reduction.py


In [46]:
%%writefile test_section13_dimensionality_reduction.py

import unittest
import numpy as np
import pandas as pd

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7
import section13_dimensionality_reduction as s13


class TestSection13DimensionalityReduction(unittest.TestCase):
    def _toy_diabetes_like_df(self, n: int = 240) -> pd.DataFrame:
        rng = np.random.RandomState(0)

        # 20% positives, 80% negatives (enough for stratified CV)
        y = np.array([0] * int(n * 0.8) + [1] * (n - int(n * 0.8)), dtype=int)
        rng.shuffle(y)

        ages = list(s5.AGE_ORDER)
        df = pd.DataFrame(
            {
                "encounter_id": np.arange(n),
                "patient_nbr": np.arange(10_000, 10_000 + n),
                "race": rng.choice(["Caucasian", "AfricanAmerican", "Hispanic", "?"], size=n),
                "gender": rng.choice(["Male", "Female"], size=n),
                "age": rng.choice(ages + ["?"], size=n),
                "weight": rng.choice(["?", "70", "80", "90", "Unknown/Invalid"], size=n),
                "admission_type_id": rng.choice([1, 2, 6], size=n),
                "time_in_hospital": rng.choice([1, 2, 3, 4, 5, 6, 7], size=n),
                "payer_code": rng.choice(["MC", "MD", "?", "SP"], size=n),
                "medical_specialty": rng.choice(
                    ["Cardiology", "InternalMedicine", "?", "RareSpecA", "RareSpecB"], size=n
                ),
                "diag_1": rng.choice(["250.83", "276", "648", "8", "401"], size=n),
                "diag_2": rng.choice(["?", "250.01", "403", "V27"], size=n),
                "diag_3": rng.choice(["?", "7", "9", "6"], size=n),
                "A1Cresult": rng.choice(["None", "Norm", ">7", ">8", "?"], size=n),
                "insulin": rng.choice(["No", "Steady", "Up", "Down"], size=n),
                "change": rng.choice(["Ch", "No"], size=n),
                "diabetesMed": rng.choice(["Yes", "No"], size=n),
                "readmitted_30d": y,
            }
        )
        return df

    def test_build_section13_pipelines_contains_baseline_and_svd_variants(self):
        df = self._toy_diabetes_like_df(n=120)
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=True, rare_min_count=1)

        variants, pipes = s13.build_section13_pipelines(
            df, preprocess_config=cfg, model_random_state=0, svd_components=(5, 10)
        )

        names = [v.name for v in variants]
        self.assertIn("no_reduction", names)
        self.assertIn("svd_5", names)
        self.assertIn("svd_10", names)

        self.assertIn("no_reduction", pipes)
        self.assertIn("svd_5", pipes)
        self.assertIn("svd_10", pipes)

    def test_run_section13_produces_summary_and_identical_folds(self):
        df = self._toy_diabetes_like_df(n=240)
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=True, rare_min_count=1)

        y = df["readmitted_30d"].astype(int).to_numpy()
        splits = s7.make_stratified_cv_splits(y, n_splits=5, random_state=123)

        out = s13.run_section13_dimensionality_reduction(
            df,
            target_col="readmitted_30d",
            preprocess_config=cfg,
            svd_components=(5, 10),
            n_splits=5,
            cv_random_state=123,
            model_random_state=0,
            cv_splits=splits,
            return_oof=False,
        )

        # Expect: baseline + 2 svd variants => 3 rows
        self.assertEqual(out.summary_table.shape[0], 3)
        self.assertIn("roc_auc_mean", out.summary_table.columns)
        self.assertIn("f1_mean", out.summary_table.columns)

        # All variants must share the same split fingerprint (same cv_splits)
        fps = set()
        for name, res in out.results_by_variant.items():
            fps.add(res.split_fingerprint)
            self.assertEqual(len(res.fold_scores["roc_auc"]), 5)
            self.assertTrue(np.isfinite(res.fold_scores["roc_auc"]).all())
        self.assertEqual(len(fps), 1)
        self.assertEqual(out.split_fingerprint, list(fps)[0])

    def test_invalid_components_raises(self):
        df = self._toy_diabetes_like_df(n=120)
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=True, rare_min_count=1)

        with self.assertRaises(ValueError):
            _ = s13.build_section13_pipelines(df, preprocess_config=cfg, svd_components=(0, 10))

        with self.assertRaises(ValueError):
            _ = s13.build_section13_pipelines(df, preprocess_config=cfg, svd_components=(-5,))

    def test_pick_best_variant_by_auc_returns_valid_name(self):
        df = self._toy_diabetes_like_df(n=240)
        cfg = s5.DiabetesPreprocessConfig(onehot_sparse=True, rare_min_count=1)

        y = df["readmitted_30d"].astype(int).to_numpy()
        splits = s7.make_stratified_cv_splits(y, n_splits=4, random_state=7)

        out = s13.run_section13_dimensionality_reduction(
            df,
            target_col="readmitted_30d",
            preprocess_config=cfg,
            svd_components=(5,),
            n_splits=4,
            cv_random_state=7,
            model_random_state=0,
            cv_splits=splits,
            return_oof=False,
        )

        best = s13.pick_best_variant_by_auc(out)
        self.assertIn(best, set(out.summary_table["variant"].tolist()))


if __name__ == "__main__":
    unittest.main(verbosity=2)


Writing test_section13_dimensionality_reduction.py


In [47]:
!python -m unittest -v test_section13_dimensionality_reduction.py


test_build_section13_pipelines_contains_baseline_and_svd_variants (test_section13_dimensionality_reduction.TestSection13DimensionalityReduction.test_build_section13_pipelines_contains_baseline_and_svd_variants) ... ok
test_invalid_components_raises (test_section13_dimensionality_reduction.TestSection13DimensionalityReduction.test_invalid_components_raises) ... ok
test_pick_best_variant_by_auc_returns_valid_name (test_section13_dimensionality_reduction.TestSection13DimensionalityReduction.test_pick_best_variant_by_auc_returns_valid_name) ... ok
test_run_section13_produces_summary_and_identical_folds (test_section13_dimensionality_reduction.TestSection13DimensionalityReduction.test_run_section13_produces_summary_and_identical_folds) ... ok

----------------------------------------------------------------------
Ran 4 tests in 3.200s

OK


In [48]:
import os, sys, importlib.util
import pandas as pd

# -----------------------------
# 1) Robustly import your code.py as "project_code"
# -----------------------------
def import_module_from_path(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {module_name} from {file_path}")

    mod = importlib.util.module_from_spec(spec)

    # IMPORTANT (fixes dataclasses issues on some setups): register BEFORE exec_module
    sys.modules.pop(module_name, None)
    sys.modules[module_name] = mod

    spec.loader.exec_module(mod)
    return mod

PROJECT_DIR = os.getcwd()
CODE_PATH = os.path.join(PROJECT_DIR, "code.py")
project_code = import_module_from_path("project_code", CODE_PATH)

# -----------------------------
# 2) Load REAL data from the required relative path
# -----------------------------
DATA_PATH = os.path.join(PROJECT_DIR, "database", "diabetic_data.csv")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"CSV not found at: {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print("Loaded:", df_raw.shape)
display(df_raw.head(3))

# -----------------------------
# 3) (Optional) Apply ID->description mappings if you have id_maps defined
#     - If you DON'T have id_maps, this block does nothing (safe).
# -----------------------------
df = df_raw.copy()

# If you have mapping DataFrames, define them BEFORE this cell like:
# id_maps = {
#   "admission_type_id": admission_type_map_df,            # columns: ["admission_type_id","description"]
#   "discharge_disposition_id": discharge_disp_map_df,     # columns: ["discharge_disposition_id","description"]
#   "admission_source_id": admission_source_map_df,        # columns: ["admission_source_id","description"]
# }

if "id_maps" not in globals():
    id_maps = {}  # safe default: no mappings available

def apply_mapping(df: pd.DataFrame, id_col: str, new_col: str) -> pd.DataFrame:
    if id_col not in df.columns or id_col not in id_maps:
        return df
    map_df = id_maps[id_col]
    if id_col not in map_df.columns or "description" not in map_df.columns:
        raise ValueError(f"id_maps['{id_col}'] must have columns [{id_col}, 'description'].")

    mapper = map_df.set_index(id_col)["description"]
    mapper.index = mapper.index.astype("string")

    keys = df[id_col].astype("string").str.strip()
    df[new_col] = keys.map(mapper)  # unknown stays <NA>
    return df

df = apply_mapping(df, "admission_type_id", "admission_type")
df = apply_mapping(df, "discharge_disposition_id", "discharge_disposition")
df = apply_mapping(df, "admission_source_id", "admission_source")

# If mappings were applied, you can drop the numeric IDs (optional)
to_drop = [c for c in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]
           if c in df.columns and c in id_maps]
if to_drop:
    df = df.drop(columns=to_drop)

# -----------------------------
# 4) Engineer targets (binary + optional multiclass)
# -----------------------------
df = project_code.engineer_targets(
    df,
    source_col="readmitted",
    binary_col="readmitted_30d",
    multiclass_col="readmitted_3class",
    add_multiclass=True,
    drop_source=False,
    unknown_policy="error",  # fail fast if unexpected values appear
)

# -----------------------------
# 5) Quick sanity checks
# -----------------------------
display(df[["readmitted", "readmitted_30d", "readmitted_3class"]].head(10))
print("\nreadmitted value counts:")
print(df["readmitted"].value_counts(dropna=False))
print("\nBinary target counts:")
print(df["readmitted_30d"].value_counts(dropna=False))


Loaded: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


,readmitted,readmitted_30d,readmitted_3class
0,NO,0,NO
1,>30,0,>30
2,NO,0,NO
3,NO,0,NO
4,NO,0,NO
5,>30,0,>30
6,NO,0,NO
7,>30,0,>30
8,NO,0,NO
9,NO,0,NO



readmitted value counts:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Binary target counts:
readmitted_30d
0    90409
1    11357
Name: count, dtype: int64
